# Кандидатогенерация объявлений услуг

## 1. Анализ и первичная предобработка данных

На этом этапе проверяем структуру, качество и пересечения данных. Исходные таблицы не изменяем: очищенные тексты, агрегаты и обучающие выборки далее будут создаваться как отдельные объекты.

In [1]:
from time import perf_counter

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def show_dataset_preview(df, rows=5):
    """Выводит схему и несколько первых строк одного датасета."""
    df.info(show_counts=True, memory_usage="deep")
    display(df.head(rows))

### 1.1. Загрузка файлов

Все входные Parquet-файлы лежат в папке `data`.

In [2]:
started_at = perf_counter()

train = pd.read_parquet(
    "data/train.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow",
)
benchmark_queries = pd.read_parquet(
    "data/benchmark_queries.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow",
)
benchmark_items = pd.read_parquet(
    "data/benchmark_items.parquet",
    engine="pyarrow",
    dtype_backend="pyarrow",
)

datasets = {
    "train": train,
    "benchmark_queries": benchmark_queries,
    "benchmark_items": benchmark_items,
}

QUERY_FEATURE_COLUMNS = benchmark_queries.columns.drop("query_id").tolist()

print(f"Все файлы загружены за {perf_counter() - started_at:.2f} сек.")

Все файлы загружены за 11.11 сек.


In [3]:
dataset_report = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "memory_mib": round(
                df.memory_usage(index=True, deep=True).sum() / 1024**2, 2
            ),
        }
        for name, df in datasets.items()
    ]
)

display(dataset_report)

,dataset,rows,columns,memory_mib
0,train,497673,19,"2,196.7500"
1,benchmark_queries,2452,6,0.3100
2,benchmark_items,189212,14,803.0800


### 1.2. Схема и первые строки

Посмотрим визуально на каждую из таблиц

In [4]:
show_dataset_preview(train)

<class 'pandas.DataFrame'>
RangeIndex: 497673 entries, 0 to 497672
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   search_query               497673 non-null  string[pyarrow]            
 1   search_location_id         497673 non-null  int64[pyarrow]             
 2   search_is_delivery_search  497673 non-null  int32[pyarrow]             
 3   search_infm_params_text    497673 non-null  string[pyarrow]            
 4   search_category            497673 non-null  int64[pyarrow]             
 5   item_title_raw             497673 non-null  string[pyarrow]            
 6   item_rating_reviews_count  478462 non-null  double[pyarrow]            
 7   item_rating                469441 non-null  double[pyarrow]            
 8   item_price                 497673 non-null  decimal128(27, 15)[pyarrow]
 9   item_microcat_id           497673 non-null  int6

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0000,4.9890,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлицами и ИП Опыт работы ...",e8b685dffe1a408e,"Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночной стоимости. Все пре...",114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0000,4.9827,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха Тип стоимости за ус...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и области. 🫴 Помогу подобрать оптимальный вариант...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",<NA>,<NA>,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое товарищество Веретен...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтобы остановиться: выключить телефон, отложить дела и подарить ...",114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0000,4.7143,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг Краснодарский край, Ка...",45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу на официальных основаниях. ЛЮБОЙ РЕГИОН России 🇷🇺. Изготовление по ...,114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,114,Ремонт и отделка квартир под ключ,1.0000,5.0000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ключ Место оказания услуг ул. Полины Осипенко Тип ...,c95a4a7daf2a967f,отделочные работы любой сложности.,114


In [5]:
show_dataset_preview(benchmark_queries)

<class 'pandas.DataFrame'>
RangeIndex: 2452 entries, 0 to 2451
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype                
---  ------                     --------------  -----                
 0   query_id                   2452 non-null   large_string[pyarrow]
 1   search_query               2452 non-null   large_string[pyarrow]
 2   search_location_id         2452 non-null   int64[pyarrow]       
 3   search_is_delivery_search  2452 non-null   int32[pyarrow]       
 4   search_infm_params_text    2452 non-null   large_string[pyarrow]
 5   search_category            2452 non-null   int64[pyarrow]       
dtypes: int32[pyarrow](1), int64[pyarrow](2), large_string[pyarrow](3)
memory usage: 319.1 KB


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


In [6]:
show_dataset_preview(benchmark_items)

<class 'pandas.DataFrame'>
RangeIndex: 189212 entries, 0 to 189211
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   item_title_raw             189212 non-null  string[pyarrow]            
 1   item_rating_reviews_count  177597 non-null  double[pyarrow]            
 2   item_rating                171581 non-null  double[pyarrow]            
 3   item_price                 189212 non-null  decimal128(27, 15)[pyarrow]
 4   item_microcat_id           189212 non-null  int64[pyarrow]             
 5   item_longitude             189211 non-null  decimal128(18, 15)[pyarrow]
 6   item_location_id           189212 non-null  int64[pyarrow]             
 7   item_latitude              189211 non-null  decimal128(17, 15)[pyarrow]
 8   item_is_phone_hidden       189212 non-null  bool[pyarrow]              
 9   item_is_message_forbidden  189212 non-null  bool

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,57.0000,5.0000,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00 ● Диагностика устройств; ● Чистка от пыли и замена термопасты; ● Пров...",114
1,Обучение ребенка чтению,7.0000,5.0000,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стои...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! Я, детский психолог/нейропсихолог, рада пригласить вас и ваших детей на за...",114
2,Афрокудри 5+,14.0000,5.0000,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парик...",3f6ae81704565b9c,"ВНИМАНИЕ Укладка !!!! 🥳 АФРОкудри отличный вариант замены хим завивки, если хочется поменять временно свой образ. ...",114
3,Монтаж малых архитектурных форм,2.0000,5.0000,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу Работа по договору ...",0618dec37a59a21a,"Оказываем все виды услуг по сборке и монтажу игровых площадок , а так же все виды работ по устройству основания и ог...",114
4,Репетитор по истории 10 класс ЕГЭ,19.0000,5.0000,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ленина Тип услуги Пред...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев! Мечтаешь поступить в вуз своей мечты? ✨ История больше не будет скучной...,114


### 1.3. Пропуски

Посмотрим на столбцы, в которых есть пропущенные значения.

In [7]:
for name, df in datasets.items():
    missing = df.isna().sum()
    print(f"\n{name}")
    display(
        missing[missing > 0]
        .sort_values(ascending=False)
        .rename("missing_count")
        .to_frame()
    )


train


,missing_count
item_rating,28232
item_rating_reviews_count,19211
item_longitude,4
item_latitude,4



benchmark_queries


,missing_count



benchmark_items


,missing_count
item_rating,17631
item_rating_reviews_count,11615
item_description_raw,35
item_longitude,1
item_latitude,1


### 1.4. Нормализация текстов

Нормализуем текстовые поля одинаково в train и benchmark: приводим к нижнему регистру, заменяем ё на е, убираем пунктуацию и лишние пробелы. Исходные Parquet-файлы не меняются.

In [4]:
def normalize_text(series):
    return (
        series.fillna("")
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"[^0-9a-zа-я]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def apply_to_existing_columns(dataframes, columns, transform):
    """Применяет преобразование только к присутствующим текстовым столбцам."""
    for df in dataframes:
        for column in columns:
            if column in df.columns:
                df[column] = transform(df[column])


text_columns = [
    "search_query",
    "search_infm_params_text",
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

apply_to_existing_columns(
    [train, benchmark_queries, benchmark_items],
    text_columns,
    normalize_text,
)

display(train.head())
display(benchmark_queries.head())
display(benchmark_items.head())

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,скупка б у техники,91.0000,4.9890,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,вид услуги место оказания услуг первомайский пр т 66 тип стоимости за услугу работаете с юрлицами и ип опыт работы 1...,e8b685dffe1a408e,скупаю практически любую современную новую и б у технику обязательно рабочую до 80 от рыночной стоимости все предлож...,114
1,автоподбор,640860,0,рейтинг пользователя 4 звезды и выше,114,автоподбор разовый осмотр автомобиля,462.0000,4.9827,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,вид услуги место оказания услуг нижний новгород советский район жилой комплекс новая кузнечиха тип стоимости за услу...,92e1b0446f827b59,автоподбор и выездная диагностика автомобиля в нижнем новгороде и области помогу подобрать оптимальный вариант автом...,114
2,баня на дровах,653240,0,онлайн запись тип услуги спа услуги массаж вид услуги красота здоровье,114,баня на дровах прованс на цветочной,<NA>,<NA>,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,вид услуги красота здоровье место оказания услуг санкт петербург садоводческое некоммерческое товарищество веретено ...,624846856ce81d69,в ритме современной жизни так сложно найти момент чтобы остановиться выключить телефон отложить дела и подарить себе...,114
3,изготовление госномера на авто,634670,0,вид услуги оборудование производство,114,изготовление дубликатов авто номеров гос номеров,14.0000,4.7143,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,вид услуги оборудование производство тип услуги производство обработка место оказания услуг краснодарский край кавка...,45b8628b9c6e7b85,изготовим дубликат номера по утере или износу на официальных основаниях любой регион россии изготовление по гост в с...,114
4,укладка плитки,658430,0,тип услуги ремонт квартир и домов под ключ вид услуги ремонт и отделка,114,ремонт и отделка квартир под ключ,1.0000,5.0000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,вид услуги ремонт и отделка тип услуги ремонт квартир и домов под ключ место оказания услуг ул полины осипенко тип с...,c95a4a7daf2a967f,отделочные работы любой сложности,114


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,вид услуги деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,вид услуги красота здоровье,114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,ремонт выкуп компьют и ноутбуков с выездом на дом,57.0000,5.0000,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,вид услуги компьютерная помощь место оказания услуг пр т ленина тип стоимости за услугу начальная цена график работы...,111eb8b979577d79,выезд на дом в любое время вплоть до 23 00 диагностика устройств чистка от пыли и замена термопасты проверка работос...,114
1,обучение ребенка чтению,7.0000,5.0000,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,вид услуги обучение курсы место оказания услуг ул чернышевского 35 тип услуги детское развитие логопеды тип стоимост...,75fc8e10f5a66fc4,дорогие родители и маленькие книголюбы я детский психолог нейропсихолог рада пригласить вас и ваших детей на захваты...,114
2,афрокудри 5,14.0000,5.0000,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,вид услуги красота здоровье место оказания услуг городской округ иваново фрунзенский район тип услуги услуги парикма...,3f6ae81704565b9c,внимание укладка афрокудри отличный вариант замены хим завивки если хочется поменять временно свой образ для взрослы...,114
3,монтаж малых архитектурных форм,2.0000,5.0000,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,вид услуги строительство место оказания услуг севастопольский пр т 28к4 тип стоимости за услугу работа по договору г...,0618dec37a59a21a,оказываем все виды услуг по сборке и монтажу игровых площадок а так же все виды работ по устройству основания и огра...,114
4,репетитор по истории 10 класс егэ,19.0000,5.0000,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,вид услуги обучение курсы место оказания услуг волгоградская обл волжский пл имени в и ленина тип услуги предметы шк...,13da2c81574677ed,сдай егэ по истории на 80 за 6 месяцев мечтаешь поступить в вуз своей мечты история больше не будет скучной и сложно...,114


### 1.5. Очистка названий полей в параметрах

Удаляем только служебные названия полей. Значения параметров остаются в строке.

In [5]:
service_labels = [
    "вид услуги",
    "тип услуги",
    "место оказания услуг",
    "название услуги",
    "марка авто",
]


def remove_service_labels(series):
    for label in service_labels:
        series = series.str.replace(label, "", regex=False)
    return series.str.replace(r"\s+", " ", regex=True).str.strip()


apply_to_existing_columns(
    [train, benchmark_queries, benchmark_items],
    ["search_infm_params_text", "item_infm_params_text"],
    remove_service_labels,
)

## 2. Запросы, объявления и их связи

В `train` одна строка — положительная пара. Выделяем уникальные запросы и уникальные объявления в отдельные таблицы, а все связи между ними сохраняем в `relevance`.

In [6]:
train_queries = train[QUERY_FEATURE_COLUMNS].drop_duplicates().reset_index(drop=True)
train_queries.insert(0, "train_query_id", range(len(train_queries)))

train_items = train[benchmark_items.columns].drop_duplicates("item_id").reset_index(drop=True)

relevance = (
    train[QUERY_FEATURE_COLUMNS + ["item_id"]]
    .merge(train_queries, on=QUERY_FEATURE_COLUMNS, how="left")
    [["train_query_id", "item_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Уникальных запросов:", len(train_queries))
print("Уникальных объявлений:", len(train_items))
print("Связей запрос — объявление:", len(relevance))

Уникальных запросов: 352885
Уникальных объявлений: 344825
Связей запрос — объявление: 466694


### 2.1. Пересечение объявлений train и benchmark_items

Сравниваем `item_id`: сколько объявлений из train есть в корпусе итогового поиска. Каждое объявление считаем один раз, даже если его выбирали по нескольким запросам. Исходные таблицы не фильтруем.

In [7]:
train_item_ids = set(train_items["item_id"])
benchmark_item_ids = set(benchmark_items["item_id"])
common_item_ids = train_item_ids & benchmark_item_ids
print("Всего разных объявлений в benchmark_items:", len(benchmark_item_ids))
print("Всего разных объявлений в train:", len(train_item_ids))
print("Из них есть в benchmark_items:", len(common_item_ids))

Всего разных объявлений в benchmark_items: 189212
Всего разных объявлений в train: 344825
Из них есть в benchmark_items: 18142


### 2.2. Вывод

Обучающая выборка значительно больше benchmark и слабо пересекается с его каталогом объявлений.

## 3. Обучающая и валидационная выборки

Выделим 20% текстов запросов для валидации. Запросы с одинаковым текстом, даже при разных локациях и фильтрах, отправляем в одну часть. Связи `relevance` делим вместе с запросами, а каталог объявлений не делим. Тексты уже нормализованы в предыдущей ячейке. Это проверка на новых текстах запросов; в benchmark часть текстов уже встречается в train, поэтому локальная метрика не будет точной копией итоговой.

Одно объявление может быть связано с запросами в обеих частях — это нормально, потому что каталог стабилен. `random_state=42` фиксирует разбиение для сравнения моделей.

### 3.1. Разделение запросов и связей

Запросы и их связи с релевантными объявлениями делим по одной маске. Каталог `train_items` остаётся единым.

In [8]:
query_groups = train_queries["search_query"]
valid_groups = query_groups.drop_duplicates().sample(frac=0.2, random_state=42)
is_valid = query_groups.isin(valid_groups)

queries_train = train_queries.loc[~is_valid].copy()
queries_valid = train_queries.loc[is_valid].copy()

relevance_is_valid = relevance["train_query_id"].isin(
    queries_valid["train_query_id"]
)
relevance_train = relevance.loc[~relevance_is_valid].copy()
relevance_valid = relevance.loc[relevance_is_valid].copy()

print("Запросов для обучения:", len(queries_train))
print("Запросов для валидации:", len(queries_valid))
print("Связей для обучения:", len(relevance_train))
print("Связей для валидации:", len(relevance_valid))
relevance

Запросов для обучения: 285699
Запросов для валидации: 67186
Связей для обучения: 377823
Связей для валидации: 88871


,train_query_id,item_id
0,0,e8b685dffe1a408e
1,1,92e1b0446f827b59
2,2,624846856ce81d69
3,3,45b8628b9c6e7b85
4,4,c95a4a7daf2a967f
...,...,...
466689,352880,789f808e94a9605b
466690,352881,ce735daa3e5b4008
466691,352882,78dd659b6d8c0709
466692,352883,e3759b5870db901d


## 4. Проверка повторяющихся запросов

Проверяем, встречаются ли полностью одинаковые наборы признаков запроса в train и benchmark.

In [13]:
def check_duplicate_queries(df, dataset_name):
    query_sizes = df.groupby(QUERY_FEATURE_COLUMNS, dropna=False).size()
    repeated_queries = query_sizes[query_sizes > 1]

    print(dataset_name)
    print("Всего разных полных запросов:", len(query_sizes))
    print("Полностью повторяющихся запросов:", len(repeated_queries))
    print("Строк в повторяющихся запросах:", repeated_queries.sum())


check_duplicate_queries(train, "train")
check_duplicate_queries(benchmark_queries, "benchmark_queries")

train
Всего разных полных запросов: 352885
Полностью повторяющихся запросов: 58574
Строк в повторяющихся запросах: 203362
benchmark_queries
Всего разных полных запросов: 2452
Полностью повторяющихся запросов: 0
Строк в повторяющихся запросах: 0


## 5. Метрика Recall@50

`predictions` — словарь вида `{train_query_id: [item_id, ...]}`. Для каждого запроса считаем долю его релевантных объявлений, попавших в первые 50 кандидатов, и усредняем результат по запросам.

In [9]:
import numpy as np

TOP_50 = 50


def make_relevant_items_by_query(relevance, item_column="item_id"):
    """Собирает множество релевантных объектов для каждого запроса."""
    return (
        relevance.groupby("train_query_id")[item_column]
        .agg(set)
        .to_dict()
    )


relevant_items_valid_by_query = make_relevant_items_by_query(
    relevance_valid
)


def recall_at_k(predictions, relevant_items_by_query, k=TOP_50):
    recall_values = []

    for query_id, relevant_items in relevant_items_by_query.items():
        predicted_items = set(predictions.get(query_id, [])[:k])
        recall_values.append(
            len(predicted_items & relevant_items) / len(relevant_items)
        )

    return sum(recall_values) / len(recall_values)


def recall_at_50(predictions, relevant_items_by_query):
    return recall_at_k(predictions, relevant_items_by_query, TOP_50)


def recall_from_top_indices(
    top_indices, query_ids, relevant_indices_by_query, top_n=TOP_50
):
    """Считает Recall@top_n, когда кандидаты заданы номерами строк корпуса."""
    recall_values = []

    for row, query_id in enumerate(query_ids):
        item_indices = top_indices[row, :top_n]
        predicted = set(item_indices[item_indices >= 0])
        relevant = relevant_indices_by_query[query_id]
        recall_values.append(len(predicted & relevant) / len(relevant))

    return float(np.mean(recall_values))


def make_relevant_indices_by_query(relevance, item_ids):
    """Заменяет item_id релевантных пар на номера строк текущего корпуса."""
    item_positions = {
        item_id: position for position, item_id in enumerate(item_ids)
    }
    relevance_with_positions = relevance.assign(
        item_position=relevance["item_id"].map(item_positions)
    ).dropna(subset=["item_position"])
    relevance_with_positions["item_position"] = (
        relevance_with_positions["item_position"].astype(np.int32)
    )
    return make_relevant_items_by_query(
        relevance_with_positions, item_column="item_position"
    )

## Общие функции поиска

Здесь только определения функций: они не строят индексы и ничего не считают. Поэтому после перезапуска ядра их можно выполнить отдельно, не запуская тяжёлые эксперименты ниже.

In [72]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


def build_filter_token_matrices(item_param_text, filters):
    """Строит бинарные токен-матрицы параметров и фильтров."""
    vectorizer = CountVectorizer(binary=True, token_pattern=r"(?u)\b\w+\b")
    vectorizer.fit(pd.concat([item_param_text.fillna(""), pd.Series(filters)]))
    item_matrix = vectorizer.transform(item_param_text.fillna(""))
    filter_matrix = vectorizer.transform(filters)
    filter_token_counts = np.asarray(filter_matrix.sum(axis=1)).ravel()
    return item_matrix, filter_matrix, filter_token_counts


def combine_text(df, columns):
    text = df[columns[0]].fillna("")
    for column in columns[1:]:
        text = text.str.cat(df[column].fillna(""), sep=" ")
    return text


def top_positions(scores, top_n):
    """Номера top_n наибольших значений, отсортированные по убыванию."""
    n_top = min(top_n, len(scores))
    if n_top == 0:
        return np.empty(0, dtype=np.int32)

    if n_top == len(scores):
        positions = np.arange(n_top)
    else:
        positions = np.argpartition(scores, -n_top)[-n_top:]
    return positions[np.argsort(scores[positions])[::-1]]


def write_top_k_from_sparse_scores(
    sparse_scores, top_indices, top_scores, first_row
):
    """Записывает ранжированные top-k из одного разреженного батча."""
    top_k = top_indices.shape[1]

    for local_row in range(sparse_scores.shape[0]):
        row_scores = sparse_scores.getrow(local_row)
        positions = top_positions(row_scores.data, top_k)
        if len(positions) == 0:
            continue

        row = first_row + local_row
        top_indices[row, :len(positions)] = row_scores.indices[positions]
        top_scores[row, :len(positions)] = row_scores.data[positions]


def get_top_k_indices_and_scores(
    query_matrix,
    item_matrix,
    top_k=TOP_50,
    batch_size=128,
    source_name=None,
):
    """Возвращает номера строк корпуса и исходные скоры top-k."""
    n_queries = query_matrix.shape[0]
    top_indices = np.full((n_queries, top_k), -1, dtype=np.int32)
    top_scores = np.full((n_queries, top_k), -np.inf, dtype=np.float32)
    item_matrix_t = item_matrix.T

    for start in range(0, n_queries, batch_size):
        stop = min(start + batch_size, n_queries)
        batch_scores = (query_matrix[start:stop] @ item_matrix_t).tocsr()
        write_top_k_from_sparse_scores(
            batch_scores, top_indices, top_scores, start
        )

        if source_name and (
            start == 0
            or stop == n_queries
            or (start // batch_size + 1) % 10 == 0
        ):
            print(f"{source_name}: {stop:,}/{n_queries:,} запросов")

    return top_indices, top_scores


def predictions_from_top_indices(top_indices, query_ids, item_ids, top_n=TOP_50):
    """Преобразует номера строк корпуса в item_id для расчёта Recall."""
    predictions = {}
    for row, query_id in enumerate(query_ids):
        item_indices = top_indices[row, :top_n]
        item_indices = item_indices[item_indices >= 0]
        predictions[query_id] = item_ids[item_indices].tolist()
    return predictions


def build_bm25_item_matrix(item_text, k1=1.5, b=0.75):
    """Строит BM25-матрицу корпуса и возвращает её vectorizer."""
    vectorizer = CountVectorizer(dtype=np.float32)
    item_counts = vectorizer.fit_transform(item_text)

    document_lengths = np.asarray(item_counts.sum(axis=1)).ravel()
    average_length = document_lengths.mean()
    document_frequency = np.asarray((item_counts > 0).sum(axis=0)).ravel()
    idf = np.log(
        1 + (item_counts.shape[0] - document_frequency + 0.5)
        / (document_frequency + 0.5)
    )

    length_norm = k1 * (1 - b + b * document_lengths / average_length)
    item_bm25 = item_counts.copy()
    item_bm25.data = (
        item_bm25.data
        * (k1 + 1)
        / (item_bm25.data + np.repeat(length_norm, np.diff(item_bm25.indptr)))
    )
    item_bm25.data *= idf[item_bm25.indices]

    return vectorizer, item_bm25.tocsr()


def build_retrieval_matrices(method, item_text, query_text, vectorizer=None):
    """Строит одинаковый интерфейс матриц для TF-IDF и BM25."""
    if method == "bm25":
        vectorizer, item_matrix = build_bm25_item_matrix(item_text)
        query_matrix = vectorizer.transform(query_text).astype(np.float32)
        query_matrix.data.fill(1.0)
    elif method == "tfidf":
        if vectorizer is None:
            raise ValueError("Для TF-IDF нужен vectorizer.")
        item_matrix = vectorizer.fit_transform(item_text)
        query_matrix = vectorizer.transform(query_text)
    else:
        raise ValueError(f"Неизвестный метод: {method}")

    return vectorizer, query_matrix, item_matrix


def retrieve_top_k(
    method,
    item_text,
    query_text,
    top_k,
    vectorizer=None,
    batch_size=128,
    source_name=None,
):
    """Строит поисковый источник и возвращает его матрицу и top-k."""
    matrix_method = "bm25" if method == "bm25" else "tfidf"
    vectorizer, query_matrix, item_matrix = build_retrieval_matrices(
        matrix_method, item_text, query_text, vectorizer
    )
    top_indices, top_scores = get_top_k_indices_and_scores(
        query_matrix,
        item_matrix,
        top_k=top_k,
        batch_size=batch_size,
        source_name=source_name,
    )
    del query_matrix
    return vectorizer, item_matrix, top_indices, top_scores


def make_tfidf_vectorizer(kind):
    """Создаёт word- или char-TF-IDF с параметрами эксперимента."""
    if kind == "word":
        return TfidfVectorizer(dtype=np.float32)
    if kind == "char":
        return TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            max_features=100_000,
            dtype=np.float32,
        )
    raise ValueError(f"Неизвестный вид TF-IDF: {kind}")


def get_retrieval_predictions(
    method,
    query_text,
    item_text,
    query_ids,
    item_ids,
    vectorizer=None,
    batch_size=128,
):
    vectorizer, item_matrix, top_indices, top_scores = retrieve_top_k(
        method,
        item_text,
        query_text,
        top_k=TOP_50,
        vectorizer=vectorizer,
        batch_size=batch_size,
    )
    predictions = predictions_from_top_indices(
        top_indices, query_ids, item_ids
    )
    del vectorizer, item_matrix, top_indices, top_scores
    return predictions


def source_candidates_and_scores(
    row,
    weights,
    source_indices_by_name,
    source_scores_by_name,
    source_names=None,
):
    """Собирает кандидатов источников и считает нормализованный текстовый H."""
    if source_names is None:
        source_names = tuple(source_indices_by_name)

    source_indices = [
        source_indices_by_name[name][row]
        for name in source_names
    ]
    candidate_parts = [
        indices[indices >= 0]
        for indices in source_indices
        if np.any(indices >= 0)
    ]
    if not candidate_parts:
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.float32)

    candidates = np.unique(np.concatenate(candidate_parts))
    combined_scores = np.zeros(len(candidates), dtype=np.float32)

    for weight, name, indices in zip(weights, source_names, source_indices):
        valid = indices >= 0
        if weight == 0 or not np.any(valid):
            continue

        source_scores = source_scores_by_name[name][row][valid]
        max_score = source_scores.max()
        if max_score > 0:
            positions = np.searchsorted(candidates, indices[valid])
            combined_scores[positions] += weight * source_scores / max_score

    return candidates, combined_scores


def location_bonus(
    candidate_indices, query_location_id, bonus, item_location_values
):
    """Даёт бонус кандидатам из той же локации, что и запрос."""
    return bonus * (
        item_location_values[candidate_indices] == query_location_id
    )


def query_row_groups(values):
    """Возвращает пары «значение признака — номера запросов с ним»."""
    values = np.asarray(values)
    for value in pd.unique(values):
        if not pd.isna(value):
            yield value, np.flatnonzero(values == value)


def item_indices_within_radius(
    location_id,
    radius_km,
    item_location_values,
    item_city_codes,
    city_ids,
    city_distances_km,
):
    """Возвращает объявления той же локации либо всех городов в радиусе."""
    if radius_km <= 0:
        return np.flatnonzero(item_location_values == location_id)

    city_code = city_ids.get_indexer([location_id])[0]
    if city_code < 0:
        return np.empty(0, dtype=np.int32)

    nearby_city_codes = np.flatnonzero(
        city_distances_km[city_code] <= radius_km
    )
    return np.flatnonzero(np.isin(item_city_codes, nearby_city_codes))


def item_indices_matching_filter(
    filter_code,
    item_token_matrix,
    filter_token_matrix,
    required_matches,
):
    """Возвращает item-строки с нужной долей токенов фильтра."""
    if required_matches[filter_code] == 0:
        return np.arange(item_token_matrix.shape[0], dtype=np.int32)

    overlap = (
        filter_token_matrix[filter_code] @ item_token_matrix.T
    ).tocsr()
    return overlap.indices[overlap.data >= required_matches[filter_code]]


def iter_top_k_in_item_subset(
    query_matrix,
    item_matrix,
    query_rows,
    item_indices,
    top_k,
    batch_size=64,
):
    """Ищет top-K для запросов только среди указанных item-строк.

    Возвращает номер запроса, item-строки и исходные скоры. Подходит для
    локаций, фильтров и любых следующих ограничений корпуса.
    """
    item_indices = np.asarray(item_indices, dtype=np.int32)
    item_matrix_t = item_matrix[item_indices].T.tocsr()

    for start in range(0, len(query_rows), batch_size):
        batch_rows = query_rows[start:start + batch_size]
        batch_scores = (query_matrix[batch_rows] @ item_matrix_t).tocsr()

        for local_row, row in enumerate(batch_rows):
            score_row = batch_scores.getrow(local_row)
            positions = top_positions(score_row.data, top_k)
            yield (
                row,
                item_indices[score_row.indices[positions]],
                score_row.data[positions],
            )


def item_indices_from_region(
    region_code,
    region_city_scores,
    item_city_codes,
):
    """Возвращает объявления городов, связанных с данным регионом."""
    if region_code < 0:
        return np.empty(0, dtype=np.int32)

    known_cities = item_city_codes >= 0
    result = np.zeros(len(item_city_codes), dtype=bool)
    result[known_cities] = (
        region_city_scores[region_code, item_city_codes[known_cities]] > 0
    )
    return np.flatnonzero(result)


## 6. Сравнение вариантов word TF-IDF

Индексируем весь каталог `train_items`, не используя клики. Сравниваем четыре сочетания текста запроса и текста объявления по `relevance_valid`.

Общие функции поиска уже определены выше. Эксперименты ниже только вызывают их, поэтому одинаковая логика не повторяется.

In [17]:
import gc

variants = [
    ("запрос — заголовок объявления", ["search_query"], ["item_title_raw"]),
    ("запрос + фильтры — заголовок объявления", ["search_query", "search_infm_params_text"], ["item_title_raw"]),
    ("запрос — заголовок + описание объявления", ["search_query"], ["item_title_raw", "item_description_raw"]),
    ("запрос + фильтры — заголовок + описание объявления", ["search_query", "search_infm_params_text"], ["item_title_raw", "item_description_raw"]),
]

query_ids = queries_valid["train_query_id"].to_numpy()
item_ids = train_items["item_id"].to_numpy()
tfidf_results = []

for name, query_columns, item_columns in variants:
    print(f"Запуск: {name}")
    started_at = perf_counter()

    predictions = get_retrieval_predictions(
        method="tfidf",
        vectorizer=make_tfidf_vectorizer("word"),
        query_text=combine_text(queries_valid, query_columns),
        item_text=combine_text(train_items, item_columns),
        query_ids=query_ids,
        item_ids=item_ids,
    )

    tfidf_results.append(
        {
            "variant": name,
            "recall_at_50": recall_at_50(
                predictions, relevant_items_valid_by_query
            ),
            "minutes": round((perf_counter() - started_at) / 60, 2),
        }
    )
    del predictions
    gc.collect()

display(pd.DataFrame(tfidf_results).sort_values("recall_at_50", ascending=False))

Запуск: запрос — заголовок объявления
Запуск: запрос + фильтры — заголовок объявления
Запуск: запрос — заголовок + описание объявления
Запуск: запрос + фильтры — заголовок + описание объявления


,variant,recall_at_50,minutes
0,запрос — заголовок объявления,0.1493,0.6100
2,запрос — заголовок + описание объявления,0.1277,9.8100
1,запрос + фильтры — заголовок объявления,0.1136,0.8300
3,запрос + фильтры — заголовок + описание объявления,0.1066,10.8100


## 7. Сравнение вариантов char TF-IDF

Используем символьные n-граммы длины 3–5. Ограничение словаря до 100 000 n-грамм сдерживает потребление памяти в вариантах с полными описаниями.

In [18]:
char_tfidf_results = []

for name, query_columns, item_columns in variants:
    print(f"Запуск: {name}")
    started_at = perf_counter()

    predictions = get_retrieval_predictions(
        method="tfidf",
        vectorizer=make_tfidf_vectorizer("char"),
        query_text=combine_text(queries_valid, query_columns),
        item_text=combine_text(train_items, item_columns),
        query_ids=query_ids,
        item_ids=item_ids,
        batch_size=64,
    )

    char_tfidf_results.append(
        {
            "variant": name,
            "recall_at_50": recall_at_50(
                predictions, relevant_items_valid_by_query
            ),
            "minutes": round((perf_counter() - started_at) / 60, 2),
        }
    )
    del predictions
    gc.collect()

display(pd.DataFrame(char_tfidf_results).sort_values("recall_at_50", ascending=False))

Запуск: запрос — заголовок объявления
Запуск: запрос + фильтры — заголовок объявления
Запуск: запрос — заголовок + описание объявления
Запуск: запрос + фильтры — заголовок + описание объявления


,variant,recall_at_50,minutes
0,запрос — заголовок объявления,0.1713,12.1500
2,запрос — заголовок + описание объявления,0.1440,389.4800
1,запрос + фильтры — заголовок объявления,0.1237,20.0800
3,запрос + фильтры — заголовок + описание объявления,0.1169,327.8000


## 8. Сравнение вариантов BM25

BM25 считаем поверх тех же слов и тех же четырёх сочетаний полей.

In [19]:
bm25_results = []

for name, query_columns, item_columns in variants:
    print(f"Запуск: {name}")
    started_at = perf_counter()

    predictions = get_retrieval_predictions(
        method="bm25",
        query_text=combine_text(queries_valid, query_columns),
        item_text=combine_text(train_items, item_columns),
        query_ids=query_ids,
        item_ids=item_ids,
    )

    bm25_results.append(
        {
            "variant": name,
            "recall_at_50": recall_at_50(
                predictions, relevant_items_valid_by_query
            ),
            "minutes": round((perf_counter() - started_at) / 60, 2),
        }
    )
    del predictions
    gc.collect()

all_ir_results = pd.concat(
    [
        pd.DataFrame(tfidf_results).assign(method="word_tfidf"),
        pd.DataFrame(char_tfidf_results).assign(method="char_tfidf"),
        pd.DataFrame(bm25_results).assign(method="bm25"),
    ],
    ignore_index=True,
)
display(all_ir_results.sort_values("recall_at_50", ascending=False))

Запуск: запрос — заголовок объявления
Запуск: запрос + фильтры — заголовок объявления
Запуск: запрос — заголовок + описание объявления
Запуск: запрос + фильтры — заголовок + описание объявления


,variant,recall_at_50,minutes,method
10,запрос — заголовок + описание объявления,0.1738,9.3700,bm25
4,запрос — заголовок объявления,0.1713,12.1500,char_tfidf
8,запрос — заголовок объявления,0.1575,0.5400,bm25
11,запрос + фильтры — заголовок + описание объявления,0.1500,10.1800,bm25
0,запрос — заголовок объявления,0.1493,0.6100,word_tfidf
6,запрос — заголовок + описание объявления,0.1440,389.4800,char_tfidf
9,запрос + фильтры — заголовок объявления,0.1310,0.6800,bm25
2,запрос — заголовок + описание объявления,0.1277,9.8100,word_tfidf
5,запрос + фильтры — заголовок объявления,0.1237,20.0800,char_tfidf
7,запрос + фильтры — заголовок + описание объявления,0.1169,327.8000,char_tfidf


## 9. Сохранение результатов четырёх лидеров

Предыдущие эксперименты остаются выше как сравнение вариантов. Ниже повторно считаем только четыре лидирующих источника на **полной валидации** и сохраняем их индексы, исходные скоры и построенные индексы на диск. Это позволит подбирать гибридный скор без нового поиска по всему корпусу.

Коэффициенты подбираем на `queries_valid`, а не на `queries_train`: иначе оценка Recall@50 будет завышена из-за подбора на тех же кликах.

In [20]:
from pathlib import Path
import json

import joblib
from scipy.sparse import save_npz


SAVE_DIR = Path("artifacts") / "retrieval" / "validation_leaders"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = 500

saved_query_ids = queries_valid["train_query_id"].to_numpy()
saved_item_ids = np.asarray(train_items["item_id"].astype(str), dtype="U16")

np.save(SAVE_DIR / "query_ids.npy", saved_query_ids)
np.save(SAVE_DIR / "item_ids.npy", saved_item_ids)


def save_source_artifacts(name, method, item_fields, vectorizer, item_matrix, top_indices, top_scores):
    """Сохраняет всё, что нужно для повторного использования источника после перезапуска ядра."""
    joblib.dump(vectorizer, SAVE_DIR / f"{name}_vectorizer.joblib")
    save_npz(SAVE_DIR / f"{name}_item_matrix.npz", item_matrix.tocsr(), compressed=False)
    np.save(SAVE_DIR / f"{name}_top{TOP_K}_indices.npy", top_indices)
    np.save(SAVE_DIR / f"{name}_top{TOP_K}_scores.npy", top_scores)

    metadata = {
        "name": name,
        "method": method,
        "query_fields": ["search_query"],
        "item_fields": item_fields,
        "top_k": TOP_K,
        "n_queries": len(saved_query_ids),
        "n_items": len(saved_item_ids),
    }
    (SAVE_DIR / f"{name}_metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


In [21]:
query_text = queries_valid["search_query"].fillna("")
item_title_text = train_items["item_title_raw"].fillna("")
item_title_description_text = combine_text(
    train_items,
    ["item_title_raw", "item_description_raw"],
)
def run_source(name, method, item_fields, item_text, vectorizer=None, batch_size=128):
    print(f"Запуск: {name}")
    started_at = perf_counter()

    vectorizer, item_matrix, top_indices, top_scores = retrieve_top_k(
        method,
        item_text,
        query_text,
        top_k=TOP_K,
        vectorizer=vectorizer,
        batch_size=batch_size,
    )

    predictions = predictions_from_top_indices(
        top_indices,
        saved_query_ids,
        saved_item_ids,
    )
    result = {
        "source": name,
        "method": method,
        "recall_at_50": recall_at_50(
            predictions, relevant_items_valid_by_query
        ),
        "minutes": round((perf_counter() - started_at) / 60, 2),
    }

    save_source_artifacts(
        name,
        method,
        item_fields,
        vectorizer,
        item_matrix,
        top_indices,
        top_scores,
    )

    del item_matrix, top_indices, top_scores, predictions
    gc.collect()
    return result


leader_results = []

leader_results.append(
    run_source(
        name="bm25_title_description",
        method="bm25",
        item_fields=["item_title_raw", "item_description_raw"],
        item_text=item_title_description_text,
    )
)
leader_results.append(
    run_source(
        name="char_tfidf_title",
        method="char_tfidf",
        item_fields=["item_title_raw"],
        vectorizer=make_tfidf_vectorizer("char"),
        item_text=item_title_text,
        batch_size=64,
    )
)
leader_results.append(
    run_source(
        name="bm25_title",
        method="bm25",
        item_fields=["item_title_raw"],
        item_text=item_title_text,
    )
)
leader_results.append(
    run_source(
        name="word_tfidf_title",
        method="word_tfidf",
        item_fields=["item_title_raw"],
        vectorizer=make_tfidf_vectorizer("word"),
        item_text=item_title_text,
        batch_size=128,
    )
)

leader_results = pd.DataFrame(leader_results).sort_values(
    "recall_at_50",
    ascending=False,
)
leader_results.to_csv(SAVE_DIR / "leader_results.csv", index=False)
display(leader_results)


Запуск: bm25_title_description
Запуск: char_tfidf_title
Запуск: bm25_title
Запуск: word_tfidf_title


,source,method,recall_at_50,minutes
0,bm25_title_description,bm25,0.1738,8.6700
1,char_tfidf_title,char_tfidf,0.1709,10.7200
2,bm25_title,bm25,0.1575,0.5000
3,word_tfidf_title,word_tfidf,0.1495,0.5000


## 10. Гибридный скор четырёх лидеров

Загружаем сохранённые top-500 и сырые скоры. Для каждого запроса объединяем кандидатов всех четырёх источников, нормализуем скор каждого источника по его максимуму и складываем с заданными коэффициентами. Новый коэффициент не запускает TF-IDF или BM25 повторно.

In [23]:
SOURCE_NAMES = [
    "bm25_title_description",
    "char_tfidf_title",
    "bm25_title",
    "word_tfidf_title",
]

saved_top_indices = {
    name: np.load(SAVE_DIR / f"{name}_top{TOP_K}_indices.npy", mmap_mode="r")
    for name in SOURCE_NAMES
}
saved_top_scores = {
    name: np.load(SAVE_DIR / f"{name}_top{TOP_K}_scores.npy", mmap_mode="r")
    for name in SOURCE_NAMES
}

relevant_indices_by_query = make_relevant_indices_by_query(
    relevance_valid, saved_item_ids
)


def hybrid_top_indices(
    weights,
    top_n=TOP_50,
    source_indices_by_name=None,
    source_scores_by_name=None,
    source_names=SOURCE_NAMES,
    candidate_bonus=None,
):
    """Возвращает top-n: текстовый H и необязательная добавка к кандидатам."""
    weights = np.asarray(weights, dtype=np.float32)
    if len(weights) != len(source_names) or np.any(weights < 0):
        raise ValueError("Нужны неотрицательные коэффициенты для всех источников.")
    if not np.isclose(weights.sum(), 1.0):
        raise ValueError("Сумма коэффициентов должна быть равна 1.")

    if source_indices_by_name is None:
        source_indices_by_name = saved_top_indices
    if source_scores_by_name is None:
        source_scores_by_name = saved_top_scores

    n_queries = source_indices_by_name[source_names[0]].shape[0]
    result = np.full((n_queries, top_n), -1, dtype=np.int32)

    for row in range(n_queries):
        candidates, scores = source_candidates_and_scores(
            row,
            weights,
            source_indices_by_name,
            source_scores_by_name,
            source_names,
        )
        if len(candidates) == 0:
            continue

        if candidate_bonus is not None:
            scores += candidate_bonus(row, candidates)

        positions = top_positions(scores, top_n)
        result[row, :len(positions)] = candidates[positions]

    return result


# Первый контрольный запуск: равный вклад всех четырёх источников.
equal_weights = np.full(len(SOURCE_NAMES), 1 / len(SOURCE_NAMES))
equal_weight_top50 = hybrid_top_indices(equal_weights)
equal_weight_recall = recall_from_top_indices(
    equal_weight_top50, saved_query_ids, relevant_indices_by_query
)
print(f"Recall@50 равного гибрида: {equal_weight_recall:.4f}")


Recall@50 равного гибрида: 0.1776


### 10.1. Грубый подбор коэффициентов

Сначала перебираем все комбинации весов с шагом `0.25`. Сумма коэффициентов всегда равна единице. Поиск использует всю валидацию и сохранённые результаты источников; TF-IDF и BM25 повторно не вычисляются.

In [24]:
WEIGHT_COLUMNS = [f"weight_{name}" for name in SOURCE_NAMES]


def simplex_weight_grid(step):
    """Все неотрицательные веса с заданным шагом и суммой, равной единице."""
    units = int(round(1 / step))
    if not np.isclose(units * step, 1.0):
        raise ValueError("Шаг должен делить единицу без остатка.")

    grid = []
    for first in range(units + 1):
        for second in range(units - first + 1):
            for third in range(units - first - second + 1):
                fourth = units - first - second - third
                grid.append([first, second, third, fourth])

    return np.asarray(grid, dtype=np.float32) / units


def evaluate_weight_grid(weight_grid):
    """Считает Recall@50 полного validation-набора для каждой строки весов."""
    rows = []

    for number, weights in enumerate(weight_grid, start=1):
        top50 = hybrid_top_indices(weights)
        row = {
            column: float(weight)
            for column, weight in zip(WEIGHT_COLUMNS, weights)
        }
        row["recall_at_50"] = recall_from_top_indices(
            top50, saved_query_ids, relevant_indices_by_query
        )
        rows.append(row)

        weights_text = ", ".join(f"{weight:.2f}" for weight in weights)
        print(
            f"{number}/{len(weight_grid)}: Recall@50={row['recall_at_50']:.4f}; "
            f"веса=[{weights_text}]"
        )

    return pd.DataFrame(rows).sort_values("recall_at_50", ascending=False)


COARSE_STEP = 0.25
coarse_weight_grid = simplex_weight_grid(COARSE_STEP)
print(f"Количество комбинаций на грубой сетке: {len(coarse_weight_grid)}")

coarse_results = evaluate_weight_grid(coarse_weight_grid)
coarse_results.to_csv(SAVE_DIR / "hybrid_coarse_results.csv", index=False)
display(coarse_results.head(15))


Количество комбинаций на грубой сетке: 35
1/35: Recall@50=0.1502; веса=[0.00, 0.00, 0.00, 1.00]
2/35: Recall@50=0.1528; веса=[0.00, 0.00, 0.25, 0.75]
3/35: Recall@50=0.1547; веса=[0.00, 0.00, 0.50, 0.50]
4/35: Recall@50=0.1565; веса=[0.00, 0.00, 0.75, 0.25]
5/35: Recall@50=0.1579; веса=[0.00, 0.00, 1.00, 0.00]
6/35: Recall@50=0.1574; веса=[0.00, 0.25, 0.00, 0.75]
7/35: Recall@50=0.1602; веса=[0.00, 0.25, 0.25, 0.50]
8/35: Recall@50=0.1624; веса=[0.00, 0.25, 0.50, 0.25]
9/35: Recall@50=0.1645; веса=[0.00, 0.25, 0.75, 0.00]
10/35: Recall@50=0.1614; веса=[0.00, 0.50, 0.00, 0.50]
11/35: Recall@50=0.1650; веса=[0.00, 0.50, 0.25, 0.25]
12/35: Recall@50=0.1675; веса=[0.00, 0.50, 0.50, 0.00]
13/35: Recall@50=0.1672; веса=[0.00, 0.75, 0.00, 0.25]
14/35: Recall@50=0.1709; веса=[0.00, 0.75, 0.25, 0.00]
15/35: Recall@50=0.1712; веса=[0.00, 1.00, 0.00, 0.00]
16/35: Recall@50=0.1676; веса=[0.25, 0.00, 0.00, 0.75]
17/35: Recall@50=0.1696; веса=[0.25, 0.00, 0.25, 0.50]
18/35: Recall@50=0.1718; веса=[0

,weight_bm25_title_description,weight_char_tfidf_title,weight_bm25_title,weight_word_tfidf_title,recall_at_50
30,0.5000,0.5000,0.0000,0.0000,0.1945
33,0.7500,0.2500,0.0000,0.0000,0.1944
29,0.5000,0.2500,0.2500,0.0000,0.1920
32,0.7500,0.0000,0.2500,0.0000,0.1913
28,0.5000,0.2500,0.0000,0.2500,0.1900
31,0.7500,0.0000,0.0000,0.2500,0.1895
24,0.2500,0.7500,0.0000,0.0000,0.1864
27,0.5000,0.0000,0.5000,0.0000,0.1845
26,0.5000,0.0000,0.2500,0.2500,0.1841
23,0.2500,0.5000,0.2500,0.0000,0.1835


### 10.2. Уточнение вокруг лучшей комбинации

Берём лучшую точку грубой сетки и проверяем ближайшие к ней комбинации с шагом `0.05`. Так не приходится перебирать все 1 771 комбинацию четырёх весов на полной валидации.

In [25]:
best_coarse_weights = coarse_results.iloc[0][WEIGHT_COLUMNS].to_numpy(
    dtype=np.float32
)

FINE_STEP = 0.05
FINE_RADIUS = 0.10
all_fine_weights = simplex_weight_grid(FINE_STEP)
fine_weight_grid = all_fine_weights[
    np.max(np.abs(all_fine_weights - best_coarse_weights), axis=1)
    <= FINE_RADIUS + 1e-8
]

print("Лучшая грубая комбинация:", best_coarse_weights)
print(f"Количество комбинаций для уточнения: {len(fine_weight_grid)}")

fine_results = evaluate_weight_grid(fine_weight_grid)
fine_results.to_csv(SAVE_DIR / "hybrid_fine_results.csv", index=False)
display(fine_results.head(15))

best_hybrid_weights = fine_results.iloc[0][WEIGHT_COLUMNS].to_numpy(
    dtype=np.float32
)
best_hybrid_top50 = hybrid_top_indices(best_hybrid_weights)
np.save(SAVE_DIR / "best_hybrid_top50_indices.npy", best_hybrid_top50)

best_hybrid_metadata = {
    name: float(weight)
    for name, weight in zip(SOURCE_NAMES, best_hybrid_weights)
}
best_hybrid_metadata["recall_at_50"] = recall_from_top_indices(
    best_hybrid_top50, saved_query_ids, relevant_indices_by_query
)
(SAVE_DIR / "best_hybrid_metadata.json").write_text(
    json.dumps(best_hybrid_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Лучшие веса:", best_hybrid_metadata)


Лучшая грубая комбинация: [0.5 0.5 0.  0. ]
Количество комбинаций для уточнения: 25
1/25: Recall@50=0.1892; веса=[0.40, 0.40, 0.10, 0.10]
2/25: Recall@50=0.1900; веса=[0.40, 0.45, 0.05, 0.10]
3/25: Recall@50=0.1903; веса=[0.40, 0.45, 0.10, 0.05]
4/25: Recall@50=0.1903; веса=[0.40, 0.50, 0.00, 0.10]
5/25: Recall@50=0.1906; веса=[0.40, 0.50, 0.05, 0.05]
6/25: Recall@50=0.1910; веса=[0.40, 0.50, 0.10, 0.00]
7/25: Recall@50=0.1904; веса=[0.40, 0.55, 0.00, 0.05]
8/25: Recall@50=0.1910; веса=[0.40, 0.55, 0.05, 0.00]
9/25: Recall@50=0.1917; веса=[0.45, 0.40, 0.05, 0.10]
10/25: Recall@50=0.1918; веса=[0.45, 0.40, 0.10, 0.05]
11/25: Recall@50=0.1921; веса=[0.45, 0.45, 0.00, 0.10]
12/25: Recall@50=0.1922; веса=[0.45, 0.45, 0.05, 0.05]
13/25: Recall@50=0.1923; веса=[0.45, 0.45, 0.10, 0.00]
14/25: Recall@50=0.1922; веса=[0.45, 0.50, 0.00, 0.05]
15/25: Recall@50=0.1927; веса=[0.45, 0.50, 0.05, 0.00]
16/25: Recall@50=0.1923; веса=[0.45, 0.55, 0.00, 0.00]
17/25: Recall@50=0.1935; веса=[0.50, 0.40, 0.

,weight_bm25_title_description,weight_char_tfidf_title,weight_bm25_title,weight_word_tfidf_title,recall_at_50
23,0.5500,0.4000,0.0500,0.0000,0.1955
22,0.5500,0.4000,0.0000,0.0500,0.1951
24,0.5500,0.4500,0.0000,0.0000,0.1950
20,0.5000,0.4500,0.0500,0.0000,0.1948
21,0.5000,0.5000,0.0000,0.0000,0.1945
19,0.5000,0.4500,0.0000,0.0500,0.1944
18,0.5000,0.4000,0.1000,0.0000,0.1943
17,0.5000,0.4000,0.0500,0.0500,0.1940
16,0.5000,0.4000,0.0000,0.1000,0.1935
14,0.4500,0.5000,0.0500,0.0000,0.1927


Лучшие веса: {'bm25_title_description': 0.550000011920929, 'char_tfidf_title': 0.4000000059604645, 'bm25_title': 0.05000000074505806, 'word_tfidf_title': 0.0, 'recall_at_50': 0.19553071136628036}


## 11. Отдельный гибрид с бонусом за совпадение локации

Сохранённый гибрид выше остаётся базовой линией. Здесь создаём отдельную функцию: она сначала считает текстовый гибрид из четырёх источников, затем добавляет одинаковый бонус только объявлениям с `item_location_id == search_location_id` и уже после этого выбирает top-50.

Формула: `итоговый скор = текстовый гибрид + location_bonus × совпадение_локации`. Жёстко исключать другие локации нельзя: в validation 18.71% запросов не имеют ни одного релевантного объявления в точной локации.

In [26]:
# Берём уже найденные лучшие веса текстового гибрида.
best_hybrid_metadata = json.loads(
    (SAVE_DIR / "best_hybrid_metadata.json").read_text(encoding="utf-8")
)
BASE_HYBRID_WEIGHTS = np.asarray(
    [best_hybrid_metadata[name] for name in SOURCE_NAMES],
    dtype=np.float32,
)

# Строки этих массивов совпадают со строками сохранённых top-500.
valid_query_locations = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_location_id"]
    .to_numpy(dtype=np.int64)
)
item_locations = train_items["item_location_id"].to_numpy(dtype=np.int64)


def location_hybrid_top_indices(
    weights,
    bonus,
    source_indices_by_name=saved_top_indices,
    source_scores_by_name=saved_top_scores,
    top_n=TOP_50,
    query_location_ids=valid_query_locations,
    item_location_values=item_locations,
):
    """Top-n отдельного гибрида: текстовый скор плюс локационный бонус."""
    def candidate_bonus(row, candidate_indices):
        return location_bonus(
            candidate_indices,
            query_location_ids[row],
            bonus,
            item_location_values,
        )

    return hybrid_top_indices(
        weights,
        top_n=top_n,
        source_indices_by_name=source_indices_by_name,
        source_scores_by_name=source_scores_by_name,
        candidate_bonus=candidate_bonus,
    )


### 11.1. Грубый подбор локационного бонуса

Текстовые веса фиксируем по предыдущему лучшему гибриду. Перебираем только размер локационной добавки в той же шкале, что и нормализованный текстовый скор: от 0 до 1 с шагом 0.1.

In [27]:
def evaluate_location_bonuses(bonuses):
    rows = []

    for number, bonus in enumerate(bonuses, start=1):
        top50 = location_hybrid_top_indices(BASE_HYBRID_WEIGHTS, bonus)
        recall = recall_from_top_indices(
            top50, saved_query_ids, relevant_indices_by_query
        )
        rows.append({"location_bonus": float(bonus), "recall_at_50": recall})
        print(f"{number}/{len(bonuses)}: bonus={bonus:.2f}; Recall@50={recall:.4f}")

    return pd.DataFrame(rows).sort_values("recall_at_50", ascending=False)


COARSE_LOCATION_BONUSES = np.round(np.arange(0.0, 1.01, 0.10), 2)
location_coarse_results = evaluate_location_bonuses(COARSE_LOCATION_BONUSES)
location_coarse_results.to_csv(
    SAVE_DIR / "location_bonus_coarse_results.csv", index=False
)
display(location_coarse_results)


1/11: bonus=0.00; Recall@50=0.1955
2/11: bonus=0.10; Recall@50=0.2815
3/11: bonus=0.20; Recall@50=0.3429
4/11: bonus=0.30; Recall@50=0.3880
5/11: bonus=0.40; Recall@50=0.4394
6/11: bonus=0.50; Recall@50=0.4989
7/11: bonus=0.60; Recall@50=0.5325
8/11: bonus=0.70; Recall@50=0.5416
9/11: bonus=0.80; Recall@50=0.5458
10/11: bonus=0.90; Recall@50=0.5513
11/11: bonus=1.00; Recall@50=0.5522


,location_bonus,recall_at_50
10,1.0000,0.5522
9,0.9000,0.5513
8,0.8000,0.5458
7,0.7000,0.5416
6,0.6000,0.5325
5,0.5000,0.4989
4,0.4000,0.4394
3,0.3000,0.3880
2,0.2000,0.3429
1,0.1000,0.2815


### 11.2. Фиксация максимального бонуса

На грубой сетке Recall@50 монотонно вырос до максимального проверенного бонуса 1.0. Поэтому не уточняем промежуточные значения, а фиксируем максимум: при текстовом скоре от 0 до 1 он почти всегда ставит кандидатов из локации поиска выше кандидатов из других локаций.

In [28]:
best_location_bonus = 1.0
best_location_top50 = location_hybrid_top_indices(
    BASE_HYBRID_WEIGHTS, best_location_bonus
)
np.save(SAVE_DIR / "best_location_hybrid_top50_indices.npy", best_location_top50)

best_location_metadata = {
    "location_bonus": best_location_bonus,
    "recall_at_50": recall_from_top_indices(
        best_location_top50, saved_query_ids, relevant_indices_by_query
    ),
}
(SAVE_DIR / "best_location_hybrid_metadata.json").write_text(
    json.dumps(best_location_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Лучший локационный бонус:", best_location_metadata)


Лучший локационный бонус: {'location_bonus': 1.0, 'recall_at_50': 0.5522429415621417}


### 11.3. Первый файл ответа для benchmark

Для каждого запроса строим top-5 000 у каждого из четырёх текстовых источников на корпусе `benchmark_items`. Затем объединяем кандидатов, считаем тот же гибрид `H + 1 × same_location`, берём первые 50 `item_id` и сохраняем только `answer.csv`.

Векторизаторы, матрицы и промежуточные top-5 000 существуют только в оперативной памяти и удаляются сразу после записи ответа.


In [30]:
import gc
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd


BENCHMARK_TOP_K = 5_000
BENCHMARK_TOP_N = TOP_50
ANSWER_PATH = Path("answer.csv")

assert len(BASE_HYBRID_WEIGHTS) == len(SOURCE_NAMES)
assert np.isclose(BASE_HYBRID_WEIGHTS.sum(), 1.0)

benchmark_query_ids = benchmark_queries["query_id"].astype(str).to_numpy()
benchmark_item_ids = benchmark_items["item_id"].astype(str).to_numpy()
benchmark_query_text = benchmark_queries["search_query"].fillna("")
benchmark_query_locations = benchmark_queries["search_location_id"].to_numpy()
benchmark_item_locations = benchmark_items["item_location_id"].to_numpy()

assert len(benchmark_query_ids) == len(set(benchmark_query_ids))
assert len(benchmark_item_ids) == len(set(benchmark_item_ids))


def get_benchmark_source_top_k(
    name, method, item_text, batch_size, vectorizer=None
):
    """Строит один источник на benchmark_items, ничего не сохраняя на диск."""
    started_at = perf_counter()
    print(f"Запуск: {name}")

    vectorizer, item_matrix, top_indices, top_scores = retrieve_top_k(
        method,
        item_text,
        benchmark_query_text,
        top_k=BENCHMARK_TOP_K,
        vectorizer=vectorizer,
        batch_size=batch_size,
        source_name=name,
    )

    del vectorizer, item_matrix
    gc.collect()

    print(f"{name}: готово за {(perf_counter() - started_at) / 60:.2f} мин.")
    return top_indices, top_scores



In [31]:
benchmark_item_title_text = benchmark_items["item_title_raw"].fillna("")
benchmark_item_title_description_text = combine_text(
    benchmark_items,
    ["item_title_raw", "item_description_raw"],
)

benchmark_source_configs = [
    (
        "bm25_title_description",
        "bm25",
        benchmark_item_title_description_text,
        128,
        None,
    ),
    (
        "char_tfidf_title",
        "char_tfidf",
        benchmark_item_title_text,
        64,
        make_tfidf_vectorizer("char"),
    ),
    ("bm25_title", "bm25", benchmark_item_title_text, 128, None),
    (
        "word_tfidf_title",
        "word_tfidf",
        benchmark_item_title_text,
        128,
        make_tfidf_vectorizer("word"),
    ),
]

benchmark_source_top_indices = {}
benchmark_source_top_scores = {}

for name, method, item_text, batch_size, vectorizer in benchmark_source_configs:
    indices, scores = get_benchmark_source_top_k(
        name=name,
        method=method,
        item_text=item_text,
        batch_size=batch_size,
        vectorizer=vectorizer,
    )
    benchmark_source_top_indices[name] = indices
    benchmark_source_top_scores[name] = scores


def benchmark_location_hybrid_top_k(top_k=BENCHMARK_TOP_K):
    """Считает H + location_bonus × same_location для каждого benchmark-запроса."""
    return location_hybrid_top_indices(
        BASE_HYBRID_WEIGHTS,
        best_location_bonus,
        source_indices_by_name=benchmark_source_top_indices,
        source_scores_by_name=benchmark_source_top_scores,
        top_n=top_k,
        query_location_ids=benchmark_query_locations,
        item_location_values=benchmark_item_locations,
    )


benchmark_hybrid_top5000 = benchmark_location_hybrid_top_k()

answer = pd.DataFrame(
    {
        "query_id": benchmark_query_ids,
        "answer": [
            " ".join(
                benchmark_item_ids[row_indices[row_indices >= 0]].tolist()
            )
            for row_indices in benchmark_hybrid_top5000[:, :BENCHMARK_TOP_N]
        ],
    }
)
answer.to_csv(ANSWER_PATH, index=False, encoding="utf-8")

# Тестовые индексы и матрицы больше не нужны: оставляем только answer.csv.
del benchmark_source_top_indices, benchmark_source_top_scores
del benchmark_hybrid_top5000
del benchmark_item_title_text, benchmark_item_title_description_text
gc.collect()

print(f"Создан файл: {ANSWER_PATH.resolve()}")
display(answer.head())



Запуск: bm25_title_description
bm25_title_description: 128/2,452 запросов
bm25_title_description: 1,280/2,452 запросов
bm25_title_description: 2,452/2,452 запросов
bm25_title_description: готово за 0.77 мин.
Запуск: char_tfidf_title
char_tfidf_title: 64/2,452 запросов
char_tfidf_title: 640/2,452 запросов
char_tfidf_title: 1,280/2,452 запросов
char_tfidf_title: 1,920/2,452 запросов
char_tfidf_title: 2,452/2,452 запросов
char_tfidf_title: готово за 0.53 мин.
Запуск: bm25_title
bm25_title: 128/2,452 запросов
bm25_title: 1,280/2,452 запросов
bm25_title: 2,452/2,452 запросов
bm25_title: готово за 0.07 мин.
Запуск: word_tfidf_title
word_tfidf_title: 128/2,452 запросов
word_tfidf_title: 1,280/2,452 запросов
word_tfidf_title: 2,452/2,452 запросов
word_tfidf_title: готово за 0.07 мин.
Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer.csv


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 255fbeaf526a1cc1 b92ee8f432cec2d1 d722bcda1a555091 e08f4ce94c268cef deb58d7ad8c27d06 603623b4bd9e8f...
1,JTrdTaZJvSiLPkXj,16ea26ff0aa0450a 8703b2a0442c5d25 cbeccbecb1fb8d86 367af128a9ea2a48 4a435cc0a254c1fd 413f6384e3c07b64 bafe6304f743b5...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 dab52187b4500d9b 3b370cc603f67947 168a9207e80b0be4 d62074dd39caefee edecb3695ab889...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 bd2cb4500fad728e a846a2a5241e1180 4c15e7abba063ac5 3cce43c811459438 e42795fcb72962a5 1e5924dc546b82...
4,YgHcM9MVbxKnxD1e,615c73ea4c3bfd15 1a62634394544781 ba78ad593ec3412d 13f58fe9542bdb1a 9c41fe789f98d702 60e3b99111bd032b 4ef5ba70fa9baa...


In [32]:
answer_check = pd.read_csv(
    ANSWER_PATH,
    dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
answer_item_lists = answer_check["answer"].map(str.split)
benchmark_item_id_set = set(benchmark_item_ids)

assert answer_check.columns.tolist() == ["query_id", "answer"]
assert len(answer_check) == len(benchmark_query_ids)
assert answer_check["query_id"].is_unique
assert set(answer_check["query_id"]) == set(benchmark_query_ids)
assert answer_check["query_id"].str.len().eq(16).all()
assert answer_item_lists.map(len).le(BENCHMARK_TOP_N).all()
assert answer_item_lists.map(lambda ids: len(ids) == len(set(ids))).all()
assert answer_item_lists.map(
    lambda ids: set(ids).issubset(benchmark_item_id_set)
).all()

answer_check_summary = pd.DataFrame(
    [
        {
            "rows": len(answer_check),
            "min_item_ids_in_answer": answer_item_lists.map(len).min(),
            "max_item_ids_in_answer": answer_item_lists.map(len).max(),
            "all_checks_passed": True,
        }
    ]
)
display(answer_check_summary)



,rows,min_item_ids_in_answer,max_item_ids_in_answer,all_checks_passed
0,2452,50,50,True


## 12. Проверка глубины текстового пула кандидатов

Локационный гибрид выше видит только объединение top-500 каждого из четырёх текстовых источников. Одним новым расчётом сохраняем top-5000 для каждого источника, а затем из этих же сохранённых результатов сравниваем K = 500, 1 000, 3 000 и 5 000. Векторизаторы и матрицы объявлений уже сохранены: их заново не обучаем и не строим.

In [11]:
# Открываем сохранённые выдачи четырёх источников и метаданные гибрида.
# Сравнение глубин ниже не требует повторного поиска по корпусу.
import gc
import json
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np


SAVE_DIR = Path("artifacts") / "retrieval" / "validation_leaders"
TOP_K = 500
TOP_50 = 50
SOURCE_NAMES = [
    "bm25_title_description",
    "char_tfidf_title",
    "bm25_title",
    "word_tfidf_title",
]

saved_query_ids = np.load(SAVE_DIR / "query_ids.npy")
saved_item_ids = np.load(SAVE_DIR / "item_ids.npy")
assert np.array_equal(
    queries_valid["train_query_id"].to_numpy(), saved_query_ids
)
assert np.array_equal(
    train_items["item_id"].astype(str).to_numpy(), saved_item_ids
)

saved_top_indices = {
    name: np.load(
        SAVE_DIR / f"{name}_top{TOP_K}_indices.npy",
        mmap_mode="r",
    )
    for name in SOURCE_NAMES
}
saved_top_scores = {
    name: np.load(
        SAVE_DIR / f"{name}_top{TOP_K}_scores.npy",
        mmap_mode="r",
    )
    for name in SOURCE_NAMES
}
expected_saved_shape = (len(saved_query_ids), TOP_K)
for name in SOURCE_NAMES:
    assert saved_top_indices[name].shape == expected_saved_shape
    assert saved_top_scores[name].shape == expected_saved_shape

relevant_indices_by_query = make_relevant_indices_by_query(
    relevance_valid, saved_item_ids
)

best_hybrid_metadata = json.loads(
    (SAVE_DIR / "best_hybrid_metadata.json").read_text(encoding="utf-8")
)
BASE_HYBRID_WEIGHTS = np.asarray(
    [best_hybrid_metadata[name] for name in SOURCE_NAMES],
    dtype=np.float32,
)
best_location_metadata = json.loads(
    (SAVE_DIR / "best_location_hybrid_metadata.json").read_text(
        encoding="utf-8"
    )
)
best_location_bonus = float(best_location_metadata["location_bonus"])

valid_query_locations = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_location_id"]
    .to_numpy(dtype=np.int64)
)
item_locations = train_items["item_location_id"].to_numpy(dtype=np.int64)
print("Сохранённые артефакты гибрида загружены.")


Сохранённые артефакты гибрида загружены.


In [12]:
from numpy.lib.format import open_memmap
from scipy.sparse import load_npz


EXTENDED_TOP_K = 5000

# Берём порядок строк именно из сохранённых артефактов.
artifact_query_ids = np.load(SAVE_DIR / "query_ids.npy")
artifact_item_ids = np.load(SAVE_DIR / "item_ids.npy")
assert np.array_equal(artifact_query_ids, saved_query_ids)
assert np.array_equal(artifact_item_ids, saved_item_ids)

query_text_for_artifacts = (
    queries_valid.set_index("train_query_id")
    .reindex(artifact_query_ids)["search_query"]
    .fillna("")
)
assert np.array_equal(query_text_for_artifacts.index.to_numpy(), artifact_query_ids)


def extended_top_k_paths(name, top_k):
    prefix = SAVE_DIR / f"{name}_top{top_k}"
    return (
        prefix.with_name(f"{prefix.name}_indices.npy"),
        prefix.with_name(f"{prefix.name}_scores.npy"),
        prefix.with_name(f"{prefix.name}_complete.json"),
    )


def extended_top_k_is_saved(name, top_k):
    indices_path, scores_path, complete_path = extended_top_k_paths(name, top_k)
    if not all(path.exists() for path in (indices_path, scores_path, complete_path)):
        return False

    metadata = json.loads(complete_path.read_text(encoding="utf-8"))
    return (
        metadata.get("top_k") == top_k
        and metadata.get("n_queries") == len(artifact_query_ids)
        and metadata.get("n_items") == len(artifact_item_ids)
    )


def save_extended_top_k(name, method, top_k, batch_size):
    """Сохраняет top-k одного источника батчами, не держа весь результат в RAM."""
    if extended_top_k_is_saved(name, top_k):
        print(f"{name}: top-{top_k} уже сохранён.")
        return {"source": name, "status": "already_saved"}

    indices_path, scores_path, complete_path = extended_top_k_paths(name, top_k)
    started_at = perf_counter()
    vectorizer = joblib.load(SAVE_DIR / f"{name}_vectorizer.joblib")
    item_matrix = load_npz(SAVE_DIR / f"{name}_item_matrix.npz").tocsr()
    item_matrix_t = item_matrix.T

    top_indices = open_memmap(
        indices_path,
        mode="w+",
        dtype=np.int32,
        shape=(len(artifact_query_ids), top_k),
    )
    top_scores = open_memmap(
        scores_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(artifact_query_ids), top_k),
    )
    top_indices[:] = -1
    top_scores[:] = -np.inf

    for start in range(0, len(artifact_query_ids), batch_size):
        stop = min(start + batch_size, len(artifact_query_ids))
        query_batch = vectorizer.transform(query_text_for_artifacts.iloc[start:stop])

        if method == "bm25":
            query_batch = query_batch.astype(np.float32)
            query_batch.data.fill(1.0)

        batch_scores = (query_batch @ item_matrix_t).tocsr()
        write_top_k_from_sparse_scores(
            batch_scores, top_indices, top_scores, start
        )

        del query_batch, batch_scores
        if stop == len(artifact_query_ids) or start % (batch_size * 50) == 0:
            print(f"{name}: {stop:,}/{len(artifact_query_ids):,} запросов")

    top_indices.flush()
    top_scores.flush()
    del top_indices, top_scores, vectorizer, item_matrix, item_matrix_t
    gc.collect()

    result = {
        "source": name,
        "status": "calculated",
        "top_k": top_k,
        "n_queries": len(artifact_query_ids),
        "n_items": len(artifact_item_ids),
        "minutes": round((perf_counter() - started_at) / 60, 2),
    }
    complete_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return result


source_methods = {
    name: json.loads(
        (SAVE_DIR / f"{name}_metadata.json").read_text(encoding="utf-8")
    )["method"]
    for name in SOURCE_NAMES
}

extended_top_k_results = []
for name in SOURCE_NAMES:
    batch_size = 64 if name == "char_tfidf_title" else 128
    extended_top_k_results.append(
        save_extended_top_k(
            name,
            source_methods[name],
            EXTENDED_TOP_K,
            batch_size,
        )
    )

extended_top_k_results = pd.DataFrame(extended_top_k_results)
extended_top_k_results.to_csv(
    SAVE_DIR / f"extended_top{EXTENDED_TOP_K}_calculation.csv",
    index=False,
)
display(extended_top_k_results)


bm25_title_description: top-5000 уже сохранён.
char_tfidf_title: top-5000 уже сохранён.
bm25_title: top-5000 уже сохранён.
word_tfidf_title: top-5000 уже сохранён.


,source,status
0,bm25_title_description,already_saved
1,char_tfidf_title,already_saved
2,bm25_title,already_saved
3,word_tfidf_title,already_saved


In [13]:
# Только открываем уже сохранённые top-5000, ничего не пересчитываем.
extended_top_indices = {}
extended_top_scores = {}

for name in SOURCE_NAMES:
    indices_path, scores_path, _ = extended_top_k_paths(
        name, EXTENDED_TOP_K
    )
    extended_top_indices[name] = np.load(
        indices_path, mmap_mode="r"
    )
    extended_top_scores[name] = np.load(
        scores_path, mmap_mode="r"
    )
    expected_shape = (len(saved_query_ids), EXTENDED_TOP_K)
    assert extended_top_indices[name].shape == expected_shape
    assert extended_top_scores[name].shape == expected_shape

best_depth_metadata = json.loads(
    (SAVE_DIR / "best_extended_location_hybrid_metadata.json")
    .read_text(encoding="utf-8")
)
assert int(best_depth_metadata["top_k_per_source"]) == EXTENDED_TOP_K


def base_hybrid_candidates_and_scores(row):
    """Кандидаты и скор H с бонусом за совпадение локации."""
    candidates, scores = source_candidates_and_scores(
        row,
        BASE_HYBRID_WEIGHTS,
        extended_top_indices,
        extended_top_scores,
    )
    scores += location_bonus(
        candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    return candidates, scores


def recall_for_item_indices(row, item_indices):
    relevant = relevant_indices_by_query[saved_query_ids[row]]
    found = sum(int(item_index) in relevant for item_index in item_indices)
    return found / len(relevant)


print("Сохранённые top-5000 источники загружены.")


Сохранённые top-5000 источники загружены.


### 12.1. Сравнение глубин 500, 1 000, 3 000 и 5 000

Для каждой глубины считаем две величины. `candidate_pool_recall` — верхняя граница: какая доля релевантных объявлений вообще попала в объединение четырёх источников. `location_hybrid_recall_at_50` — фактический Recall@50 после той же текстово-локационной формулы. Если первая величина растёт, а вторая нет, проблема уже не в глубине пула, а в ранжировании внутри него.

In [17]:
def candidate_pool_recall(source_indices_by_name):
    """Доля релевантных объявлений, видимых хотя бы одному источнику."""
    recall_values = []

    for row, query_id in enumerate(saved_query_ids):
        parts = []
        for name in SOURCE_NAMES:
            indices = source_indices_by_name[name][row]
            if np.any(indices >= 0):
                parts.append(indices[indices >= 0])

        if not parts:
            recall_values.append(0.0)
            continue

        candidates = np.unique(np.concatenate(parts))
        relevant = np.fromiter(
            relevant_indices_by_query[query_id],
            dtype=np.int32,
        )
        recall_values.append(np.isin(relevant, candidates).mean())

    return float(np.mean(recall_values))


DEPTHS_TO_COMPARE = [500, 1000, 3000, 5000]
assert max(DEPTHS_TO_COMPARE) <= EXTENDED_TOP_K
depth_rows = []
top50_by_depth = {}

for depth in DEPTHS_TO_COMPARE:
    if depth == TOP_K:
        source_indices_for_depth = saved_top_indices
        source_scores_for_depth = saved_top_scores
    else:
        source_indices_for_depth = {
            name: extended_top_indices[name][:, :depth]
            for name in SOURCE_NAMES
        }
        source_scores_for_depth = {
            name: extended_top_scores[name][:, :depth]
            for name in SOURCE_NAMES
        }

    top50 = location_hybrid_top_indices(
        BASE_HYBRID_WEIGHTS,
        best_location_bonus,
        source_indices_by_name=source_indices_for_depth,
        source_scores_by_name=source_scores_for_depth,
    )
    top50_by_depth[depth] = top50
    depth_rows.append(
        {
            "top_k_per_source": depth,
            "candidate_pool_recall": candidate_pool_recall(
                source_indices_for_depth
            ),
            "location_hybrid_recall_at_50": recall_from_top_indices(
                top50, saved_query_ids, relevant_indices_by_query
            ),
        }
    )

depth_results = pd.DataFrame(depth_rows).sort_values("top_k_per_source")
depth_results.to_csv(SAVE_DIR / "candidate_depth_results.csv", index=False)
display(depth_results)

best_depth_row = depth_results.loc[
    depth_results["location_hybrid_recall_at_50"].idxmax()
]
best_depth = int(best_depth_row["top_k_per_source"])
np.save(
    SAVE_DIR / "best_extended_location_hybrid_top50_indices.npy",
    top50_by_depth[best_depth],
)

best_depth_metadata = {
    "top_k_per_source": best_depth,
    "location_bonus": float(best_location_bonus),
    "candidate_pool_recall": float(best_depth_row["candidate_pool_recall"]),
    "recall_at_50": float(best_depth_row["location_hybrid_recall_at_50"]),
}
(SAVE_DIR / "best_extended_location_hybrid_metadata.json").write_text(
    json.dumps(best_depth_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Лучшая глубина:", best_depth_metadata)


,top_k_per_source,candidate_pool_recall,location_hybrid_recall_at_50
0,500,0.6470,0.5522
1,1000,0.7472,0.6306
2,3000,0.8558,0.7079
3,5000,0.8861,0.7238


Лучшая глубина: {'top_k_per_source': 5000, 'location_bonus': 1.0, 'candidate_pool_recall': 0.8860543991728579, 'recall_at_50': 0.723775184449846}


## 13. Семантический поиск по эмбеддингам

Используем локальную модель `intfloat/multilingual-e5-base`. Запрос кодируем как `query: search_query`, объявление — как `passage: item_title_raw + item_description_raw`. Параметры объявления сюда не добавляем: их проверим отдельным источником.

Модель при первом запуске загружается и сохраняется в `artifacts/models`; следующие запуски используют локальную копию. Эмбеддинги, top-5000 кандидатов и исходные cosine-скоры также сохраняются на диск.

### 13.1. Подготовка исходных текстов

Для объявлений повторно читаем только исходные заголовок и описание из Parquet, потому что тексты в рабочем датафрейме уже нормализованы. Одинаковые тексты запросов кодируем один раз, затем восстанавливаем результаты для всех `train_query_id`.

In [43]:
import faiss
import torch
from sentence_transformers import SentenceTransformer

print("Импорты работают")

Импорты работают


In [44]:
import os

import faiss
import torch
from numpy.lib.format import open_memmap
from sentence_transformers import SentenceTransformer


EMBEDDING_SOURCE_NAME = "multilingual_e5_title_description"
EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-base"
EMBEDDING_TOP_K = 5000
MODEL_DIR = Path("artifacts") / "models" / "multilingual-e5-base"
EMBEDDING_DIR = SAVE_DIR / EMBEDDING_SOURCE_NAME
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

raw_text_data = pd.read_parquet(
    "data/train.parquet",
    columns=[
        "item_id",
        "item_title_raw",
        "item_description_raw",
        "search_query",
    ],
    engine="pyarrow",
    dtype_backend="pyarrow",
)

raw_items = raw_text_data[
    ["item_id", "item_title_raw", "item_description_raw"]
].drop_duplicates("item_id").copy()
raw_items["item_id"] = raw_items["item_id"].astype(str)
raw_items["row_found"] = True
raw_items = raw_items.set_index("item_id").reindex(saved_item_ids)
assert not raw_items["row_found"].isna().any()

item_titles = raw_items["item_title_raw"].fillna("").astype(str).str.strip()
item_descriptions = (
    raw_items["item_description_raw"].fillna("").astype(str).str.strip()
)
embedding_item_texts = (
    "passage: " + item_titles + ". " + item_descriptions
).str.strip().to_numpy(dtype=str)

raw_query_variants = pd.DataFrame(
    {
        "raw_query": raw_text_data["search_query"].fillna("").astype(str),
        "normalized_query": normalize_text(
            raw_text_data["search_query"]
        ).astype(str),
    }
).drop_duplicates("normalized_query")
raw_query_by_normalized = raw_query_variants.set_index(
    "normalized_query"
)["raw_query"]

ordered_valid_queries = (
    queries_valid.set_index("train_query_id").reindex(saved_query_ids)
)
normalized_valid_queries = (
    ordered_valid_queries["search_query"].fillna("").astype(str)
)
raw_valid_queries = normalized_valid_queries.map(raw_query_by_normalized)
raw_valid_queries = raw_valid_queries.fillna(normalized_valid_queries)
embedding_query_texts = (
    "query: " + raw_valid_queries.str.strip()
).to_numpy(dtype=str)

unique_query_texts, query_to_unique = np.unique(
    embedding_query_texts,
    return_inverse=True,
)
query_to_unique = query_to_unique.astype(np.int32)
np.save(EMBEDDING_DIR / "query_to_unique.npy", query_to_unique)
np.save(EMBEDDING_DIR / "unique_query_texts.npy", unique_query_texts)

del raw_text_data, raw_items, raw_query_variants
gc.collect()

print("Объявлений:", len(embedding_item_texts))
print("Строк validation:", len(embedding_query_texts))
print("Уникальных текстов запросов:", len(unique_query_texts))


Объявлений: 344825
Строк validation: 67186
Уникальных текстов запросов: 14741


### 13.2. Вычисление и сохранение эмбеддингов

Модель сохраняем локально. Эмбеддинги вычисляются батчами и сразу записываются в `.npy`; завершённый массив повторно не рассчитывается.

In [45]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(4)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

ITEM_MAX_SEQ_LENGTH = 128
ITEM_BATCH_SIZE = 8
QUERY_MAX_SEQ_LENGTH = 128
QUERY_BATCH_SIZE = 64
ENCODE_CHUNK_SIZE = 2048

model_source = (
    MODEL_DIR
    if (MODEL_DIR / "modules.json").exists()
    else EMBEDDING_MODEL_ID
)
embedding_model = SentenceTransformer(str(model_source), device=DEVICE)
if model_source == EMBEDDING_MODEL_ID:
    embedding_model.save(str(MODEL_DIR))

if DEVICE == "cuda":
    embedding_model.half()

EMBEDDING_DIM = int(embedding_model.get_embedding_dimension())
print("Устройство:", DEVICE)
print("Размерность эмбеддинга:", EMBEDDING_DIM)
print("Параметры объявлений:", ITEM_MAX_SEQ_LENGTH, ITEM_BATCH_SIZE)
print("Параметры запросов:", QUERY_MAX_SEQ_LENGTH, QUERY_BATCH_SIZE)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Устройство: cpu
Размерность эмбеддинга: 768
Параметры объявлений: 128 8
Параметры запросов: 128 64


In [46]:
def save_embeddings(
    texts,
    output_path,
    batch_size,
    max_seq_length,
    chunk_size=ENCODE_CHUNK_SIZE,
):
    complete_path = output_path.with_name(f"{output_path.stem}_complete.json")
    progress_path = output_path.with_name(f"{output_path.stem}_progress.json")
    progress_temp_path = progress_path.with_suffix(".tmp")
    expected_metadata = {
        "model": EMBEDDING_MODEL_ID,
        "rows": len(texts),
        "dimension": EMBEDDING_DIM,
        "max_seq_length": max_seq_length,
        "normalized": True,
    }

    if output_path.exists() and complete_path.exists():
        saved = np.load(output_path, mmap_mode="r")
        saved_metadata = json.loads(
            complete_path.read_text(encoding="utf-8")
        )
        metadata_matches = all(
            saved_metadata.get(key) == value
            for key, value in expected_metadata.items()
        )
        if saved.shape == (len(texts), EMBEDDING_DIM) and metadata_matches:
            print(f"{output_path.name}: уже сохранён.")
            return saved
        del saved

    next_row = 0
    can_resume = output_path.exists() and progress_path.exists()
    if can_resume:
        progress = json.loads(progress_path.read_text(encoding="utf-8"))
        can_resume = all(
            progress.get(key) == value
            for key, value in expected_metadata.items()
        )
        if can_resume:
            next_row = int(progress.get("next_row", 0))
            result = np.load(output_path, mmap_mode="r+")
            can_resume = (
                result.shape == (len(texts), EMBEDDING_DIM)
                and 0 <= next_row <= len(texts)
            )
            if not can_resume:
                del result

    if can_resume:
        print(f"{output_path.stem}: продолжаем со строки {next_row:,}.")
    else:
        next_row = 0
        result = open_memmap(
            output_path,
            mode="w+",
            dtype=np.float32,
            shape=(len(texts), EMBEDDING_DIM),
        )

    embedding_model.max_seq_length = max_seq_length
    started_at = perf_counter()

    for start in range(next_row, len(texts), chunk_size):
        stop = min(start + chunk_size, len(texts))
        vectors = embedding_model.encode(
            texts[start:stop].tolist(),
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        result[start:stop] = np.asarray(vectors, dtype=np.float32)
        result.flush()

        progress = {**expected_metadata, "next_row": stop}
        progress_temp_path.write_text(
            json.dumps(progress, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        progress_temp_path.replace(progress_path)
        print(f"{output_path.stem}: {stop:,}/{len(texts):,}")

    result.flush()
    del result
    metadata = {
        **expected_metadata,
        "batch_size": batch_size,
        "chunk_size": chunk_size,
        "minutes_this_run": round(
            (perf_counter() - started_at) / 60, 2
        ),
    }
    complete_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    progress_path.unlink(missing_ok=True)
    return np.load(output_path, mmap_mode="r")


item_embeddings = save_embeddings(
    embedding_item_texts,
    EMBEDDING_DIR / "item_embeddings_maxlen128.npy",
    ITEM_BATCH_SIZE,
    ITEM_MAX_SEQ_LENGTH,
)
unique_query_embeddings = save_embeddings(
    unique_query_texts,
    EMBEDDING_DIR / "unique_query_embeddings_maxlen128.npy",
    QUERY_BATCH_SIZE,
    QUERY_MAX_SEQ_LENGTH,
)

del embedding_model, embedding_item_texts, embedding_query_texts
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Batches:   0%|          | 0/256 [00:00<?, ?it/s]

item_embeddings_maxlen128: 2,048/344,825


Batches:   0%|          | 0/256 [00:00<?, ?it/s]

item_embeddings_maxlen128: 4,096/344,825


Batches:   0%|          | 0/256 [00:00<?, ?it/s]

...

unique_query_embeddings_maxlen128: 12,288/14,741


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

unique_query_embeddings_maxlen128: 14,336/14,741


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

unique_query_embeddings_maxlen128: 14,741/14,741


### 13.3. Индекс семантического поиска

Нормализованные эмбеддинги индексируем по cosine similarity через локальный HNSW-индекс FAISS. Индекс строится один раз и сохраняется на диск.

In [47]:
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 10000
FAISS_ADD_BATCH_SIZE = 5000
FAISS_SEARCH_BATCH_SIZE = 128

faiss.omp_set_num_threads(max(1, (os.cpu_count() or 2) - 1))
index_path = EMBEDDING_DIR / "hnsw_cosine.index"
index_complete_path = EMBEDDING_DIR / "hnsw_cosine_complete.json"

if index_path.exists() and index_complete_path.exists():
    embedding_index = faiss.read_index(str(index_path))
    assert embedding_index.ntotal == len(saved_item_ids)
    print("FAISS-индекс загружен с диска.")
else:
    started_at = perf_counter()
    embedding_index = faiss.IndexHNSWFlat(
        EMBEDDING_DIM,
        HNSW_M,
        faiss.METRIC_INNER_PRODUCT,
    )
    embedding_index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION

    for start in range(0, len(item_embeddings), FAISS_ADD_BATCH_SIZE):
        stop = min(start + FAISS_ADD_BATCH_SIZE, len(item_embeddings))
        batch = np.ascontiguousarray(
            item_embeddings[start:stop],
            dtype=np.float32,
        )
        embedding_index.add(batch)
        print(f"Индекс: {stop:,}/{len(item_embeddings):,} объявлений")

    faiss.write_index(embedding_index, str(index_path))
    index_complete_path.write_text(
        json.dumps(
            {
                "items": int(embedding_index.ntotal),
                "dimension": EMBEDDING_DIM,
                "hnsw_m": HNSW_M,
                "ef_construction": HNSW_EF_CONSTRUCTION,
                "minutes": round((perf_counter() - started_at) / 60, 2),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

embedding_index.hnsw.efSearch = HNSW_EF_SEARCH


Индекс: 5,000/344,825 объявлений
Индекс: 10,000/344,825 объявлений
Индекс: 15,000/344,825 объявлений
Индекс: 20,000/344,825 объявлений
Индекс: 25,000/344,825 объявлений
Индекс: 30,000/344,825 объявлений
Индекс: 35,000/344,825 объявлений
Индекс: 40,000/344,825 объявлений
Индекс: 45,000/344,825 объявлений
Индекс: 50,000/344,825 объявлений
Индекс: 55,000/344,825 объявлений
Индекс: 60,000/344,825 объявлений
Индекс: 65,000/344,825 объявлений
Индекс: 70,000/344,825 объявлений
Индекс: 75,000/344,825 объявлений
Индекс: 80,000/344,825 объявлений
Индекс: 85,000/344,825 объявлений
Индекс: 90,000/344,825 объявлений
Индекс: 95,000/344,825 объявлений
Индекс: 100,000/344,825 объявлений
Индекс: 105,000/344,825 объявлений
Индекс: 110,000/344,825 объявлений
Индекс: 115,000/344,825 объявлений
Индекс: 120,000/344,825 объявлений
Индекс: 125,000/344,825 объявлений
Индекс: 130,000/344,825 объявлений
Индекс: 135,000/344,825 объявлений
Индекс: 140,000/344,825 объявлений
Индекс: 145,000/344,825 объявлений
Индек

### 13.4. Top-5000 кандидатов и Recall

Сначала ищем соседей только для уникальных текстов запросов, затем разворачиваем результаты в порядок `saved_query_ids`. Сохраняем номера строк объявлений и исходные cosine-скоры — в том же формате, что и у остальных источников гибрида.

In [48]:
unique_indices_path = (
    EMBEDDING_DIR / f"unique_top{EMBEDDING_TOP_K}_indices.npy"
)
unique_scores_path = (
    EMBEDDING_DIR / f"unique_top{EMBEDDING_TOP_K}_scores.npy"
)
unique_complete_path = (
    EMBEDDING_DIR / f"unique_top{EMBEDDING_TOP_K}_complete.json"
)

unique_search_is_saved = (
    unique_indices_path.exists()
    and unique_scores_path.exists()
    and unique_complete_path.exists()
)
if unique_search_is_saved:
    saved_unique_indices = np.load(unique_indices_path, mmap_mode="r")
    saved_unique_scores = np.load(unique_scores_path, mmap_mode="r")
    saved_unique_metadata = json.loads(
        unique_complete_path.read_text(encoding="utf-8")
    )
    expected_unique_shape = (
        len(unique_query_embeddings), EMBEDDING_TOP_K
    )
    unique_search_is_saved = (
        saved_unique_indices.shape == expected_unique_shape
        and saved_unique_scores.shape == expected_unique_shape
        and saved_unique_metadata.get("top_k") == EMBEDDING_TOP_K
        and saved_unique_metadata.get("ef_search") == HNSW_EF_SEARCH
    )
    del saved_unique_indices, saved_unique_scores, saved_unique_metadata

if not unique_search_is_saved:
    unique_top_indices = open_memmap(
        unique_indices_path,
        mode="w+",
        dtype=np.int32,
        shape=(len(unique_query_embeddings), EMBEDDING_TOP_K),
    )
    unique_top_scores = open_memmap(
        unique_scores_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(unique_query_embeddings), EMBEDDING_TOP_K),
    )

    started_at = perf_counter()
    for start in range(
        0, len(unique_query_embeddings), FAISS_SEARCH_BATCH_SIZE
    ):
        stop = min(
            start + FAISS_SEARCH_BATCH_SIZE,
            len(unique_query_embeddings),
        )
        query_batch = np.ascontiguousarray(
            unique_query_embeddings[start:stop],
            dtype=np.float32,
        )
        scores, indices = embedding_index.search(
            query_batch, EMBEDDING_TOP_K
        )
        unique_top_indices[start:stop] = indices.astype(np.int32)
        unique_top_scores[start:stop] = scores.astype(np.float32)

        if (
            stop == len(unique_query_embeddings)
            or start % (FAISS_SEARCH_BATCH_SIZE * 20) == 0
        ):
            print(
                f"Поиск: {stop:,}/{len(unique_query_embeddings):,} запросов"
            )

    unique_top_indices.flush()
    unique_top_scores.flush()
    del unique_top_indices, unique_top_scores

    unique_complete_path.write_text(
        json.dumps(
            {
                "top_k": EMBEDDING_TOP_K,
                "unique_queries": len(unique_query_embeddings),
                "ef_search": HNSW_EF_SEARCH,
                "minutes": round(
                    (perf_counter() - started_at) / 60, 2
                ),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    print(f"Top-{EMBEDDING_TOP_K} для уникальных запросов уже сохранён.")

del embedding_index
gc.collect()


Поиск: 128/14,741 запросов
Поиск: 2,688/14,741 запросов
Поиск: 5,248/14,741 запросов
Поиск: 7,808/14,741 запросов
Поиск: 10,368/14,741 запросов
Поиск: 12,928/14,741 запросов
Поиск: 14,741/14,741 запросов


33

In [49]:
unique_top_indices = np.load(unique_indices_path, mmap_mode="r")
unique_top_scores = np.load(unique_scores_path, mmap_mode="r")

embedding_indices_path = (
    SAVE_DIR
    / f"{EMBEDDING_SOURCE_NAME}_top{EMBEDDING_TOP_K}_indices.npy"
)
embedding_scores_path = (
    SAVE_DIR
    / f"{EMBEDDING_SOURCE_NAME}_top{EMBEDDING_TOP_K}_scores.npy"
)
embedding_complete_path = (
    SAVE_DIR
    / f"{EMBEDDING_SOURCE_NAME}_top{EMBEDDING_TOP_K}_complete.json"
)

expanded_search_is_saved = (
    embedding_indices_path.exists()
    and embedding_scores_path.exists()
    and embedding_complete_path.exists()
)
if expanded_search_is_saved:
    saved_embedding_indices = np.load(
        embedding_indices_path, mmap_mode="r"
    )
    saved_embedding_scores = np.load(
        embedding_scores_path, mmap_mode="r"
    )
    expected_embedding_shape = (len(saved_query_ids), EMBEDDING_TOP_K)
    expanded_search_is_saved = (
        saved_embedding_indices.shape == expected_embedding_shape
        and saved_embedding_scores.shape == expected_embedding_shape
    )
    del saved_embedding_indices, saved_embedding_scores

if not expanded_search_is_saved:
    embedding_top_indices = open_memmap(
        embedding_indices_path,
        mode="w+",
        dtype=np.int32,
        shape=(len(saved_query_ids), EMBEDDING_TOP_K),
    )
    embedding_top_scores = open_memmap(
        embedding_scores_path,
        mode="w+",
        dtype=np.float32,
        shape=(len(saved_query_ids), EMBEDDING_TOP_K),
    )

    for start in range(0, len(saved_query_ids), 256):
        stop = min(start + 256, len(saved_query_ids))
        unique_rows = query_to_unique[start:stop]
        embedding_top_indices[start:stop] = (
            unique_top_indices[unique_rows]
        )
        embedding_top_scores[start:stop] = (
            unique_top_scores[unique_rows]
        )

    embedding_top_indices.flush()
    embedding_top_scores.flush()
    del embedding_top_indices, embedding_top_scores

    embedding_complete_path.write_text(
        json.dumps(
            {
                "source": EMBEDDING_SOURCE_NAME,
                "model": EMBEDDING_MODEL_ID,
                "query_fields": ["search_query"],
                "item_fields": [
                    "item_title_raw",
                    "item_description_raw",
                ],
                "top_k": EMBEDDING_TOP_K,
                "queries": len(saved_query_ids),
                "items": len(saved_item_ids),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    print(f"Полные массивы top-{EMBEDDING_TOP_K} уже сохранены.")

embedding_top_indices = np.load(
    embedding_indices_path, mmap_mode="r"
)
embedding_top_scores = np.load(
    embedding_scores_path, mmap_mode="r"
)


In [50]:
expected_shape = (len(saved_query_ids), EMBEDDING_TOP_K)
assert embedding_top_indices.shape == expected_shape
assert embedding_top_scores.shape == expected_shape

total_relevant = sum(
    len(relevant_indices_by_query[query_id])
    for query_id in saved_query_ids
)


def embedding_recall_summary(top_k):
    found_relevant = 0
    recall_by_query = []

    for row, query_id in enumerate(saved_query_ids):
        predicted = set(embedding_top_indices[row, :top_k])
        predicted.discard(-1)
        relevant = relevant_indices_by_query[query_id]
        found_for_query = len(predicted & relevant)
        found_relevant += found_for_query
        recall_by_query.append(found_for_query / len(relevant))

    return {
        "top_k": top_k,
        "found_relevant": found_relevant,
        "total_relevant": total_relevant,
        "found_relevant_pct": 100 * found_relevant / total_relevant,
        "recall_at_k": float(np.mean(recall_by_query)),
    }


embedding_depths = [50, 100, 2000, 3000, 5000]
assert max(embedding_depths) <= embedding_top_indices.shape[1]

embedding_results = pd.DataFrame(
    [
        embedding_recall_summary(top_k)
        for top_k in embedding_depths
    ]
)
embedding_results.to_csv(
    SAVE_DIR / "embedding_validation_results.csv",
    index=False,
)
display(embedding_results)


,top_k,found_relevant,total_relevant,found_relevant_pct,recall_at_k
0,50,15425,88871,17.3566,0.1968
1,100,21970,88871,24.7212,0.2751
2,2000,64756,88871,72.8652,0.7512
3,3000,68806,88871,77.4223,0.7948
4,5000,73265,88871,82.4397,0.8402


## 14. Анализ фильтров поиска

Перед применением фильтра измеряем его верхнюю границу качества: какую долю уже известных релевантных объявлений он сохранил бы. Если жёсткое условие удаляет релевантные ответы, ранжирование уже не сможет их вернуть.

In [34]:
filter_pairs = (
    relevance_valid
    .merge(
        queries_valid[["train_query_id", "search_infm_params_text"]],
        on="train_query_id",
    )
    .merge(
        train_items[["item_id", "item_infm_params_text"]],
        on="item_id",
    )
)

query_param_tokens = filter_pairs["search_infm_params_text"].str.split().map(set)
item_param_tokens = filter_pairs["item_infm_params_text"].str.split().map(set)
has_filter = filter_pairs["search_infm_params_text"].ne("")

filter_pairs["param_token_coverage"] = [
    len(query_tokens & item_tokens) / len(query_tokens)
    if query_tokens else 1.0
    for query_tokens, item_tokens in zip(query_param_tokens, item_param_tokens)
]
filtered_query_ids = queries_valid.loc[
    queries_valid["search_infm_params_text"].ne(""),
    "train_query_id",
]

filter_rows = []
for threshold in [0.5, 0.7, 0.8, 0.9, 1.0]:
    kept = ~has_filter | (filter_pairs["param_token_coverage"] >= threshold)
    recall_by_query = (
        filter_pairs.assign(kept=kept)
        .groupby("train_query_id")["kept"]
        .mean()
    )
    filter_rows.append(
        {
            "token_coverage_threshold": threshold,
            "relevant_pairs_kept_with_filter": kept[has_filter].mean(),
            "recall_ceiling_filtered_queries": recall_by_query.reindex(
                filtered_query_ids
            ).mean(),
            "recall_ceiling_all_queries": recall_by_query.mean(),
        }
    )

filter_overview = pd.DataFrame(
    {
        "dataset": ["validation", "benchmark"],
        "queries": [len(queries_valid), len(benchmark_queries)],
        "queries_with_filters": [
            queries_valid["search_infm_params_text"].ne("").sum(),
            benchmark_queries["search_infm_params_text"].ne("").sum(),
        ],
    }
)
display(filter_overview)
display(pd.DataFrame(filter_rows).round(4))


,dataset,queries,queries_with_filters
0,validation,67186,43049
1,benchmark,2452,846


,token_coverage_threshold,relevant_pairs_kept_with_filter,recall_ceiling_filtered_queries,recall_ceiling_all_queries
0,0.5000,0.9675,0.9578,0.9729
1,0.7000,0.9218,0.9106,0.9427
2,0.8000,0.9155,0.9027,0.9377
3,0.9000,0.9124,0.8991,0.9354
4,1.0000,0.9119,0.8983,0.9349


### Вывод по жёсткому фильтру

Порог покрытия 90% сохраняет только 89.91% релевантных объявлений у запросов с фильтрами. Даже с учётом запросов без фильтров верхняя граница Recall составляет 0.9354. Поэтому `search_infm_params_text` нельзя использовать как обязательное условие: совпадение параметров будем добавлять к гибридному скору как отдельный бонус, сохраняя кандидатов без совпадения.

### 14.1. Размер отдельного пула по параметрам

Проверяем на benchmark-корпусе, сколько объявлений проходит по токенам каждого уникального фильтра. Это не предсказание разметки, а оценка размера будущего источника кандидатов.

In [35]:
unique_filters = pd.Index(
    benchmark_queries.loc[
        benchmark_queries["search_infm_params_text"].ne(""),
        "search_infm_params_text",
    ].unique()
)
item_param_matrix, filter_param_matrix, filter_token_counts = (
    build_filter_token_matrices(
        benchmark_items["item_infm_params_text"],
        unique_filters,
    )
)

thresholds = [0.8, 0.9, 1.0]
match_counts = {threshold: [] for threshold in thresholds}
for start in range(0, len(unique_filters), 16):
    stop = min(start + 16, len(unique_filters))
    overlap = (
        filter_param_matrix[start:stop] @ item_param_matrix.T
    ).tocsr()
    for row in range(overlap.shape[0]):
        values = overlap.data[overlap.indptr[row]:overlap.indptr[row + 1]]
        token_count = filter_token_counts[start + row]
        for threshold in thresholds:
            required = np.ceil(threshold * token_count)
            match_counts[threshold].append(int((values >= required).sum()))

filter_pool_rows = []
for threshold, counts in match_counts.items():
    counts = np.asarray(counts)
    filter_pool_rows.append(
        {
            "threshold": threshold,
            "min": counts.min(),
            "p25": np.quantile(counts, 0.25),
            "median": np.median(counts),
            "p75": np.quantile(counts, 0.75),
            "p90": np.quantile(counts, 0.90),
            "max": counts.max(),
            "filters_with_fewer_than_50_items": (counts < 50).sum(),
            "filters_with_more_than_5000_items": (counts > 5000).sum(),
        }
    )
display(pd.DataFrame(filter_pool_rows))

del item_param_matrix, filter_param_matrix
gc.collect()


,threshold,min,p25,median,p75,p90,max,filters_with_fewer_than_50_items,filters_with_more_than_5000_items
0,0.8000,0,464.0000,"1,459.0000","3,156.0000","7,975.0000",21404,16,18
1,0.9000,0,373.0000,"1,066.0000","2,714.0000","5,275.4000",21404,16,13
2,1.0000,0,373.0000,"1,066.0000","2,590.5000","5,004.2000",21404,16,13


3346

При полном совпадении медианный фильтр оставляет 1 066 объявлений, p90 — 5 004, максимум — 21 404. Значит, параметры подходят как отдельная ветка кандидатогенерации, но не дают готовый top-50: внутри этого пула всё равно потребуется текстовый гибрид, локационный бонус и обязательное дополнение глобальными кандидатами.

### 14.2. BM25-источник: фильтры запроса → параметры объявления

Отдельно ищем по `search_infm_params_text` в `item_infm_params_text` по всему корпусу. Проверяем только полноту среди запросов с непустым фильтром: что даёт этот источник сам по себе и сколько релевантных объявлений он добавляет к объединению текущих четырёх top-5 000 источников. Результаты не сохраняются на диск.

In [19]:
import gc


def text_pool_relevant_mask(row, relevant):
    mask = np.zeros(len(relevant), dtype=bool)
    for name in SOURCE_NAMES:
        mask |= np.isin(relevant, extended_top_indices[name][row])
    return mask


def evaluate_bm25_candidate_source(
    query_text, item_text, source_name, top_k=EXTENDED_TOP_K, batch_size=32
):
    """Проверяет, что BM25-источник добавляет к текущему пулу."""
    query_text = pd.Series(query_text).fillna("").astype(str).reset_index(drop=True)
    assert len(query_text) == len(saved_query_ids)

    unique_texts = pd.Index(query_text.loc[query_text.ne("")].unique())
    text_codes = unique_texts.get_indexer(query_text)
    rows_by_text = [
        np.flatnonzero(text_codes == code) for code in range(len(unique_texts))
    ]
    _, query_matrix, item_matrix = build_retrieval_matrices(
        "bm25", item_text.fillna(""), unique_texts
    )
    item_matrix_t = item_matrix.T
    text_sum = source_sum = union_sum = 0.0

    for start in range(0, len(unique_texts), batch_size):
        stop = min(start + batch_size, len(unique_texts))
        batch_scores = (query_matrix[start:stop] @ item_matrix_t).tocsr()

        for text_row in range(start, stop):
            scores = batch_scores.getrow(text_row - start)
            source_candidates = scores.indices[top_positions(scores.data, top_k)]
            for row in rows_by_text[text_row]:
                relevant = np.fromiter(
                    relevant_indices_by_query[saved_query_ids[row]], dtype=np.int32
                )
                text_mask = text_pool_relevant_mask(row, relevant)
                source_mask = np.isin(relevant, source_candidates)
                text_sum += text_mask.mean()
                source_sum += source_mask.mean()
                union_sum += (text_mask | source_mask).mean()

        if stop == len(unique_texts) or stop % 500 == 0:
            print(f"{source_name}: {stop:,}/{len(unique_texts):,} строк")

    n_queries = int((text_codes >= 0).sum())
    result = pd.DataFrame([{
        "source": source_name,
        "queries": n_queries,
        "unique_texts": len(unique_texts),
        "top_k": top_k,
        "text_pool_recall": text_sum / n_queries,
        "source_bm25_recall": source_sum / n_queries,
        "union_recall": union_sum / n_queries,
        "added_recall": (union_sum - text_sum) / n_queries,
    }])
    del query_matrix, item_matrix, item_matrix_t
    gc.collect()
    return result


In [18]:
parameter_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_infm_params_text"]
)
parameter_bm25_results = evaluate_bm25_candidate_source(
    parameter_query_text,
    train_items["item_infm_params_text"],
    "BM25 параметров",
).rename(columns={"source_bm25_recall": "parameter_bm25_recall"})
display(parameter_bm25_results)



BM25 по параметрам: 1,196/1,196 фильтров


,queries_with_filters,unique_filters,top_k,text_pool_recall,parameter_bm25_recall,union_recall,added_recall
0,43049,1196,5000,0.8782,0.5124,0.9322,0.0541


0

### 14.3. BM25-источник: запрос → описание объявления

Проверяем описание отдельно от заголовка: прежний источник использует их склейку, а здесь слова из описания получают собственный BM25-скор.

In [20]:
description_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_query"]
)
description_bm25_results = evaluate_bm25_candidate_source(
    description_query_text,
    train_items["item_description_raw"],
    "BM25 описаний",
)
display(description_bm25_results)


BM25 описаний: 4,000/14,741 строк
BM25 описаний: 8,000/14,741 строк
BM25 описаний: 12,000/14,741 строк
BM25 описаний: 14,741/14,741 строк


,source,queries,unique_texts,top_k,text_pool_recall,source_bm25_recall,union_recall,added_recall
0,BM25 описаний,67186,14741,5000,0.8861,0.6677,0.8896,0.0035


## 15. Добавление BM25 по параметрам в гибрид

Для каждого уникального фильтра держим top-5 000 номеров строк и BM25-скоров только в оперативной памяти. Для кандидата из этого источника считаем:

```python
score = H + location_bonus * same_location + parameter_weight * parameter_bm25
```

`parameter_bm25` нормализуется внутри запроса. Если параметрический кандидат не был в top-5 000 ни одного текстового источника, его `H` равен нулю — так же, как в уже используемом усечённом объединении источников.

In [28]:
PARAMETER_TOP_K = EXTENDED_TOP_K
PARAMETER_BATCH_SIZE = 32

parameter_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_infm_params_text"]
    .fillna("")
    .astype(str)
    .reset_index(drop=True)
)
unique_parameter_filters = pd.Index(
    parameter_query_text.loc[parameter_query_text.ne("")].unique()
)
parameter_filter_codes = unique_parameter_filters.get_indexer(
    parameter_query_text
)

_, parameter_query_matrix, parameter_item_matrix = build_retrieval_matrices(
    "bm25", train_items["item_infm_params_text"].fillna(""),
    unique_parameter_filters,
)
parameter_top_indices = np.full(
    (len(unique_parameter_filters), PARAMETER_TOP_K), -1, dtype=np.int32
)
parameter_top_scores = np.full(
    (len(unique_parameter_filters), PARAMETER_TOP_K), -np.inf, dtype=np.float32
)
item_matrix_t = parameter_item_matrix.T

for start in range(0, len(unique_parameter_filters), PARAMETER_BATCH_SIZE):
    stop = min(start + PARAMETER_BATCH_SIZE, len(unique_parameter_filters))
    batch_scores = (parameter_query_matrix[start:stop] @ item_matrix_t).tocsr()
    write_top_k_from_sparse_scores(
        batch_scores, parameter_top_indices, parameter_top_scores, start
    )
    if stop == len(unique_parameter_filters) or stop % 500 == 0:
        print(f"Параметрические top-5000: {stop:,}/{len(unique_parameter_filters):,}")

del parameter_query_matrix, parameter_item_matrix, item_matrix_t
gc.collect()

baseline_top50_indices = np.load(
    SAVE_DIR / "best_extended_location_hybrid_top50_indices.npy",
    mmap_mode="r",
)
baseline_recall_by_row = np.fromiter(
    (recall_for_item_indices(row, baseline_top50_indices[row])
     for row in range(len(saved_query_ids))),
    dtype=np.float32,
    count=len(saved_query_ids),
)


def parameter_hybrid_parts(row, base_top_k=TOP_50):
    base_candidates, base_scores = base_hybrid_candidates_and_scores(row)
    if base_top_k is not None:
        base_top = top_positions(base_scores, base_top_k)
        base_candidates = base_candidates[base_top]
        base_scores = base_scores[base_top]
        order = np.argsort(base_candidates)
        base_candidates = base_candidates[order]
        base_scores = base_scores[order]
    filter_code = parameter_filter_codes[row]
    if filter_code < 0:
        return (
            base_candidates,
            base_scores,
            np.zeros(len(base_candidates), dtype=np.float32),
        )
    parameter_indices = parameter_top_indices[filter_code]
    valid = parameter_indices >= 0
    parameter_indices = parameter_indices[valid]
    parameter_scores = parameter_top_scores[filter_code][valid]

    candidates = np.union1d(base_candidates, parameter_indices)
    scores = location_bonus(
        candidates, valid_query_locations[row], best_location_bonus, item_locations
    ).astype(np.float32)
    base_positions = np.searchsorted(base_candidates, candidates)
    from_base = (base_positions < len(base_candidates)) & (
        base_candidates[np.minimum(base_positions, len(base_candidates) - 1)] == candidates
    )
    scores[from_base] = base_scores[base_positions[from_base]]

    parameter_bonus = np.zeros(len(candidates), dtype=np.float32)
    parameter_positions = np.searchsorted(candidates, parameter_indices)
    if len(parameter_scores) and parameter_scores[0] > 0:
        parameter_bonus[parameter_positions] = parameter_scores / parameter_scores[0]
    return candidates, scores, parameter_bonus


parameter_filtered_rows = np.flatnonzero(parameter_filter_codes >= 0)


def evaluate_parameter_weights(weights):
    weights = np.asarray(weights, dtype=np.float32)
    recall_sums = np.full(
        len(weights), baseline_recall_by_row.sum(), dtype=np.float64
    )

    for position, row in enumerate(parameter_filtered_rows, start=1):
        candidates, scores, parameter_bonus = parameter_hybrid_parts(row)
        for weight_position, weight in enumerate(weights):
            top50 = candidates[top_positions(
                scores + weight * parameter_bonus, TOP_50
            )]
            recall_sums[weight_position] += (
                recall_for_item_indices(row, top50) - baseline_recall_by_row[row]
            )

        if position % 5000 == 0 or position == len(parameter_filtered_rows):
            print(f"Подбор веса: {position:,}/{len(parameter_filtered_rows):,} запросов")

    return (
        pd.DataFrame({"parameter_weight": weights,
                      "recall_at_50": recall_sums / len(saved_query_ids)})
        .sort_values("recall_at_50", ascending=False)
        .reset_index(drop=True)
    )


Параметрические top-5000: 1,196/1,196


### 15.1. Грубый подбор веса BM25 по параметрам

In [29]:
COARSE_PARAMETER_WEIGHTS = np.array(
    [0.0, 0.25, 0.50, 0.75, 1.00, 1.50, 2.00], dtype=np.float32
)
parameter_weight_coarse_results = evaluate_parameter_weights(
    COARSE_PARAMETER_WEIGHTS
)
display(parameter_weight_coarse_results)


Подбор веса: 5,000/43,049 запросов
Подбор веса: 10,000/43,049 запросов
Подбор веса: 15,000/43,049 запросов
Подбор веса: 20,000/43,049 запросов
Подбор веса: 25,000/43,049 запросов
Подбор веса: 30,000/43,049 запросов
Подбор веса: 35,000/43,049 запросов
Подбор веса: 40,000/43,049 запросов
Подбор веса: 43,049/43,049 запросов


,parameter_weight,recall_at_50
0,0.2500,0.7404
1,0.0000,0.7393
2,0.5000,0.7275
3,0.7500,0.7062
4,1.0000,0.6812
5,1.5000,0.6259
6,2.0000,0.5535


### 15.2. Уточнение вокруг лучшего веса

In [30]:
best_coarse_parameter_weight = float(
    parameter_weight_coarse_results.iloc[0]["parameter_weight"]
)
FINE_PARAMETER_WEIGHTS = np.round(
    np.arange(
        max(0.0, best_coarse_parameter_weight - 0.20),
        best_coarse_parameter_weight + 0.201,
        0.05,
    ),
    2,
)
parameter_weight_fine_results = evaluate_parameter_weights(
    FINE_PARAMETER_WEIGHTS
)
display(parameter_weight_fine_results)


Подбор веса: 5,000/43,049 запросов
Подбор веса: 10,000/43,049 запросов
Подбор веса: 15,000/43,049 запросов
Подбор веса: 20,000/43,049 запросов
Подбор веса: 25,000/43,049 запросов
Подбор веса: 30,000/43,049 запросов
Подбор веса: 35,000/43,049 запросов
Подбор веса: 40,000/43,049 запросов
Подбор веса: 43,049/43,049 запросов


,parameter_weight,recall_at_50
0,0.1500,0.7421
1,0.1000,0.7419
2,0.2000,0.7414
3,0.0500,0.7413
4,0.2500,0.7404
5,0.3000,0.7387
6,0.3500,0.7366
7,0.4000,0.7341
8,0.4500,0.7311


## 16. Бонус за совпадение фильтра с параметрами объявления

Здесь не исключаем объявления и не запускаем текстовый поиск заново. Для уже сохранённого объединения top-5 000 четырёх источников считаем:

```python
score = H + location_bonus * same_location + filter_bonus * filter_match
```

`filter_match = 1`, если не менее 90% уникальных слов очищенного `search_infm_params_text` есть в `item_infm_params_text`; для пустого фильтра он равен нулю. Нулевой `filter_bonus` обязан воспроизвести текущий локационный гибрид.


In [41]:
FILTER_MATCH_THRESHOLD = 0.90
# Один блок нужен только при чтении top-5000 списков из memmap.
FILTER_GROUP_ROW_BATCH = 64

valid_filter_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_infm_params_text"]
    .fillna("")
    .astype(str)
    .reset_index(drop=True)
)
assert len(valid_filter_text) == len(saved_query_ids)

unique_valid_filters = pd.Index(
    valid_filter_text.loc[valid_filter_text.ne("")].unique()
)
valid_filter_codes = unique_valid_filters.get_indexer(valid_filter_text)

filter_rows_by_code = [
    np.flatnonzero(valid_filter_codes == filter_code).astype(np.int32)
    for filter_code in range(len(unique_valid_filters))
]

item_param_matrix, filter_param_matrix, filter_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"],
        unique_valid_filters,
    )
)
required_token_matches = np.ceil(
    FILTER_MATCH_THRESHOLD * filter_token_counts
).astype(np.int32)

# Сопоставление намеренно пока не считаем. В следующей ячейке
# фильтр будет проверяться только у кандидатов из четырёх top-5000
# списков, а не у всех train_items.
_ = gc.collect()

print(
    "Фильтры в validation:",
    f"{(valid_filter_codes >= 0).sum():,} запросов,",
    f"{len(unique_valid_filters):,} уникальных строк.",
)



Фильтры в validation: 43,049 запросов, 1,196 уникальных строк.


### 16.1. Грубый подбор коэффициента

Во время одного прохода по validation текстовый и локационный скор каждого кандидата считаются один раз. Для разных значений `filter_bonus` меняется только добавка за совпавший фильтр.


In [42]:
# Базовая выдача уже была посчитана на top-5000. Она нужна, чтобы
# нулевой бонус в точности совпал с прежней метрикой.
baseline_top50_indices = np.load(
    SAVE_DIR / "best_extended_location_hybrid_top50_indices.npy",
    mmap_mode="r",
)
assert baseline_top50_indices.shape == (len(saved_query_ids), TOP_50)
assert int(best_depth_metadata["top_k_per_source"]) == EXTENDED_TOP_K

baseline_recall_by_row = np.fromiter(
    (
        recall_for_item_indices(row, baseline_top50_indices[row])
        for row in range(len(saved_query_ids))
    ),
    dtype=np.float32,
    count=len(saved_query_ids),
)
assert np.isclose(
    baseline_recall_by_row.mean(),
    best_depth_metadata["recall_at_50"],
    atol=1e-8,
)

# Для постоянного бонуса достаточно хранить 50 лучших совпавших и
# 50 лучших несовпавших кандидатов. Любой другой кандидат из той же
# группы уже не сможет попасть в итоговый top-50.
filtered_rows = np.flatnonzero(valid_filter_codes >= 0).astype(np.int32)
row_to_filtered_position = np.full(
    len(saved_query_ids), -1, dtype=np.int32
)
row_to_filtered_position[filtered_rows] = np.arange(
    len(filtered_rows), dtype=np.int32
)
prepared_item_indices = np.full(
    (len(filtered_rows), 2 * TOP_50), -1, dtype=np.int32
)
prepared_base_scores = np.full(
    (len(filtered_rows), 2 * TOP_50), -np.inf, dtype=np.float32
)
prepared_match_flags = np.zeros(
    (len(filtered_rows), 2 * TOP_50), dtype=bool
)

# Один целочисленный массив заменяет множество: для каждого фильтра
# он собирает только объявления из top-5000 списков его запросов.
filter_group_mark = np.zeros(len(train_items), dtype=np.int32)
processed_rows = 0

for filter_code, rows in enumerate(filter_rows_by_code, start=1):
    for start in range(0, len(rows), FILTER_GROUP_ROW_BATCH):
        query_rows = rows[start:start + FILTER_GROUP_ROW_BATCH]
        for name in SOURCE_NAMES:
            item_indices = np.asarray(
                extended_top_indices[name][query_rows]
            ).ravel()
            item_indices = item_indices[item_indices >= 0]
            filter_group_mark[item_indices] = filter_code

    group_candidates = np.flatnonzero(
        filter_group_mark == filter_code
    ).astype(np.int32, copy=False)
    required_matches = int(required_token_matches[filter_code - 1])

    if required_matches == 0 or len(group_candidates) == 0:
        matching_items = np.empty(0, dtype=np.int32)
    else:
        overlap = (
            filter_param_matrix[filter_code - 1]
            @ item_param_matrix[group_candidates].T
        ).tocsr()
        matching_positions = overlap.indices[
            overlap.data >= required_matches
        ]
        matching_items = group_candidates[matching_positions]

    for row in rows:
        candidates, base_scores = base_hybrid_candidates_and_scores(row)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        cache_position = row_to_filtered_position[row]

        for offset, group_mask, is_match in (
            (0, filter_match, True),
            (TOP_50, ~filter_match, False),
        ):
            positions = np.flatnonzero(group_mask)
            if len(positions) == 0:
                continue

            positions = positions[top_positions(base_scores[positions], TOP_50)]
            stop = offset + len(positions)
            prepared_item_indices[cache_position, offset:stop] = (
                candidates[positions]
            )
            prepared_base_scores[cache_position, offset:stop] = (
                base_scores[positions]
            )
            prepared_match_flags[cache_position, offset:stop] = is_match

        processed_rows += 1

    if (
        filter_code % 100 == 0
        or filter_code == len(filter_rows_by_code)
    ):
        print(
            f"Подготовлено: {processed_rows:,}/{len(filtered_rows):,} "
            "запросов с фильтрами"
        )

assert processed_rows == len(filtered_rows)
del filter_group_mark, item_param_matrix, filter_param_matrix
_ = gc.collect()


def recall_with_filter_bonus(row, cache_position, bonus):
    valid = prepared_item_indices[cache_position] >= 0
    item_indices = prepared_item_indices[cache_position, valid]
    adjusted_scores = (
        prepared_base_scores[cache_position, valid]
        + bonus * prepared_match_flags[cache_position, valid]
    )
    top_item_indices = item_indices[top_positions(adjusted_scores, TOP_50)]
    return recall_for_item_indices(row, top_item_indices)


def evaluate_filter_bonus_grid(filter_bonuses):
    """Считает Recall@50 по подготовленным top-5000 кандидатам."""
    filter_bonuses = np.asarray(filter_bonuses, dtype=np.float32)
    recall_sums = np.full(
        len(filter_bonuses), baseline_recall_by_row.sum(), dtype=np.float64
    )

    for cache_position, row in enumerate(filtered_rows):
        base_recall = baseline_recall_by_row[row]
        for bonus_position, bonus in enumerate(filter_bonuses):
            if np.isclose(bonus, 0.0):
                continue

            recall_sums[bonus_position] += (
                recall_with_filter_bonus(row, cache_position, bonus)
                - base_recall
            )

    return (
        pd.DataFrame(
            {
                "filter_bonus": filter_bonuses,
                "recall_at_50": recall_sums / len(saved_query_ids),
            }
        )
        .sort_values("recall_at_50", ascending=False)
        .reset_index(drop=True)
    )


COARSE_FILTER_BONUSES = np.array(
    [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00],
    dtype=np.float32,
)
filter_bonus_coarse_results = evaluate_filter_bonus_grid(
    COARSE_FILTER_BONUSES
)
display(filter_bonus_coarse_results)

baseline_filter_recall = float(
    filter_bonus_coarse_results.loc[
        np.isclose(filter_bonus_coarse_results["filter_bonus"], 0.0),
        "recall_at_50",
    ].iloc[0]
)
assert np.isclose(
    baseline_filter_recall,
    best_depth_metadata["recall_at_50"],
    atol=1e-8,
)



Подготовлено: 37,816/43,049 запросов с фильтрами
Подготовлено: 40,744/43,049 запросов с фильтрами
Подготовлено: 41,508/43,049 запросов с фильтрами
Подготовлено: 41,932/43,049 запросов с фильтрами
Подготовлено: 42,144/43,049 запросов с фильтрами
Подготовлено: 42,321/43,049 запросов с фильтрами
Подготовлено: 42,470/43,049 запросов с фильтрами
Подготовлено: 42,605/43,049 запросов с фильтрами
Подготовлено: 42,726/43,049 запросов с фильтрами
Подготовлено: 42,846/43,049 запросов с фильтрами
Подготовлено: 42,951/43,049 запросов с фильтрами
Подготовлено: 43,049/43,049 запросов с фильтрами


,filter_bonus,recall_at_50
0,0.5000,0.7345
1,0.2500,0.7342
2,0.7500,0.7317
3,1.0000,0.7259
4,0.0000,0.7238
5,1.2500,0.7220
6,1.5000,0.7201
7,2.0000,0.7192


### 16.2. Уточнение вокруг лучшего значения

После грубой сетки проверяем значения с шагом `0.05` вокруг её победителя.


In [43]:
best_coarse_filter_bonus = float(
    filter_bonus_coarse_results.iloc[0]["filter_bonus"]
)
FINE_FILTER_BONUSES = np.unique(
    np.round(
        np.arange(
            max(0.0, best_coarse_filter_bonus - 0.25),
            best_coarse_filter_bonus + 0.251,
            0.05,
        ),
        2,
    )
).astype(np.float32)
filter_bonus_fine_results = evaluate_filter_bonus_grid(
    FINE_FILTER_BONUSES
)
display(filter_bonus_fine_results)

best_filter_bonus = float(
    filter_bonus_fine_results.iloc[0]["filter_bonus"]
)
best_filter_bonus_metadata = {
    "filter_match_threshold": FILTER_MATCH_THRESHOLD,
    "filter_bonus": best_filter_bonus,
    "recall_at_50": float(
        filter_bonus_fine_results.iloc[0]["recall_at_50"]
    ),
}
print("Лучший бонус за фильтр:", best_filter_bonus_metadata)

# Подготовленный top-5000 пул нужен только для validation-подбора.
# После выбора коэффициента освобождаем занятую им память.
del (
    prepared_item_indices,
    prepared_base_scores,
    prepared_match_flags,
    baseline_recall_by_row,
    baseline_top50_indices,
)
_ = gc.collect()



,filter_bonus,recall_at_50
0,0.3500,0.7350
1,0.4000,0.7350
2,0.4500,0.7348
3,0.3000,0.7348
4,0.5000,0.7345
5,0.2500,0.7342
6,0.5500,0.7338
7,0.6000,0.7334
8,0.6500,0.7330
9,0.7000,0.7324


Лучший бонус за фильтр: {'filter_match_threshold': 0.9, 'filter_bonus': 0.3499999940395355, 'recall_at_50': 0.7350403735526095}


### 16.3. Совместный подбор BM25 по параметрам и бонуса фильтра

Подбираем сразу два коэффициента на полном объединении кандидатов. Здесь `filter_match = 1` при покрытии 90% слов фильтра. После грубой сетки уточним область вокруг лучшей пары.

In [31]:
COARSE_PARAMETER_WEIGHTS = np.array(
    [0.00, 0.05, 0.10, 0.15, 0.20], dtype=np.float32
)
COARSE_FILTER_BONUSES = np.array(
    [0.00, 0.15, 0.30, 0.45], dtype=np.float32
)
parameter_weights, filter_bonuses = np.meshgrid(
    COARSE_PARAMETER_WEIGHTS, COARSE_FILTER_BONUSES, indexing="ij"
)
parameter_weights = parameter_weights.ravel()
filter_bonuses = filter_bonuses.ravel()
item_param_matrix, filter_param_matrix, filter_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"], unique_parameter_filters
    )
)
required_token_matches = np.ceil(
    0.90 * filter_token_counts
).astype(np.int32)

recall_sums = np.full(
    len(parameter_weights),
    baseline_recall_by_row.sum(),
    dtype=np.float64,
)
candidate_mark = np.zeros(len(train_items), dtype=np.int32)
processed_rows = 0

for filter_code in range(len(unique_parameter_filters)):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    for row in rows:
        candidates, _, _ = parameter_hybrid_parts(row, base_top_k=None)
        candidate_mark[candidates] = filter_code + 1

    group_candidates = np.flatnonzero(
        candidate_mark == filter_code + 1
    ).astype(np.int32, copy=False)
    overlap = (
        filter_param_matrix[filter_code]
        @ item_param_matrix[group_candidates].T
    ).tocsr()
    matching_items = group_candidates[overlap.indices[
        overlap.data >= required_token_matches[filter_code]
    ]]

    for row in rows:
        candidates, scores, parameter_bonus = parameter_hybrid_parts(
            row, base_top_k=None
        )
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )

        parameter_candidates = parameter_bonus > 0
        kept_positions = [np.flatnonzero(parameter_candidates)]
        for match_value in (False, True):
            positions = np.flatnonzero(
                ~parameter_candidates & (filter_match == match_value)
            )
            if len(positions):
                kept_positions.append(
                    positions[top_positions(scores[positions], TOP_50)]
                )
        kept_positions = np.unique(np.concatenate(kept_positions))
        candidates = candidates[kept_positions]
        scores = scores[kept_positions]
        parameter_bonus = parameter_bonus[kept_positions]
        filter_match = filter_match[kept_positions].astype(np.float32)

        all_scores = (
            scores[None, :]
            + parameter_weights[:, None] * parameter_bonus[None, :]
            + filter_bonuses[:, None] * filter_match[None, :]
        )
        top_positions_by_pair = np.argpartition(
            all_scores, -TOP_50, axis=1
        )[:, -TOP_50:]
        top_items_by_pair = candidates[top_positions_by_pair]
        relevant = np.fromiter(
            relevant_indices_by_query[saved_query_ids[row]], dtype=np.int32
        )
        recall_sums += (
            np.isin(top_items_by_pair, relevant).sum(axis=1) / len(relevant)
            - baseline_recall_by_row[row]
        )

    processed_rows += len(rows)
    if (filter_code + 1) % 100 == 0 or filter_code + 1 == len(unique_parameter_filters):
        print(f"Бонус фильтра: {processed_rows:,}/{len(parameter_filtered_rows):,} запросов")

parameter_filter_hybrid_results = (
    pd.DataFrame({
        "parameter_weight": parameter_weights,
        "filter_bonus": filter_bonuses,
        "recall_at_50": recall_sums / len(saved_query_ids),
    })
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(parameter_filter_hybrid_results)

del candidate_mark, item_param_matrix, filter_param_matrix
gc.collect()


Бонус фильтра: 37,816/43,049 запросов
Бонус фильтра: 40,744/43,049 запросов
Бонус фильтра: 41,508/43,049 запросов
Бонус фильтра: 41,932/43,049 запросов
Бонус фильтра: 42,144/43,049 запросов
Бонус фильтра: 42,321/43,049 запросов
Бонус фильтра: 42,470/43,049 запросов
Бонус фильтра: 42,605/43,049 запросов
Бонус фильтра: 42,726/43,049 запросов
Бонус фильтра: 42,846/43,049 запросов
Бонус фильтра: 42,951/43,049 запросов
Бонус фильтра: 43,049/43,049 запросов


,parameter_weight,filter_bonus,recall_at_50
0,0.0000,0.3000,0.7566
1,0.0500,0.3000,0.7566
2,0.0000,0.4500,0.7563
3,0.0500,0.4500,0.7560
4,0.1000,0.3000,0.7550
5,0.1000,0.4500,0.7541
6,0.0500,0.1500,0.7535
7,0.1500,0.3000,0.7532
8,0.0000,0.1500,0.7530
9,0.1000,0.1500,0.7528


0

## 17. Бонусы за рейтинг и число отзывов

Во всех четырёх проверках база одинакова: текстовый гибрид с бонусом за локацию на top-5000 кандидатах. Пропуск в рейтинге или числе отзывов даёт нулевой бонус.

In [14]:
item_rating_values = (
    train_items["item_rating"].fillna(0).to_numpy(dtype=np.float32)
)
item_review_values = (
    train_items["item_rating_reviews_count"]
    .fillna(0)
    .to_numpy(dtype=np.float32)
)
assert len(item_rating_values) == len(saved_item_ids)
assert len(item_review_values) == len(saved_item_ids)


def evaluate_item_value_bonus(item_values, coefficients, value_name):
    """Проверяет формулу: базовый гибрид + coefficient × значение объявления."""
    coefficients = np.asarray(coefficients, dtype=np.float32)
    recall_sums = np.zeros(len(coefficients), dtype=np.float64)

    for row in range(len(saved_query_ids)):
        candidates, base_scores = base_hybrid_candidates_and_scores(row)
        candidate_values = item_values[candidates]

        for position, coefficient in enumerate(coefficients):
            scores = base_scores + coefficient * candidate_values
            top50 = candidates[top_positions(scores, TOP_50)]
            recall_sums[position] += recall_for_item_indices(row, top50)

        if (row + 1) % 5_000 == 0 or row + 1 == len(saved_query_ids):
            print(f"{value_name}: {row + 1:,}/{len(saved_query_ids):,}")

    return pd.DataFrame(
        {
            "coefficient": coefficients,
            "recall_at_50": recall_sums / len(saved_query_ids),
        }
    ).sort_values("recall_at_50", ascending=False).reset_index(drop=True)


rating_fixed_results = evaluate_item_value_bonus(
    item_rating_values,
    coefficients=[0.0, 0.20],
    value_name="rating x fixed coefficient",
)
baseline_location_recall = float(
    rating_fixed_results.loc[
        np.isclose(rating_fixed_results["coefficient"], 0.0),
        "recall_at_50",
    ].iloc[0]
)
assert np.isclose(
    baseline_location_recall,
    best_depth_metadata["recall_at_50"],
    atol=1e-8,
)
display(rating_fixed_results)


rating x fixed coefficient: 5,000/67,186
rating x fixed coefficient: 10,000/67,186
rating x fixed coefficient: 15,000/67,186
rating x fixed coefficient: 20,000/67,186
rating x fixed coefficient: 25,000/67,186
rating x fixed coefficient: 30,000/67,186
rating x fixed coefficient: 35,000/67,186
rating x fixed coefficient: 40,000/67,186
rating x fixed coefficient: 45,000/67,186
rating x fixed coefficient: 50,000/67,186
rating x fixed coefficient: 55,000/67,186
rating x fixed coefficient: 60,000/67,186
rating x fixed coefficient: 65,000/67,186
rating x fixed coefficient: 67,186/67,186


,coefficient,recall_at_50
0,0.0000,0.7238
1,0.2000,0.6792


In [15]:
RATING_COEFFICIENTS = np.round(
    np.arange(0.0, 1.01, 0.10),
    2,
)
rating_grid_results = evaluate_item_value_bonus(
    item_rating_values,
    coefficients=RATING_COEFFICIENTS,
    value_name="rating x grid coefficient",
)
display(rating_grid_results)


rating x grid coefficient: 5,000/67,186
rating x grid coefficient: 10,000/67,186
rating x grid coefficient: 15,000/67,186
rating x grid coefficient: 20,000/67,186
rating x grid coefficient: 25,000/67,186
rating x grid coefficient: 30,000/67,186
rating x grid coefficient: 35,000/67,186
rating x grid coefficient: 40,000/67,186
rating x grid coefficient: 45,000/67,186
rating x grid coefficient: 50,000/67,186
rating x grid coefficient: 55,000/67,186
rating x grid coefficient: 60,000/67,186
rating x grid coefficient: 65,000/67,186
rating x grid coefficient: 67,186/67,186


,coefficient,recall_at_50
0,0.0000,0.7238
1,0.1000,0.7069
2,0.2000,0.6792
3,0.3000,0.6726
4,0.4000,0.6683
5,0.5000,0.6646
6,0.6000,0.6606
7,0.7000,0.6569
8,0.8000,0.6530
9,0.9000,0.6491


In [16]:
review_fixed_results = evaluate_item_value_bonus(
    item_review_values,
    coefficients=[0.0, 0.01, 0.02],
    value_name="review count x fixed coefficient",
)
assert np.isclose(
    float(
        review_fixed_results.loc[
            np.isclose(review_fixed_results["coefficient"], 0.0),
            "recall_at_50",
        ].iloc[0]
    ),
    baseline_location_recall,
    atol=1e-8,
)
display(review_fixed_results)


review count x fixed coefficient: 5,000/67,186
review count x fixed coefficient: 10,000/67,186
review count x fixed coefficient: 15,000/67,186
review count x fixed coefficient: 20,000/67,186
review count x fixed coefficient: 25,000/67,186
review count x fixed coefficient: 30,000/67,186
review count x fixed coefficient: 35,000/67,186
review count x fixed coefficient: 40,000/67,186
review count x fixed coefficient: 45,000/67,186
review count x fixed coefficient: 50,000/67,186
review count x fixed coefficient: 55,000/67,186
review count x fixed coefficient: 60,000/67,186
review count x fixed coefficient: 65,000/67,186
review count x fixed coefficient: 67,186/67,186


,coefficient,recall_at_50
0,0.0000,0.7238
1,0.0100,0.0087
2,0.0200,0.0067


In [17]:
REVIEW_COEFFICIENTS = np.round(
    np.arange(0.0, 1.01, 0.10),
    2,
)
review_grid_results = evaluate_item_value_bonus(
    item_review_values,
    coefficients=REVIEW_COEFFICIENTS,
    value_name="review count x grid coefficient",
)
display(review_grid_results)


review count x grid coefficient: 5,000/67,186
review count x grid coefficient: 10,000/67,186
review count x grid coefficient: 15,000/67,186
review count x grid coefficient: 20,000/67,186
review count x grid coefficient: 25,000/67,186
review count x grid coefficient: 30,000/67,186
review count x grid coefficient: 35,000/67,186
review count x grid coefficient: 40,000/67,186
review count x grid coefficient: 45,000/67,186
review count x grid coefficient: 50,000/67,186
review count x grid coefficient: 55,000/67,186
review count x grid coefficient: 60,000/67,186
review count x grid coefficient: 65,000/67,186
review count x grid coefficient: 67,186/67,186


,coefficient,recall_at_50
0,0.0000,0.7238
1,0.1000,0.0058
2,0.2000,0.0057
3,0.3000,0.0057
4,0.9000,0.0057
5,0.8000,0.0057
6,0.7000,0.0057
7,1.0000,0.0057
8,0.6000,0.0057
9,0.4000,0.0057


## 18. Ранний эксперимент с CatBoost

Ранжировщик проверяется на top-300 тогдашнего гибрида. Этот эксперимент предшествует расширению пула E5 и географическим поиском.

### 18.1. Признаки и кандидаты

In [33]:
from catboost import CatBoostRanker, Pool

CATBOOST_TOP_K = 300
CATBOOST_TRAIN_QUERY_COUNT = 6_000
CATBOOST_VALID_QUERY_COUNT = 3_000
FINAL_PARAMETER_WEIGHT = 0.0
FINAL_FILTER_BONUS = 0.30
RANDOM_SEED = 42

query_table = queries_valid.set_index("train_query_id").reindex(saved_query_ids)
query_category_values = pd.to_numeric(
    query_table["search_category"], errors="coerce"
).fillna(-1).to_numpy(dtype=np.int32)
query_delivery_values = pd.to_numeric(
    query_table["search_is_delivery_search"], errors="coerce"
).fillna(0).to_numpy(dtype=np.float32)

item_price = pd.to_numeric(train_items["item_price"], errors="coerce").astype("float64")
item_rating = pd.to_numeric(train_items["item_rating"], errors="coerce").astype("float64")
item_reviews = pd.to_numeric(
    train_items["item_rating_reviews_count"], errors="coerce"
).astype("float64")
item_log_price = np.log1p(item_price.fillna(0).clip(lower=0)).to_numpy(np.float32)
item_price_missing = item_price.isna().to_numpy(np.float32)
item_rating_values = item_rating.fillna(0).to_numpy(np.float32)
item_rating_missing = item_rating.isna().to_numpy(np.float32)
item_log_reviews = np.log1p(item_reviews.fillna(0)).to_numpy(np.float32)
item_reviews_missing = item_reviews.isna().to_numpy(np.float32)
item_phone_hidden = pd.to_numeric(
    train_items["item_is_phone_hidden"], errors="coerce"
).fillna(0).to_numpy(np.float32)
item_message_forbidden = pd.to_numeric(
    train_items["item_is_message_forbidden"], errors="coerce"
).fillna(0).to_numpy(np.float32)
item_category_values = pd.to_numeric(
    train_items["item_category_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
item_microcat_values = pd.to_numeric(
    train_items["item_microcat_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)

rng = np.random.default_rng(RANDOM_SEED)
filter_groups = parameter_filter_codes >= 0
model_train_rows, model_valid_rows = [], []
for has_filter in (False, True):
    rows = rng.permutation(np.flatnonzero(filter_groups == has_filter))
    train_count = round(CATBOOST_TRAIN_QUERY_COUNT * len(rows) / len(saved_query_ids))
    valid_count = round(CATBOOST_VALID_QUERY_COUNT * len(rows) / len(saved_query_ids))
    model_train_rows.append(rows[:train_count])
    model_valid_rows.append(rows[train_count:train_count + valid_count])
model_train_rows = np.sort(np.concatenate(model_train_rows))
model_valid_rows = np.sort(np.concatenate(model_valid_rows))

NUMERIC_FEATURES = [
    "hybrid_score", "text_score", "parameter_bm25", "filter_match",
    "same_location", "log_price", "price_missing", "rating",
    "rating_missing", "log_reviews", "reviews_missing",
    "phone_hidden", "message_forbidden", "delivery",
]
CATEGORICAL_FEATURES = [
    "search_category", "item_category_id", "item_microcat_id",
]
print(f"CatBoost: {len(model_train_rows):,} train и {len(model_valid_rows):,} valid групп")


CatBoost: 6,000 train и 3,000 valid групп


In [35]:
# CatBoost: построение кандидатов и признаков.
# Он ранжирует только 300 кандидатов текущего гибрида, а не весь корпус.
FILTER_MATCH_THRESHOLD = 0.90

def build_catboost_features(rows):
    n_groups = len(rows)
    n_candidates = n_groups * CATBOOST_TOP_K
    numeric = np.empty((n_candidates, len(NUMERIC_FEATURES)), dtype=np.float32)
    categorical = np.empty((n_candidates, len(CATEGORICAL_FEATURES)), dtype=np.int32)
    labels = np.empty(n_candidates, dtype=bool)
    candidate_indices = np.empty((n_groups, CATBOOST_TOP_K), dtype=np.int32)
    candidate_recall = np.empty(n_groups, dtype=np.float32)
    formula_recall = np.empty(n_groups, dtype=np.float32)
    row_position = np.full(len(saved_query_ids), -1, dtype=np.int32)
    row_position[rows] = np.arange(n_groups, dtype=np.int32)

    def add_group(row, candidates, base_scores, parameter_bonus, filter_match):
        group_position = row_position[row]
        final_scores = (
            base_scores
            + FINAL_PARAMETER_WEIGHT * parameter_bonus
            + FINAL_FILTER_BONUS * filter_match
        )
        positions = top_positions(final_scores, CATBOOST_TOP_K)
        item_indices = candidates[positions]
        assert len(item_indices) == CATBOOST_TOP_K
        start = group_position * CATBOOST_TOP_K
        stop = start + CATBOOST_TOP_K
        same_location = (
            item_locations[item_indices] == valid_query_locations[row]
        ).astype(np.float32)
        numeric[start:stop] = np.column_stack([
            final_scores[positions],
            base_scores[positions] - best_location_bonus * same_location,
            parameter_bonus[positions],
            filter_match[positions],
            same_location,
            item_log_price[item_indices],
            item_price_missing[item_indices],
            item_rating_values[item_indices],
            item_rating_missing[item_indices],
            item_log_reviews[item_indices],
            item_reviews_missing[item_indices],
            item_phone_hidden[item_indices],
            item_message_forbidden[item_indices],
            np.full(CATBOOST_TOP_K, query_delivery_values[row], dtype=np.float32),
        ])
        categorical[start:stop] = np.column_stack([
            np.full(CATBOOST_TOP_K, query_category_values[row], dtype=np.int32),
            item_category_values[item_indices],
            item_microcat_values[item_indices],
        ])
        relevant = np.fromiter(
            relevant_indices_by_query[saved_query_ids[row]], dtype=np.int32
        )
        found = np.isin(item_indices, relevant, assume_unique=True)
        labels[start:stop] = found
        candidate_indices[group_position] = item_indices
        candidate_recall[group_position] = found.sum() / len(relevant)
        formula_recall[group_position] = found[:TOP_50].sum() / len(relevant)

    item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
        train_items["item_infm_params_text"], unique_parameter_filters
    )
    required_matches = np.ceil(
        FILTER_MATCH_THRESHOLD * filter_token_counts
    ).astype(np.int32)
    candidate_mark = np.zeros(len(train_items), dtype=np.int32)
    filter_codes = parameter_filter_codes[rows]

    for number, filter_code in enumerate(np.unique(filter_codes), start=1):
        group_rows = rows[filter_codes == filter_code]
        if filter_code < 0:
            matching_items = np.empty(0, dtype=np.int32)
        else:
            mark_value = int(filter_code) + 1
            for row in group_rows:
                candidates, _, _ = parameter_hybrid_parts(row, base_top_k=None)
                candidate_mark[candidates] = mark_value
            group_candidates = np.flatnonzero(candidate_mark == mark_value)
            if required_matches[filter_code] == 0 or len(group_candidates) == 0:
                matching_items = np.empty(0, dtype=np.int32)
            else:
                overlap = (
                    filter_matrix[filter_code] @ item_matrix[group_candidates].T
                ).tocsr()
                matching_items = group_candidates[
                    overlap.indices[overlap.data >= required_matches[filter_code]]
                ]

        for row in group_rows:
            candidates, base_scores, parameter_bonus = parameter_hybrid_parts(
                row, base_top_k=None
            )
            filter_match = np.isin(
                candidates, matching_items, assume_unique=True
            ).astype(np.float32)
            add_group(row, candidates, base_scores, parameter_bonus, filter_match)

        if number % 100 == 0 or number == len(np.unique(filter_codes)):
            print(f"Кандидаты CatBoost: {number:,}/{len(np.unique(filter_codes)):,} фильтров")

    del candidate_mark, item_matrix, filter_matrix
    gc.collect()
    return numeric, categorical, labels, candidate_indices, candidate_recall, formula_recall


model_rows = np.concatenate([model_train_rows, model_valid_rows])
(
    catboost_numeric, catboost_categorical, catboost_labels,
    catboost_candidate_indices, catboost_candidate_recall,
    catboost_formula_recall,
) = build_catboost_features(model_rows)

n_model_groups = len(model_rows)
catboost_is_train = np.repeat(
    np.arange(n_model_groups) < len(model_train_rows), CATBOOST_TOP_K
)
catboost_group_id = np.repeat(
    np.arange(n_model_groups, dtype=np.int32), CATBOOST_TOP_K
)
catboost_feature_frame = pd.DataFrame(
    catboost_numeric, columns=NUMERIC_FEATURES
)
for position, column in enumerate(CATEGORICAL_FEATURES):
    catboost_feature_frame[column] = catboost_categorical[:, position]

catboost_candidate_summary = pd.DataFrame({
    "split": np.where(
        np.arange(n_model_groups) < len(model_train_rows), "train", "valid"
    ),
    "candidate_recall_at_300": catboost_candidate_recall,
    "formula_recall_at_50": catboost_formula_recall,
}).groupby("split", as_index=False).mean()
display(catboost_candidate_summary)


Кандидаты CatBoost: 100/367 фильтров
Кандидаты CatBoost: 200/367 фильтров
Кандидаты CatBoost: 300/367 фильтров
Кандидаты CatBoost: 367/367 фильтров


,split,candidate_recall_at_300,formula_recall_at_50
0,train,0.8400,0.7584
1,valid,0.8467,0.7606


### 18.2. Обучение ранжировщика и проверка Recall@50

Группы без релевантного объявления в top-300 не используются при обучении, но остаются в честной проверке Recall@50.

In [36]:
group_has_positive = catboost_labels.reshape(n_model_groups, CATBOOST_TOP_K).any(axis=1)
train_group_mask = (
    (np.arange(n_model_groups) < len(model_train_rows)) & group_has_positive
)
train_mask = np.repeat(train_group_mask, CATBOOST_TOP_K)
valid_mask = ~catboost_is_train

train_pool = Pool(
    catboost_feature_frame.loc[train_mask],
    label=catboost_labels[train_mask],
    group_id=catboost_group_id[train_mask],
    cat_features=CATEGORICAL_FEATURES,
)
valid_pool = Pool(
    catboost_feature_frame.loc[valid_mask],
    label=catboost_labels[valid_mask],
    group_id=catboost_group_id[valid_mask],
    cat_features=CATEGORICAL_FEATURES,
)

catboost_model = CatBoostRanker(
    loss_function="YetiRank", eval_metric="NDCG:top=50",
    iterations=300, depth=6, learning_rate=0.08, l2_leaf_reg=5,
    random_seed=RANDOM_SEED, thread_count=4,
    allow_writing_files=False, verbose=50,
)
catboost_model.fit(
    train_pool, eval_set=valid_pool, early_stopping_rounds=30
)

valid_scores = catboost_model.predict(valid_pool).reshape(
    len(model_valid_rows), CATBOOST_TOP_K
)
catboost_recall = np.mean([
    recall_for_item_indices(
        row, catboost_candidate_indices[len(model_train_rows) + position][
            top_positions(valid_scores[position], TOP_50)
        ]
    )
    for position, row in enumerate(model_valid_rows)
])
formula_recall = catboost_candidate_summary.loc[
    catboost_candidate_summary["split"].eq("valid"),
    "formula_recall_at_50",
].iloc[0]
display(pd.DataFrame({
    "method": ["Текущая формула", "CatBoost"],
    "recall_at_50": [formula_recall, catboost_recall],
}))
display(pd.DataFrame({
    "feature": catboost_model.feature_names_,
    "importance": catboost_model.get_feature_importance(
        type="PredictionValuesChange"
    ),
}).sort_values("importance", ascending=False).reset_index(drop=True))


Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.5119922	best: 0.5119922 (0)	total: 5.62s	remaining: 28m 1s
50:	test: 0.6027709	best: 0.6033881 (37)	total: 3m 27s	remaining: 16m 51s
100:	test: 0.6089647	best: 0.6089647 (100)	total: 6m 34s	remaining: 12m 56s
150:	test: 0.6139613	best: 0.6142454 (146)	total: 9m 40s	remaining: 9m 32s
200:	test: 0.6178857	best: 0.6179308 (199)	total: 12m 45s	remaining: 6m 17s
250:	test: 0.6188757	best: 0.6190002 (242)	total: 15m 51s	remaining: 3m 5s
299:	test: 0.6209739	best: 0.6211013 (297)	total: 18m 53s	remaining: 0us

bestTest = 0.6211013091
bestIteration = 297

Shrink model to first 298 iterations.


,method,recall_at_50
0,Текущая формула,0.7606
1,CatBoost,0.7482


,feature,importance
0,hybrid_score,None
1,text_score,None
2,parameter_bm25,None
3,filter_match,None
4,same_location,None
5,log_price,None
6,price_missing,None
7,rating,None
8,rating_missing,None
9,log_reviews,None


In [37]:
# Вес 0 — только текущая формула, вес 1 — только CatBoost.
def normalize_scores_by_query(scores):
    minimum = scores.min(axis=1, keepdims=True)
    spread = scores.max(axis=1, keepdims=True) - minimum
    return np.divide(
        scores - minimum, spread, out=np.zeros_like(scores), where=spread > 0
    )

formula_scores = catboost_feature_frame.loc[valid_mask, "hybrid_score"].to_numpy(
    dtype=np.float32
).reshape(len(model_valid_rows), CATBOOST_TOP_K)
formula_scores = normalize_scores_by_query(formula_scores)
model_scores = normalize_scores_by_query(valid_scores)
valid_candidate_indices = catboost_candidate_indices[len(model_train_rows):]
row_positions = np.arange(len(model_valid_rows))[:, None]

blend_rows = []
for catboost_weight in np.round(np.arange(0.0, 1.01, 0.05), 2):
    scores = (
        (1 - catboost_weight) * formula_scores
        + catboost_weight * model_scores
    )
    top50_positions = np.argpartition(scores, -TOP_50, axis=1)[:, -TOP_50:]
    predictions = valid_candidate_indices[row_positions, top50_positions]
    recall = np.mean([
        recall_for_item_indices(row, prediction)
        for row, prediction in zip(model_valid_rows, predictions)
    ])
    blend_rows.append({
        "catboost_weight": catboost_weight, "recall_at_50": recall
    })

catboost_blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(catboost_blend_results)


,catboost_weight,recall_at_50
0,0.3500,0.7618
1,0.2500,0.7615
2,0.3000,0.7615
3,0.1500,0.7615
4,0.4000,0.7612
5,0.4500,0.7611
6,0.0000,0.7609
7,0.2000,0.7608
8,0.5000,0.7605
9,0.1000,0.7605


## 19. Ответ для benchmark: гибрид с локацией и фильтрами

Для benchmark заново считаем четыре текстовых top-5 000 и отдельный top-5 000 по параметрам. Все промежуточные данные остаются только в памяти; сохраняется один новый CSV-файл.

In [38]:
from pathlib import Path
from time import perf_counter

BENCHMARK_FILTER_TOP_K = 5_000
BENCHMARK_FILTER_BONUS = 0.30
BENCHMARK_FILTER_THRESHOLD = 0.90
BENCHMARK_FILTER_ANSWER_PATH = Path("answer_location_filter.csv")

# CatBoost не войдёт в этот ответ; освобождаем память для top-5000 источников.
for name in [
    "catboost_model", "train_pool", "valid_pool", "catboost_feature_frame",
    "catboost_numeric", "catboost_categorical", "catboost_labels",
    "catboost_candidate_indices", "valid_scores",
]:
    globals().pop(name, None)
gc.collect()

benchmark_query_ids = benchmark_queries["query_id"].astype(str).to_numpy()
benchmark_item_ids = benchmark_items["item_id"].astype(str).to_numpy()
benchmark_query_text = benchmark_queries["search_query"].fillna("")
benchmark_query_locations = benchmark_queries["search_location_id"].to_numpy()
benchmark_item_locations = benchmark_items["item_location_id"].to_numpy()

def build_benchmark_top_k(name, method, item_text, query_text, batch_size, vectorizer=None):
    started_at = perf_counter()
    _, item_matrix, indices, scores = retrieve_top_k(
        method, item_text, query_text, BENCHMARK_FILTER_TOP_K,
        vectorizer=vectorizer, batch_size=batch_size, source_name=name,
    )
    del item_matrix
    gc.collect()
    print(f"{name}: {(perf_counter() - started_at) / 60:.2f} мин.")
    return indices, scores

benchmark_title = benchmark_items["item_title_raw"].fillna("")
benchmark_title_description = combine_text(
    benchmark_items, ["item_title_raw", "item_description_raw"]
)
benchmark_configs = [
    ("bm25_title_description", "bm25", benchmark_title_description, 128, None),
    ("char_tfidf_title", "char_tfidf", benchmark_title, 64, make_tfidf_vectorizer("char")),
    ("bm25_title", "bm25", benchmark_title, 128, None),
    ("word_tfidf_title", "word_tfidf", benchmark_title, 128, make_tfidf_vectorizer("word")),
]
benchmark_source_indices, benchmark_source_scores = {}, {}
for name, method, item_text, batch_size, vectorizer in benchmark_configs:
    indices, scores = build_benchmark_top_k(
        name, method, item_text, benchmark_query_text, batch_size, vectorizer
    )
    benchmark_source_indices[name] = indices
    benchmark_source_scores[name] = scores

benchmark_filter_text = benchmark_queries["search_infm_params_text"].fillna("")
benchmark_unique_filters = pd.Index(
    benchmark_filter_text.loc[benchmark_filter_text.ne("")].unique()
)
benchmark_filter_codes = benchmark_unique_filters.get_indexer(benchmark_filter_text)
_, parameter_item_matrix, benchmark_parameter_indices, _ = retrieve_top_k(
    "bm25",
    benchmark_items["item_infm_params_text"].fillna(""),
    benchmark_unique_filters,
    BENCHMARK_FILTER_TOP_K,
    batch_size=32,
    source_name="bm25_parameters",
)
del parameter_item_matrix, benchmark_title, benchmark_title_description
gc.collect()
print(f"Фильтров: {len(benchmark_unique_filters):,}; запросов: {len(benchmark_query_ids):,}.")


bm25_title_description: 128/2,452 запросов
bm25_title_description: 1,280/2,452 запросов
bm25_title_description: 2,452/2,452 запросов
bm25_title_description: 0.83 мин.
char_tfidf_title: 64/2,452 запросов
char_tfidf_title: 640/2,452 запросов
char_tfidf_title: 1,280/2,452 запросов
char_tfidf_title: 1,920/2,452 запросов
char_tfidf_title: 2,452/2,452 запросов
char_tfidf_title: 0.42 мин.
bm25_title: 128/2,452 запросов
bm25_title: 1,280/2,452 запросов
bm25_title: 2,452/2,452 запросов
bm25_title: 0.04 мин.
word_tfidf_title: 128/2,452 запросов
word_tfidf_title: 1,280/2,452 запросов
word_tfidf_title: 2,452/2,452 запросов
word_tfidf_title: 0.04 мин.
bm25_parameters: 32/127 запросов
bm25_parameters: 127/127 запросов
Фильтров: 127; запросов: 2,452.


In [39]:
def benchmark_base_candidates(row):
    candidates, scores = source_candidates_and_scores(
        row, BASE_HYBRID_WEIGHTS, benchmark_source_indices,
        benchmark_source_scores, SOURCE_NAMES,
    )
    scores += location_bonus(
        candidates, benchmark_query_locations[row], best_location_bonus,
        benchmark_item_locations,
    )
    return candidates, scores


def benchmark_candidates_with_parameters(row):
    base_candidates, base_scores = benchmark_base_candidates(row)
    filter_code = benchmark_filter_codes[row]
    if filter_code < 0:
        return base_candidates, base_scores

    parameter_candidates = benchmark_parameter_indices[filter_code]
    parameter_candidates = parameter_candidates[parameter_candidates >= 0]
    candidates = np.union1d(base_candidates, parameter_candidates)
    scores = location_bonus(
        candidates, benchmark_query_locations[row], best_location_bonus,
        benchmark_item_locations,
    ).astype(np.float32)
    base_positions = np.searchsorted(candidates, base_candidates)
    scores[base_positions] = base_scores
    return candidates, scores


item_filter_matrix, query_filter_matrix, filter_token_counts = (
    build_filter_token_matrices(
        benchmark_items["item_infm_params_text"], benchmark_unique_filters
    )
)
required_matches = np.ceil(
    BENCHMARK_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
benchmark_top50_indices = np.full(
    (len(benchmark_query_ids), TOP_50), -1, dtype=np.int32
)

for row in np.flatnonzero(benchmark_filter_codes < 0):
    candidates, scores = benchmark_candidates_with_parameters(row)
    benchmark_top50_indices[row] = candidates[top_positions(scores, TOP_50)]

candidate_mark = np.zeros(len(benchmark_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(benchmark_unique_filters)), start=1):
    rows = np.flatnonzero(benchmark_filter_codes == filter_code)
    mark_value = filter_code + 1
    for row in rows:
        candidates, _ = benchmark_candidates_with_parameters(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    if required_matches[filter_code] == 0 or len(group_candidates) == 0:
        matching_items = np.empty(0, dtype=np.int32)
    else:
        overlap = (
            query_filter_matrix[filter_code] @ item_filter_matrix[group_candidates].T
        ).tocsr()
        matching_items = group_candidates[
            overlap.indices[overlap.data >= required_matches[filter_code]]
        ]

    for row in rows:
        candidates, scores = benchmark_candidates_with_parameters(row)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        benchmark_top50_indices[row] = candidates[top_positions(
            scores + BENCHMARK_FILTER_BONUS * filter_match, TOP_50
        )]

    if position % 100 == 0 or position == len(benchmark_unique_filters):
        print(f"Фильтры в benchmark: {position:,}/{len(benchmark_unique_filters):,}")

benchmark_filter_answer = pd.DataFrame({
    "query_id": benchmark_query_ids,
    "answer": [
        " ".join(benchmark_item_ids[indices[indices >= 0]])
        for indices in benchmark_top50_indices
    ],
})
benchmark_filter_answer.to_csv(
    BENCHMARK_FILTER_ANSWER_PATH, index=False, encoding="utf-8"
)

del candidate_mark, item_filter_matrix, query_filter_matrix
del benchmark_source_indices, benchmark_source_scores, benchmark_parameter_indices
gc.collect()
print(f"Создан файл: {BENCHMARK_FILTER_ANSWER_PATH.resolve()}")
display(benchmark_filter_answer.head())


Фильтры в benchmark: 100/127
Фильтры в benchmark: 127/127
Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_location_filter.csv


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 255fbeaf526a1cc1 b92ee8f432cec2d1 d722bcda1a555091 e08f4ce94c268cef deb58d7ad8c27d06 603623b4bd9e8f...
1,JTrdTaZJvSiLPkXj,16ea26ff0aa0450a 8703b2a0442c5d25 cbeccbecb1fb8d86 367af128a9ea2a48 4a435cc0a254c1fd 413f6384e3c07b64 bafe6304f743b5...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 dab52187b4500d9b 3b370cc603f67947 168a9207e80b0be4 d62074dd39caefee edecb3695ab889...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 bd2cb4500fad728e a846a2a5241e1180 4c15e7abba063ac5 3cce43c811459438 e42795fcb72962a5 1e5924dc546b82...
4,YgHcM9MVbxKnxD1e,615c73ea4c3bfd15 1a62634394544781 ba78ad593ec3412d 13f58fe9542bdb1a 9c41fe789f98d702 60e3b99111bd032b 4ef5ba70fa9baa...


In [40]:
benchmark_answer_check = pd.read_csv(
    BENCHMARK_FILTER_ANSWER_PATH,
    dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
answer_item_lists = benchmark_answer_check["answer"].map(str.split)
benchmark_item_id_set = set(benchmark_item_ids)

assert benchmark_answer_check.columns.tolist() == ["query_id", "answer"]
assert len(benchmark_answer_check) == len(benchmark_query_ids)
assert benchmark_answer_check["query_id"].is_unique
assert set(benchmark_answer_check["query_id"]) == set(benchmark_query_ids)
assert answer_item_lists.map(len).le(TOP_50).all()
assert answer_item_lists.map(lambda ids: len(ids) == len(set(ids))).all()
assert answer_item_lists.map(
    lambda ids: set(ids).issubset(benchmark_item_id_set)
).all()

display(pd.DataFrame([{
    "file": str(BENCHMARK_FILTER_ANSWER_PATH),
    "rows": len(benchmark_answer_check),
    "min_item_ids": answer_item_lists.map(len).min(),
    "max_item_ids": answer_item_lists.map(len).max(),
    "all_checks_passed": True,
}]))


,file,rows,min_item_ids,max_item_ids,all_checks_passed
0,answer_location_filter.csv,2452,50,50,True


## 20. Добавление семантических кандидатов

Сравниваем полноту текстового пула, E5 и их объединения. Затем проверяем влияние нового пула на фактический Recall@50.

In [51]:
# Проверка объединения текстового и embedding-пула.
EMBEDDING_UNION_K = 5_000
assert EMBEDDING_TOP_K >= EMBEDDING_UNION_K
assert all(
    extended_top_indices[name].shape[1] >= EMBEDDING_UNION_K
    for name in SOURCE_NAMES
)

text_recall_sum = 0.0
embedding_recall_sum = 0.0
union_recall_sum = 0.0
new_embedding_hits = 0
total_relevant = 0

for row, query_id in enumerate(saved_query_ids):
    relevant = relevant_indices_by_query[query_id]
    text_rows = [
        extended_top_indices[name][row, :EMBEDDING_UNION_K]
        for name in SOURCE_NAMES
    ]
    embedding_row = embedding_top_indices[row, :EMBEDDING_UNION_K]
    text_hits = embedding_hits = union_hits = 0
    for item_index in relevant:
        in_text = any(np.any(source_row == item_index) for source_row in text_rows)
        in_embedding = np.any(embedding_row == item_index)
        text_hits += in_text
        embedding_hits += in_embedding
        union_hits += in_text or in_embedding
        new_embedding_hits += in_embedding and not in_text
        total_relevant += 1
    text_recall_sum += text_hits / len(relevant)
    embedding_recall_sum += embedding_hits / len(relevant)
    union_recall_sum += union_hits / len(relevant)

    if (row + 1) % 5_000 == 0 or row + 1 == len(saved_query_ids):
        print(f"Проверка объединения: {row + 1:,}/{len(saved_query_ids):,}")

embedding_union_results = pd.DataFrame([{
    "text_pool_recall_at_5000": text_recall_sum / len(saved_query_ids),
    "embedding_recall_at_5000": embedding_recall_sum / len(saved_query_ids),
    "union_recall_at_5000": union_recall_sum / len(saved_query_ids),
    "new_relevant_hits_from_embeddings": new_embedding_hits,
    "all_relevant_pairs": total_relevant,
}])
display(embedding_union_results)


Проверка объединения: 5,000/67,186
Проверка объединения: 10,000/67,186
Проверка объединения: 15,000/67,186
Проверка объединения: 20,000/67,186
Проверка объединения: 25,000/67,186
Проверка объединения: 30,000/67,186
Проверка объединения: 35,000/67,186
Проверка объединения: 40,000/67,186
Проверка объединения: 45,000/67,186
Проверка объединения: 50,000/67,186
Проверка объединения: 55,000/67,186
Проверка объединения: 60,000/67,186
Проверка объединения: 65,000/67,186
Проверка объединения: 67,186/67,186


,text_pool_recall_at_5000,embedding_recall_at_5000,union_recall_at_5000,new_relevant_hits_from_embeddings,all_relevant_pairs
0,0.8861,0.8402,0.9263,3496,88871


### 20.1. Новые кандидаты при прежней формуле

In [52]:
# Embedding-кандидаты с неизменной текущей формулой.
EMBEDDING_FORMULA_K = 5_000
EMBEDDING_FORMULA_FILTER_BONUS = 0.30
EMBEDDING_FORMULA_FILTER_THRESHOLD = 0.90
assert EMBEDDING_TOP_K >= EMBEDDING_FORMULA_K

def candidates_with_embedding_pool(row):
    base_candidates, base_scores, _ = parameter_hybrid_parts(
        row, base_top_k=None
    )
    embedding_candidates = embedding_top_indices[row, :EMBEDDING_FORMULA_K]
    embedding_candidates = embedding_candidates[embedding_candidates >= 0]
    candidates = np.union1d(base_candidates, embedding_candidates)
    scores = location_bonus(
        candidates, valid_query_locations[row], best_location_bonus, item_locations
    ).astype(np.float32)
    base_positions = np.searchsorted(candidates, base_candidates)
    scores[base_positions] = base_scores
    return candidates, scores

item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"], unique_parameter_filters
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
candidate_counts = np.empty(len(saved_query_ids), dtype=np.int32)
recall_sum = 0.0

for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    candidate_counts[row] = len(candidates)
    top50 = candidates[top_positions(scores, TOP_50)]
    recall_sum += recall_for_item_indices(row, top50)

candidate_mark = np.zeros(len(train_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    mark_value = filter_code + 1
    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    if required_matches[filter_code] == 0 or len(group_candidates) == 0:
        matching_items = np.empty(0, dtype=np.int32)
    else:
        overlap = (
            filter_matrix[filter_code] @ item_matrix[group_candidates].T
        ).tocsr()
        matching_items = group_candidates[
            overlap.indices[overlap.data >= required_matches[filter_code]]
        ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        candidate_counts[row] = len(candidates)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        top50 = candidates[top_positions(
            scores + EMBEDDING_FORMULA_FILTER_BONUS * filter_match, TOP_50
        )]
        recall_sum += recall_for_item_indices(row, top50)

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(f"Формула с embeddings: {position:,}/{len(unique_parameter_filters):,} фильтров")

embedding_current_formula_results = pd.DataFrame([{
    "previous_recall_at_50": 0.7566,
    "recall_at_50_with_embedding_candidates": recall_sum / len(saved_query_ids),
    "candidate_count_median": int(np.median(candidate_counts)),
    "candidate_count_p90": int(np.percentile(candidate_counts, 90)),
    "candidate_count_max": int(candidate_counts.max()),
}])
display(embedding_current_formula_results)

del candidate_mark, item_matrix, filter_matrix
gc.collect()


Формула с embeddings: 100/1,196 фильтров
Формула с embeddings: 200/1,196 фильтров
Формула с embeddings: 300/1,196 фильтров
Формула с embeddings: 400/1,196 фильтров
Формула с embeddings: 500/1,196 фильтров
Формула с embeddings: 600/1,196 фильтров
Формула с embeddings: 700/1,196 фильтров
Формула с embeddings: 800/1,196 фильтров
Формула с embeddings: 900/1,196 фильтров
Формула с embeddings: 1,000/1,196 фильтров
Формула с embeddings: 1,100/1,196 фильтров
Формула с embeddings: 1,196/1,196 фильтров


,previous_recall_at_50,recall_at_50_with_embedding_candidates,candidate_count_median,candidate_count_p90,candidate_count_max
0,0.7566,0.7669,14874,19060,25909


0

### 20.2. Вес косинусной близости

In [53]:
# Подбор веса cosine similarity в дополнение к текущей формуле.
# Итоговый скор: текущий скор + 0.30 * совпадение фильтра + c * cosine_similarity.
EMBEDDING_COSINE_COEFFICIENTS = np.array(
    [0.00, 0.01, 0.025, 0.05, 0.10, 0.20, 0.40, 0.80],
    dtype=np.float32,
)


def cosine_values_in_embedding_pool(row, candidates):
    embedding_indices = np.asarray(
        embedding_top_indices[row, :EMBEDDING_FORMULA_K]
    )
    embedding_scores = np.asarray(
        embedding_top_scores[row, :EMBEDDING_FORMULA_K], dtype=np.float32
    )
    valid = embedding_indices >= 0
    embedding_indices = embedding_indices[valid]
    embedding_scores = embedding_scores[valid]

    cosine_values = np.zeros(len(candidates), dtype=np.float32)
    if len(embedding_scores) and embedding_scores[0] > 0:
        positions = np.searchsorted(candidates, embedding_indices)
        cosine_values[positions] = embedding_scores / embedding_scores[0]
    return cosine_values


def add_cosine_grid_result(row, candidates, fixed_scores, filter_match):
    cosine_values = cosine_values_in_embedding_pool(row, candidates)

    # У всех не-embedding кандидатов cosine равен нулю.
    # Достаточно сохранить из них только уже сильнейшие 50:
    # остальные не смогут попасть в финальный top-50 ни при каком c >= 0.
    non_embedding = np.flatnonzero(cosine_values == 0)
    keep_non_embedding = non_embedding[
        top_positions(fixed_scores[non_embedding], TOP_50)
    ]
    keep = np.unique(np.concatenate([
        np.flatnonzero(cosine_values > 0),
        keep_non_embedding,
    ]))

    score_grid = (
        fixed_scores[keep][None, :]
        + EMBEDDING_COSINE_COEFFICIENTS[:, None] * cosine_values[keep][None, :]
    )
    top50_positions = np.argpartition(
        score_grid, -TOP_50, axis=1
    )[:, -TOP_50:]

    for coefficient_position, positions in enumerate(top50_positions):
        cosine_recall_sums[coefficient_position] += recall_for_item_indices(
            row, candidates[keep[positions]]
        )


item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"], unique_parameter_filters
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
cosine_recall_sums = np.zeros(
    len(EMBEDDING_COSINE_COEFFICIENTS), dtype=np.float64
)

for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    add_cosine_grid_result(
        row, candidates, scores, np.zeros(len(candidates), dtype=bool)
    )

candidate_mark = np.zeros(len(train_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    mark_value = filter_code + 1

    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    overlap = (filter_matrix[filter_code] @ item_matrix[group_candidates].T).tocsr()
    matching_items = group_candidates[
        overlap.indices[overlap.data >= required_matches[filter_code]]
    ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        add_cosine_grid_result(
            row,
            candidates,
            scores + EMBEDDING_FORMULA_FILTER_BONUS * filter_match,
            filter_match,
        )

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(f"Cosine grid: {position:,}/{len(unique_parameter_filters):,} фильтров")

embedding_cosine_grid_results = (
    pd.DataFrame({
        "cosine_coefficient": EMBEDDING_COSINE_COEFFICIENTS,
        "recall_at_50": cosine_recall_sums / len(saved_query_ids),
    })
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(embedding_cosine_grid_results)

del candidate_mark, item_matrix, filter_matrix
gc.collect()


Cosine grid: 100/1,196 фильтров
Cosine grid: 200/1,196 фильтров
Cosine grid: 300/1,196 фильтров
Cosine grid: 400/1,196 фильтров
Cosine grid: 500/1,196 фильтров
Cosine grid: 600/1,196 фильтров
Cosine grid: 700/1,196 фильтров
Cosine grid: 800/1,196 фильтров
Cosine grid: 900/1,196 фильтров
Cosine grid: 1,000/1,196 фильтров
Cosine grid: 1,100/1,196 фильтров
Cosine grid: 1,196/1,196 фильтров


,cosine_coefficient,recall_at_50
0,0.0500,0.7734
1,0.1000,0.7731
2,0.0250,0.7723
3,0.2000,0.7713
4,0.0100,0.7708
5,0.0000,0.7675
6,0.4000,0.7570
7,0.8000,0.7201


0

## 21. Проверка категориального сигнала

По положительным парам обучающей части оцениваем связь категории запроса с категорией объявления и проверяем бонус на validation.

In [54]:
# Бонус за соответствие search_category → item_category_id.
# Вероятности строятся только по relevance_train, то есть без кликов validation.
category_training_pairs = (
    relevance_train
    .merge(
        queries_train[["train_query_id", "search_category"]],
        on="train_query_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        train_items[["item_id", "item_category_id"]],
        on="item_id",
        how="left",
        validate="many_to_one",
    )
)
category_probability = (
    category_training_pairs
    .groupby(["search_category", "item_category_id"])
    .size()
    .unstack(fill_value=0)
)
category_probability = category_probability.div(
    category_probability.sum(axis=1), axis=0
)
display(category_probability.round(3))

valid_query_categories = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_category"]
    .to_numpy()
)
category_score_lookup = category_probability.reindex(
    valid_query_categories, fill_value=0
).fillna(0).to_numpy(dtype=np.float32)
item_category_codes = category_probability.columns.get_indexer(
    train_items["item_category_id"]
)
CATEGORY_KEEP_PER_GROUP = 300
CATEGORY_COEFFICIENTS = np.array(
    [0.00, 0.025, 0.05, 0.10, 0.20, 0.40, 0.80, 1.20],
    dtype=np.float32,
)


def category_scores_for_candidates(row, candidates):
    category_codes = item_category_codes[candidates]
    category_scores = np.zeros(len(candidates), dtype=np.float32)
    known = category_codes >= 0
    category_scores[known] = category_score_lookup[
        row, category_codes[known]
    ]
    return category_scores


def add_category_grid_result(row, candidates, fixed_scores):
    category_scores = category_scores_for_candidates(row, candidates)

    # Внутри одной item-категории category-бонус одинаков.
    # Оставляем её сильнейшие 300 объявлений с учётом всех уже добавленных сигналов.
    keep_parts = []
    for category_code in np.unique(item_category_codes[candidates]):
        positions = np.flatnonzero(
            item_category_codes[candidates] == category_code
        )
        keep_parts.append(
            positions[top_positions(fixed_scores[positions], CATEGORY_KEEP_PER_GROUP)]
        )
    keep = np.unique(np.concatenate(keep_parts))

    score_grid = (
        fixed_scores[keep][None, :]
        + CATEGORY_COEFFICIENTS[:, None] * category_scores[keep][None, :]
    )
    top50_positions = np.argpartition(
        score_grid, -TOP_50, axis=1
    )[:, -TOP_50:]

    for coefficient_position, positions in enumerate(top50_positions):
        category_recall_sums[coefficient_position] += recall_for_item_indices(
            row, candidates[keep[positions]]
        )


item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"], unique_parameter_filters
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
category_recall_sums = np.zeros(len(CATEGORY_COEFFICIENTS), dtype=np.float64)

for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    fixed_scores = scores + 0.05 * cosine_values_in_embedding_pool(
        row, candidates
    )
    add_category_grid_result(row, candidates, fixed_scores)

candidate_mark = np.zeros(len(train_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    mark_value = filter_code + 1

    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    overlap = (filter_matrix[filter_code] @ item_matrix[group_candidates].T).tocsr()
    matching_items = group_candidates[
        overlap.indices[overlap.data >= required_matches[filter_code]]
    ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        fixed_scores = (
            scores
            + EMBEDDING_FORMULA_FILTER_BONUS * filter_match
            + 0.05 * cosine_values_in_embedding_pool(row, candidates)
        )
        add_category_grid_result(row, candidates, fixed_scores)

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(f"Category grid: {position:,}/{len(unique_parameter_filters):,} фильтров")

category_grid_results = (
    pd.DataFrame({
        "category_coefficient": CATEGORY_COEFFICIENTS,
        "recall_at_50": category_recall_sums / len(saved_query_ids),
    })
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(category_grid_results)

del category_training_pairs, candidate_mark, item_matrix, filter_matrix
gc.collect()


item_category_id,10,19,27,28,40,81,112,114
search_category,,,,,,,,
0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
10,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
81,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
114,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


Category grid: 100/1,196 фильтров
Category grid: 200/1,196 фильтров
Category grid: 300/1,196 фильтров
Category grid: 400/1,196 фильтров
Category grid: 500/1,196 фильтров
Category grid: 600/1,196 фильтров
Category grid: 700/1,196 фильтров
Category grid: 800/1,196 фильтров
Category grid: 900/1,196 фильтров
Category grid: 1,000/1,196 фильтров
Category grid: 1,100/1,196 фильтров
Category grid: 1,196/1,196 фильтров


,category_coefficient,recall_at_50
0,0.0000,0.7734
1,0.0250,0.7734
2,0.0500,0.7734
3,0.1000,0.7734
4,0.2000,0.7734
5,0.4000,0.7734
6,0.8000,0.7734
7,1.2000,0.7734


0

## 22. Географическая иерархия

Сравниваем точный ID локации, фиксированный бонус по расстоянию и статистику выбора города для регионального запроса.

In [73]:
# Два альтернативных способа использовать локацию.
# Все статистики ниже строятся только по relevance_train; validation в них не участвует.

EARTH_RADIUS_KM = 6_371.0088
GEO_DISTANCE_LIMITS_KM = np.array([25.0, 75.0], dtype=np.float32)

# Ручной вариант — лишь исходная гипотеза для сравнения.
# 1.0: тот же город; 0.55: до 25 км; 0.25: от 25 до 75 км.
STATIC_GEO_SCORES = np.array([1.00, 0.55, 0.25, 0.00], dtype=np.float32)

geo_items = train_items[
    ["item_id", "item_location_id", "item_latitude", "item_longitude"]
].drop_duplicates("item_id").copy()
geo_items = geo_items.astype(
    {"item_latitude": "float64", "item_longitude": "float64"}
)

# Координата локации — медиана координат её объявлений:
# отдельное объявление может быть выставлено не в точке города.
geo_city_centers = (
    geo_items.dropna(
        subset=["item_location_id", "item_latitude", "item_longitude"]
    )
    .groupby("item_location_id")[["item_latitude", "item_longitude"]]
    .median()
    .sort_index()
)
geo_city_ids = pd.Index(geo_city_centers.index)
geo_catalog_location_ids = pd.Index(
    pd.unique(geo_items["item_location_id"].dropna())
)

geo_latitude = np.deg2rad(geo_city_centers["item_latitude"].to_numpy())
geo_longitude = np.deg2rad(geo_city_centers["item_longitude"].to_numpy())
geo_vectors = np.column_stack(
    [
        np.cos(geo_latitude) * np.cos(geo_longitude),
        np.cos(geo_latitude) * np.sin(geo_longitude),
        np.sin(geo_latitude),
    ]
).astype(np.float32)
geo_city_distances_km = (
    np.arccos(np.clip(geo_vectors @ geo_vectors.T, -1.0, 1.0))
    * EARTH_RADIUS_KM
).astype(np.float32)
np.fill_diagonal(geo_city_distances_km, 0.0)
del geo_vectors

geo_item_city_codes = geo_city_ids.get_indexer(item_locations)
geo_valid_query_city_codes = geo_city_ids.get_indexer(valid_query_locations)
geo_valid_is_city = np.isin(
    valid_query_locations, geo_catalog_location_ids
)


def geo_distance_bins(query_city_codes, item_city_codes):
    """0: тот же город; 1: до 25 км; 2: 25–75 км; 3: дальше/нет координат."""
    result = np.full(len(item_city_codes), 3, dtype=np.int8)
    known = (query_city_codes >= 0) & (item_city_codes >= 0)
    if not np.any(known):
        return result

    distances = geo_city_distances_km[
        query_city_codes[known], item_city_codes[known]
    ]
    result[known] = np.select(
        [
            query_city_codes[known] == item_city_codes[known],
            distances <= GEO_DISTANCE_LIMITS_KM[0],
            distances <= GEO_DISTANCE_LIMITS_KM[1],
        ],
        [0, 1, 2],
        default=3,
    ).astype(np.int8)
    return result


# Соединяем только клики обучающей части с признаками их запроса и объявления.
geo_training_pairs = (
    relevance_train
    .merge(
        queries_train[["train_query_id", "search_location_id"]],
        on="train_query_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        geo_items[["item_id", "item_location_id"]],
        on="item_id",
        how="left",
        validate="many_to_one",
    )
)
geo_train_query_city_codes = geo_city_ids.get_indexer(
    geo_training_pairs["search_location_id"]
)
geo_train_item_city_codes = geo_city_ids.get_indexer(
    geo_training_pairs["item_location_id"]
)
geo_city_pair_mask = (
    (geo_train_query_city_codes >= 0) & (geo_train_item_city_codes >= 0)
)
geo_positive_bins = geo_distance_bins(
    geo_train_query_city_codes[geo_city_pair_mask],
    geo_train_item_city_codes[geo_city_pair_mask],
)
geo_positive_counts = np.bincount(
    geo_positive_bins, minlength=len(STATIC_GEO_SCORES)
).astype(np.float64)

# Для каждого города запроса оцениваем, как были бы распределены все объявления
# каталога. Так редкая дистанция получает высокий балл лишь при избытке кликов,
# а не только потому, что таких объявлений в принципе мало.
geo_catalog_city_counts = np.bincount(
    geo_item_city_codes[geo_item_city_codes >= 0],
    minlength=len(geo_city_ids),
).astype(np.float64)
geo_query_city_counts = np.bincount(
    geo_train_query_city_codes[geo_city_pair_mask],
    minlength=len(geo_city_ids),
).astype(np.float64)
geo_catalog_bin_counts = np.zeros(len(STATIC_GEO_SCORES), dtype=np.float64)

for city_code, query_count in enumerate(geo_query_city_counts):
    if query_count == 0:
        continue
    all_city_codes = np.arange(len(geo_city_ids), dtype=np.int32)
    bins = geo_distance_bins(
        np.full(len(all_city_codes), city_code, dtype=np.int32),
        all_city_codes,
    )
    geo_catalog_bin_counts += query_count * np.bincount(
        bins,
        weights=geo_catalog_city_counts,
        minlength=len(STATIC_GEO_SCORES),
    )

geo_positive_share = (geo_positive_counts + 1.0) / (
    geo_positive_counts.sum() + len(geo_positive_counts)
)
geo_catalog_share = (geo_catalog_bin_counts + 1.0) / (
    geo_catalog_bin_counts.sum() + len(geo_catalog_bin_counts)
)
geo_enrichment = geo_positive_share / geo_catalog_share
DYNAMIC_GEO_SCORES = (
    geo_enrichment / geo_enrichment[0]
).clip(0.0, 1.0).astype(np.float32)
DYNAMIC_GEO_SCORES[0] = 1.0

# Для запросов, чья локация — не город из каталога, учим отдельный
# «регион → город» скор. Неизвестный регион получает нулевой бонус.
geo_region_pair_mask = (
    (geo_train_query_city_codes < 0) & (geo_train_item_city_codes >= 0)
)
geo_region_ids = pd.Index(
    pd.unique(
        geo_training_pairs.loc[
            geo_region_pair_mask, "search_location_id"
        ].dropna()
    )
)
geo_train_region_codes = geo_region_ids.get_indexer(
    geo_training_pairs.loc[
        geo_region_pair_mask, "search_location_id"
    ]
)
geo_region_city_counts = np.zeros(
    (len(geo_region_ids), len(geo_city_ids)), dtype=np.int32
)
np.add.at(
    geo_region_city_counts,
    (
        geo_train_region_codes,
        geo_train_item_city_codes[geo_region_pair_mask],
    ),
    1,
)

DYNAMIC_REGION_SCORES = np.zeros_like(
    geo_region_city_counts, dtype=np.float32
)
for region_code, city_counts in enumerate(geo_region_city_counts):
    maximum = city_counts.max()
    if maximum:
        # Логарифм не даёт одному частому городу полностью подавить остальные.
        DYNAMIC_REGION_SCORES[region_code] = (
            np.log1p(city_counts) / np.log1p(maximum)
        )
geo_valid_region_codes = geo_region_ids.get_indexer(valid_query_locations)

# Отдельные имена сохраняют validation-геометрию,
# если benchmark-ячейки позже переопределят общие geo_* переменные.
validation_geo_city_ids = geo_city_ids
validation_geo_item_city_codes = geo_item_city_codes
validation_geo_city_distances_km = geo_city_distances_km
validation_geo_region_codes = geo_valid_region_codes
validation_region_city_scores = DYNAMIC_REGION_SCORES

geo_band_report = pd.DataFrame(
    {
        "distance_band": [
            "same city", "up to 25 km", "25 to 75 km", "far or unknown"
        ],
        "selected_pairs": geo_positive_counts.astype(int),
        "catalog_exposure": geo_catalog_bin_counts.round().astype(np.int64),
        "enrichment": geo_enrichment.round(3),
        "static_score": STATIC_GEO_SCORES,
        "learned_score": DYNAMIC_GEO_SCORES.round(3),
    }
)
display(geo_band_report)
print(
    "Центры локаций:",
    f"{len(geo_city_ids):,}; "
    f"регионов с обучающими кликами: {len(geo_region_ids):,}"
)


,distance_band,selected_pairs,catalog_exposure,enrichment,static_score,learned_score
0,same city,309999,1695090408,63.0610,1.0000,1.0000
1,up to 25 km,9624,626930051,5.2940,0.5500,0.0840
2,25 to 75 km,7530,1347806405,1.9270,0.2500,0.0310
3,far or unknown,7279,111650687536,0.0220,0.0000,0.0000


Центры локаций: 2,637; регионов с обучающими кликами: 445


In [57]:
# Сравнение вариантов на той же формуле, кандидатах и top-50.
# Вариант "direct" воспроизводит текущий прямой матч локации.
# Его вклад вычитается из ранее сохранённого скора и добавляется заново
# вместе с двумя альтернативами, поэтому ничего не удваивается.

GEO_METHOD_NAMES = ("direct", "static_distance", "learned_geo")
GEO_LOCATION_COEFFICIENT = best_location_bonus


def geo_scores_for_candidates(row, candidates, method):
    item_city_codes = geo_item_city_codes[candidates]
    query_city_code = geo_valid_query_city_codes[row]
    scores = np.zeros(len(candidates), dtype=np.float32)

    if method == "direct":
        return (
            item_locations[candidates] == valid_query_locations[row]
        ).astype(np.float32)

    if query_city_code >= 0:
        bins = geo_distance_bins(
            np.full(len(candidates), query_city_code, dtype=np.int32),
            item_city_codes,
        )
        lookup = (
            STATIC_GEO_SCORES
            if method == "static_distance"
            else DYNAMIC_GEO_SCORES
        )
        scores = lookup[bins]
        return scores

    if method == "learned_geo":
        region_code = geo_valid_region_codes[row]
        known_city = item_city_codes >= 0
        if region_code >= 0 and np.any(known_city):
            scores[known_city] = DYNAMIC_REGION_SCORES[
                region_code, item_city_codes[known_city]
            ]
    return scores


def add_geo_comparison(row, candidates, base_scores):
    geo_score_grid = np.vstack(
        [
            geo_scores_for_candidates(row, candidates, method)
            for method in GEO_METHOD_NAMES
        ]
    )
    score_grid = (
        base_scores[None, :]
        + GEO_LOCATION_COEFFICIENT * geo_score_grid
    )
    top50_positions = np.argpartition(
        score_grid, -TOP_50, axis=1
    )[:, -TOP_50:]

    segment = int(not geo_valid_is_city[row])
    for method_position, positions in enumerate(top50_positions):
        value = recall_for_item_indices(row, candidates[positions])
        geo_recall_sums[method_position] += value
        geo_recall_by_segment[method_position, segment] += value


item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"], unique_parameter_filters
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
geo_recall_sums = np.zeros(len(GEO_METHOD_NAMES), dtype=np.float64)
geo_recall_by_segment = np.zeros(
    (len(GEO_METHOD_NAMES), 2), dtype=np.float64
)

# У запросов без фильтров совпадение фильтра равно нулю.
for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    direct_score = location_bonus(
        candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    base_scores = (
        scores
        - direct_score
        + 0.05 * cosine_values_in_embedding_pool(row, candidates)
    )
    add_geo_comparison(row, candidates, base_scores)

candidate_mark = np.zeros(len(train_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    mark_value = filter_code + 1

    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    overlap = (
        filter_matrix[filter_code] @ item_matrix[group_candidates].T
    ).tocsr()
    matching_items = group_candidates[
        overlap.indices[
            overlap.data >= required_matches[filter_code]
        ]
    ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        direct_score = location_bonus(
            candidates,
            valid_query_locations[row],
            best_location_bonus,
            item_locations,
        )
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        )
        base_scores = (
            scores
            - direct_score
            + EMBEDDING_FORMULA_FILTER_BONUS * filter_match
            + 0.05 * cosine_values_in_embedding_pool(row, candidates)
        )
        add_geo_comparison(row, candidates, base_scores)

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(
            f"Гео-сравнение: {position:,}/"
            f"{len(unique_parameter_filters):,} фильтров"
        )

geo_city_query_count = int(geo_valid_is_city.sum())
geo_region_query_count = len(geo_valid_is_city) - geo_city_query_count
geo_comparison_results = pd.DataFrame(
    {
        "geo_method": GEO_METHOD_NAMES,
        "recall_at_50": geo_recall_sums / len(saved_query_ids),
        "city_query_recall_at_50": (
            geo_recall_by_segment[:, 0] / geo_city_query_count
        ),
        "region_query_recall_at_50": (
            geo_recall_by_segment[:, 1] / geo_region_query_count
        ),
    }
).sort_values("recall_at_50", ascending=False).reset_index(drop=True)
display(geo_comparison_results)

del candidate_mark, item_matrix, filter_matrix
gc.collect()


Гео-сравнение: 100/1,196 фильтров
Гео-сравнение: 200/1,196 фильтров
Гео-сравнение: 300/1,196 фильтров
Гео-сравнение: 400/1,196 фильтров
Гео-сравнение: 500/1,196 фильтров
Гео-сравнение: 600/1,196 фильтров
Гео-сравнение: 700/1,196 фильтров
Гео-сравнение: 800/1,196 фильтров
Гео-сравнение: 900/1,196 фильтров
Гео-сравнение: 1,000/1,196 фильтров
Гео-сравнение: 1,100/1,196 фильтров
Гео-сравнение: 1,196/1,196 фильтров


,geo_method,recall_at_50,city_query_recall_at_50,region_query_recall_at_50
0,learned_geo,0.8204,0.8421,0.6658
1,static_distance,0.7910,0.8587,0.3091
2,direct,0.7735,0.8388,0.3091


1244

In [58]:
# Смешанный вариант: расстояние для городского запроса,
# выученная связь «регион → город» для регионального.
# Это точный Recall: группы запросов не пересекаются, а их суммы уже
# получены в предыдущей ячейке на тех же кандидатах и top-50.

static_position = GEO_METHOD_NAMES.index("static_distance")
learned_position = GEO_METHOD_NAMES.index("learned_geo")

mixed_geo_recall = (
    geo_recall_by_segment[static_position, 0]
    + geo_recall_by_segment[learned_position, 1]
) / len(saved_query_ids)

geo_mixed_result = pd.DataFrame(
    [{
        "geo_method": "static_for_city__learned_for_region",
        "recall_at_50": mixed_geo_recall,
        "city_query_recall_at_50": (
            geo_recall_by_segment[static_position, 0]
            / geo_city_query_count
        ),
        "region_query_recall_at_50": (
            geo_recall_by_segment[learned_position, 1]
            / geo_region_query_count
        ),
    }]
)
display(geo_mixed_result)


,geo_method,recall_at_50,city_query_recall_at_50,region_query_recall_at_50
0,static_for_city__learned_for_region,0.8349,0.8587,0.6658


### 22.1. Повторный подбор бонуса фильтра с E5 и географией

In [59]:
# Подбор коэффициента filter_match после добавления E5 и смешанной географии.
# Все варианты используют одинаковый кандидатный пул; меняется только коэффициент.

FILTER_COEFFICIENTS = np.array(
    [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 1.00],
    dtype=np.float32,
)


def mixed_geo_scores_for_candidates(row, candidates):
    method = (
        "static_distance"
        if geo_valid_is_city[row]
        else "learned_geo"
    )
    return geo_scores_for_candidates(row, candidates, method)


def add_filter_grid_result(row, candidates, fixed_scores, filter_match):
    score_grid = (
        fixed_scores[None, :]
        + FILTER_COEFFICIENTS[:, None] * filter_match[None, :]
    )
    top50_positions = np.argpartition(
        score_grid, -TOP_50, axis=1
    )[:, -TOP_50:]

    for coefficient_position, positions in enumerate(top50_positions):
        filter_grid_recall_sums[coefficient_position] += (
            recall_for_item_indices(row, candidates[positions])
        )


item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"], unique_parameter_filters
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
filter_grid_recall_sums = np.zeros(
    len(FILTER_COEFFICIENTS), dtype=np.float64
)

for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    direct_score = location_bonus(
        candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    fixed_scores = (
        scores
        - direct_score
        + GEO_LOCATION_COEFFICIENT
        * mixed_geo_scores_for_candidates(row, candidates)
        + 0.05 * cosine_values_in_embedding_pool(row, candidates)
    )
    add_filter_grid_result(
        row,
        candidates,
        fixed_scores,
        np.zeros(len(candidates), dtype=np.float32),
    )

candidate_mark = np.zeros(len(train_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    mark_value = filter_code + 1

    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = mark_value

    group_candidates = np.flatnonzero(candidate_mark == mark_value)
    overlap = (
        filter_matrix[filter_code] @ item_matrix[group_candidates].T
    ).tocsr()
    matching_items = group_candidates[
        overlap.indices[
            overlap.data >= required_matches[filter_code]
        ]
    ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        direct_score = location_bonus(
            candidates,
            valid_query_locations[row],
            best_location_bonus,
            item_locations,
        )
        fixed_scores = (
            scores
            - direct_score
            + GEO_LOCATION_COEFFICIENT
            * mixed_geo_scores_for_candidates(row, candidates)
            + 0.05 * cosine_values_in_embedding_pool(row, candidates)
        )
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        ).astype(np.float32)
        add_filter_grid_result(
            row, candidates, fixed_scores, filter_match
        )

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(
            f"Filter grid: {position:,}/"
            f"{len(unique_parameter_filters):,} фильтров"
        )

filter_grid_results = (
    pd.DataFrame(
        {
            "filter_coefficient": FILTER_COEFFICIENTS,
            "recall_at_50": (
                filter_grid_recall_sums / len(saved_query_ids)
            ),
        }
    )
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(filter_grid_results)

del candidate_mark, item_matrix, filter_matrix
gc.collect()


Filter grid: 100/1,196 фильтров
Filter grid: 200/1,196 фильтров
Filter grid: 300/1,196 фильтров
Filter grid: 400/1,196 фильтров
Filter grid: 500/1,196 фильтров
Filter grid: 600/1,196 фильтров
Filter grid: 700/1,196 фильтров
Filter grid: 800/1,196 фильтров
Filter grid: 900/1,196 фильтров
Filter grid: 1,000/1,196 фильтров
Filter grid: 1,100/1,196 фильтров
Filter grid: 1,196/1,196 фильтров


,filter_coefficient,recall_at_50
0,0.4000,0.8354
1,0.3000,0.8349
2,0.5000,0.8343
3,0.2000,0.8330
4,0.7000,0.8310
5,0.1000,0.8281
6,0.0500,0.8236
7,1.0000,0.8236
8,0.0000,0.8156


0

## 23. Benchmark: текст, E5 и смешанная география

Готовим источники на benchmark-корпусе и отдельный CSV. Сохранённые поисковые массивы используются повторно, если уже существуют.

In [61]:
# Финальные источники кандидатов на benchmark_items.
# E5-векторы и его top-5000 сохраняются и переживают перезапуск ядра.

FINAL_TOP_K = 5_000
FINAL_FILTER_BONUS = 0.40
FINAL_FILTER_THRESHOLD = 0.90
FINAL_COSINE_COEFFICIENT = 0.05
FINAL_ARTIFACT_DIR = Path("artifacts/retrieval/benchmark_mixed_geo_e5")
FINAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for variable_name in [
    "item_embeddings", "unique_query_embeddings", "embedding_index",
    "embedding_top_indices", "embedding_top_scores",
    "extended_top_indices", "extended_top_scores",
]:
    globals().pop(variable_name, None)
gc.collect()

final_query_ids = benchmark_queries["query_id"].astype(str).to_numpy()
final_item_ids = benchmark_items["item_id"].astype(str).to_numpy()
final_query_text = benchmark_queries["search_query"].fillna("")
final_query_locations = benchmark_queries["search_location_id"].to_numpy()
final_item_locations = benchmark_items["item_location_id"].to_numpy()

# Четыре текстовых источника и BM25 по параметрам используют уже проверенные функции.
BENCHMARK_FILTER_TOP_K = FINAL_TOP_K
final_title = benchmark_items["item_title_raw"].fillna("")
final_title_description = combine_text(
    benchmark_items, ["item_title_raw", "item_description_raw"]
)
final_source_configs = [
    ("bm25_title_description", "bm25", final_title_description, 128, None),
    ("char_tfidf_title", "char_tfidf", final_title, 64, make_tfidf_vectorizer("char")),
    ("bm25_title", "bm25", final_title, 128, None),
    ("word_tfidf_title", "word_tfidf", final_title, 128, make_tfidf_vectorizer("word")),
]
final_source_names = tuple(config[0] for config in final_source_configs)
final_source_indices, final_source_scores = {}, {}
for name, method, item_text, batch_size, vectorizer in final_source_configs:
    indices, scores = build_benchmark_top_k(
        name, method, item_text, final_query_text, batch_size, vectorizer
    )
    final_source_indices[name] = indices
    final_source_scores[name] = scores

final_filter_text = benchmark_queries["search_infm_params_text"].fillna("")
final_unique_filters = pd.Index(
    final_filter_text.loc[final_filter_text.ne("")].unique()
)
final_filter_codes = final_unique_filters.get_indexer(final_filter_text)
_, parameter_item_matrix, final_parameter_indices, _ = retrieve_top_k(
    "bm25",
    benchmark_items["item_infm_params_text"].fillna(""),
    final_unique_filters,
    FINAL_TOP_K,
    batch_size=32,
    source_name="bm25_parameters",
)
del parameter_item_matrix, final_title, final_title_description
gc.collect()

# Для E5 повторяем validation-эксперимент: берём исходные тексты из Parquet.
raw_items = pd.read_parquet(
    "data/benchmark_items.parquet",
    columns=["item_id", "item_title_raw", "item_description_raw"],
)
raw_items = raw_items.assign(item_id=raw_items["item_id"].astype(str))
raw_items = raw_items.set_index("item_id").reindex(final_item_ids)
raw_queries = pd.read_parquet(
    "data/benchmark_queries.parquet", columns=["query_id", "search_query"]
)
assert np.array_equal(
    raw_queries["query_id"].astype(str).to_numpy(), final_query_ids
)

embedding_item_texts = (
    "passage: "
    + raw_items["item_title_raw"].fillna("").astype(str).str.strip()
    + ". "
    + raw_items["item_description_raw"].fillna("").astype(str).str.strip()
).to_numpy(dtype=str)
embedding_query_texts = (
    "query: "
    + raw_queries["search_query"].fillna("").astype(str).str.strip()
).to_numpy(dtype=str)

embedding_model = SentenceTransformer(str(MODEL_DIR), device=DEVICE)
if DEVICE == "cuda":
    embedding_model.half()
benchmark_item_embeddings = save_embeddings(
    embedding_item_texts,
    FINAL_ARTIFACT_DIR / "item_embeddings_maxlen128.npy",
    ITEM_BATCH_SIZE,
    ITEM_MAX_SEQ_LENGTH,
)
benchmark_query_embeddings = save_embeddings(
    embedding_query_texts,
    FINAL_ARTIFACT_DIR / "query_embeddings_maxlen128.npy",
    QUERY_BATCH_SIZE,
    QUERY_MAX_SEQ_LENGTH,
)

final_index_path = FINAL_ARTIFACT_DIR / "hnsw_cosine.index"
final_index_done = FINAL_ARTIFACT_DIR / "hnsw_cosine_complete.json"
if final_index_path.exists() and final_index_done.exists():
    final_embedding_index = faiss.read_index(str(final_index_path))
    assert final_embedding_index.ntotal == len(final_item_ids)
    print("FAISS-индекс benchmark загружен.")
else:
    faiss.omp_set_num_threads(max(1, (os.cpu_count() or 2) - 1))
    final_embedding_index = faiss.IndexHNSWFlat(
        EMBEDDING_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT
    )
    final_embedding_index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
    for start in range(0, len(benchmark_item_embeddings), FAISS_ADD_BATCH_SIZE):
        stop = min(start + FAISS_ADD_BATCH_SIZE, len(benchmark_item_embeddings))
        final_embedding_index.add(np.ascontiguousarray(
            benchmark_item_embeddings[start:stop], dtype=np.float32
        ))
        print(f"FAISS benchmark: {stop:,}/{len(benchmark_item_embeddings):,}")
    faiss.write_index(final_embedding_index, str(final_index_path))
    final_index_done.write_text(
        json.dumps({"items": len(final_item_ids), "dimension": EMBEDDING_DIM}),
        encoding="utf-8",
    )

final_embedding_index.hnsw.efSearch = HNSW_EF_SEARCH
embedding_indices_path = FINAL_ARTIFACT_DIR / "top5000_indices.npy"
embedding_scores_path = FINAL_ARTIFACT_DIR / "top5000_scores.npy"
if embedding_indices_path.exists() and embedding_scores_path.exists():
    final_embedding_indices = np.load(embedding_indices_path, mmap_mode="r")
    final_embedding_scores = np.load(embedding_scores_path, mmap_mode="r")
    assert final_embedding_indices.shape == (len(final_query_ids), FINAL_TOP_K)
    assert final_embedding_scores.shape == (len(final_query_ids), FINAL_TOP_K)
    print("E5 top-5000 benchmark загружен.")
else:
    final_embedding_indices = np.empty(
        (len(final_query_ids), FINAL_TOP_K), dtype=np.int32
    )
    final_embedding_scores = np.empty(
        (len(final_query_ids), FINAL_TOP_K), dtype=np.float32
    )
    for start in range(0, len(benchmark_query_embeddings), FAISS_SEARCH_BATCH_SIZE):
        stop = min(start + FAISS_SEARCH_BATCH_SIZE, len(benchmark_query_embeddings))
        scores, indices = final_embedding_index.search(
            np.ascontiguousarray(
                benchmark_query_embeddings[start:stop], dtype=np.float32
            ),
            FINAL_TOP_K,
        )
        final_embedding_indices[start:stop] = indices
        final_embedding_scores[start:stop] = scores
        print(f"E5 benchmark: {stop:,}/{len(benchmark_query_embeddings):,}")
    np.save(embedding_indices_path, final_embedding_indices)
    np.save(embedding_scores_path, final_embedding_scores)

del embedding_model, final_embedding_index
del benchmark_item_embeddings, benchmark_query_embeddings
del raw_items, raw_queries, embedding_item_texts, embedding_query_texts
gc.collect()
print(f"E5 сохранён в: {FINAL_ARTIFACT_DIR.resolve()}")


bm25_title_description: 128/2,452 запросов
bm25_title_description: 1,280/2,452 запросов
bm25_title_description: 2,452/2,452 запросов
bm25_title_description: 0.81 мин.
char_tfidf_title: 64/2,452 запросов
char_tfidf_title: 640/2,452 запросов
char_tfidf_title: 1,280/2,452 запросов
char_tfidf_title: 1,920/2,452 запросов
char_tfidf_title: 2,452/2,452 запросов
char_tfidf_title: 0.45 мин.
bm25_title: 128/2,452 запросов
bm25_title: 1,280/2,452 запросов
bm25_title: 2,452/2,452 запросов
bm25_title: 0.05 мин.
word_tfidf_title: 128/2,452 запросов
word_tfidf_title: 1,280/2,452 запросов
word_tfidf_title: 2,452/2,452 запросов
word_tfidf_title: 0.05 мин.
bm25_parameters: 32/127 запросов
bm25_parameters: 127/127 запросов


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/256 [00:00<?, ?it/s]

item_embeddings_maxlen128: 2,048/189,212


Batches:   0%|          | 0/256 [00:00<?, ?it/s]

...

item_embeddings_maxlen128: 189,212/189,212


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

query_embeddings_maxlen128: 2,048/2,452


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

query_embeddings_maxlen128: 2,452/2,452
FAISS benchmark: 5,000/189,212
FAISS benchmark: 10,000/189,212
FAISS benchmark: 15,000/189,212
FAISS benchmark: 20,000/189,212
FAISS benchmark: 25,000/189,212
FAISS benchmark: 30,000/189,212
FAISS benchmark: 35,000/189,212
FAISS benchmark: 40,000/189,212
FAISS benchmark: 45,000/189,212
FAISS benchmark: 50,000/189,212
FAISS benchmark: 55,000/189,212
FAISS benchmark: 60,000/189,212
FAISS benchmark: 65,000/189,212
FAISS benchmark: 70,000/189,212
FAISS benchmark: 75,000/189,212
FAISS benchmark: 80,000/189,212
FAISS benchmark: 85,000/189,212
FAISS benchmark: 90,000/189,212
FAISS benchmark: 95,000/189,212
FAISS benchmark: 100,000/189,212
FAISS benchmark: 105,000/189,212
FAISS benchmark: 110,000/189,212
FAISS benchmark: 115,000/189,212
FAISS benchmark: 120,000/189,212
FAISS benchmark: 125,000/189,212
FAISS benchmark: 130,000/189,212
FAISS benchmark: 135,000/189,212
FAISS benchmark: 140,000/189,212
FAISS benchmark: 145,000/189,212
FAISS benchmark: 150,00

In [62]:
# Финальная география, ранжирование и проверенный CSV.
# train используется только для статистики «регион → город»;
# возвращаемые item_id всегда берутся только из benchmark_items.

FINAL_STATIC_GEO_SCORES = STATIC_GEO_SCORES.copy()
FINAL_GEO_DISTANCE_LIMITS_KM = GEO_DISTANCE_LIMITS_KM.copy()

geo_coordinates = pd.concat(
    [
        train_items[["item_location_id", "item_latitude", "item_longitude"]],
        benchmark_items[["item_location_id", "item_latitude", "item_longitude"]],
    ],
    ignore_index=True,
).astype({"item_latitude": "float64", "item_longitude": "float64"})
geo_centers = (
    geo_coordinates.dropna(
        subset=["item_location_id", "item_latitude", "item_longitude"]
    )
    .groupby("item_location_id")[["item_latitude", "item_longitude"]]
    .median()
    .sort_index()
)
geo_city_ids = pd.Index(geo_centers.index)
geo_latitude = np.deg2rad(geo_centers["item_latitude"].to_numpy())
geo_longitude = np.deg2rad(geo_centers["item_longitude"].to_numpy())
geo_vectors = np.column_stack(
    [
        np.cos(geo_latitude) * np.cos(geo_longitude),
        np.cos(geo_latitude) * np.sin(geo_longitude),
        np.sin(geo_latitude),
    ]
).astype(np.float32)
geo_distances_km = (
    np.arccos(np.clip(geo_vectors @ geo_vectors.T, -1.0, 1.0))
    * 6_371.0088
).astype(np.float32)
np.fill_diagonal(geo_distances_km, 0.0)
del geo_coordinates, geo_vectors


def final_geo_distance_bins(query_codes, item_codes):
    bins = np.full(len(item_codes), 3, dtype=np.int8)
    known = (query_codes >= 0) & (item_codes >= 0)
    if np.any(known):
        distances = geo_distances_km[query_codes[known], item_codes[known]]
        bins[known] = np.select(
            [
                query_codes[known] == item_codes[known],
                distances <= FINAL_GEO_DISTANCE_LIMITS_KM[0],
                distances <= FINAL_GEO_DISTANCE_LIMITS_KM[1],
            ],
            [0, 1, 2],
            default=3,
        ).astype(np.int8)
    return bins


final_item_city_codes = geo_city_ids.get_indexer(final_item_locations)
final_query_city_codes = geo_city_ids.get_indexer(final_query_locations)

geo_train_pairs = (
    relevance
    .merge(
        train_queries[["train_query_id", "search_location_id"]],
        on="train_query_id",
        validate="many_to_one",
    )
    .merge(
        train_items[["item_id", "item_location_id"]],
        on="item_id",
        validate="many_to_one",
    )
)
train_query_city_codes = geo_city_ids.get_indexer(
    geo_train_pairs["search_location_id"]
)
train_item_city_codes = geo_city_ids.get_indexer(
    geo_train_pairs["item_location_id"]
)
region_mask = (train_query_city_codes < 0) & (train_item_city_codes >= 0)
region_ids = pd.Index(
    pd.unique(
        geo_train_pairs.loc[region_mask, "search_location_id"].dropna()
    )
)
train_region_codes = region_ids.get_indexer(
    geo_train_pairs.loc[region_mask, "search_location_id"]
)
region_city_counts = np.zeros(
    (len(region_ids), len(geo_city_ids)), dtype=np.int32
)
known_regions = train_region_codes >= 0
np.add.at(
    region_city_counts,
    (
        train_region_codes[known_regions],
        train_item_city_codes[region_mask][known_regions],
    ),
    1,
)
region_scores = np.zeros_like(region_city_counts, dtype=np.float32)
for region_code, counts in enumerate(region_city_counts):
    if counts.max():
        region_scores[region_code] = np.log1p(counts) / np.log1p(counts.max())
final_query_region_codes = region_ids.get_indexer(final_query_locations)


def final_geo_scores(row, candidates):
    item_codes = final_item_city_codes[candidates]
    query_code = final_query_city_codes[row]
    scores = np.zeros(len(candidates), dtype=np.float32)
    if query_code >= 0:
        scores = FINAL_STATIC_GEO_SCORES[final_geo_distance_bins(
            np.full(len(candidates), query_code, dtype=np.int32), item_codes
        )]
        scores[
            final_item_locations[candidates] == final_query_locations[row]
        ] = 1.0
        return scores

    region_code = final_query_region_codes[row]
    known = item_codes >= 0
    if region_code >= 0 and np.any(known):
        scores[known] = region_scores[region_code, item_codes[known]]
    return scores


def final_candidates_and_scores(row):
    text_candidates, text_scores = source_candidates_and_scores(
        row,
        BASE_HYBRID_WEIGHTS,
        final_source_indices,
        final_source_scores,
        final_source_names,
    )
    parts = [text_candidates, final_embedding_indices[row]]
    filter_code = final_filter_codes[row]
    if filter_code >= 0:
        parameter_candidates = final_parameter_indices[filter_code]
        parts.append(parameter_candidates[parameter_candidates >= 0])
    candidates = np.unique(np.concatenate(parts))

    scores = np.zeros(len(candidates), dtype=np.float32)
    scores[np.searchsorted(candidates, text_candidates)] = text_scores
    embedding_scores = final_embedding_scores[row]
    if embedding_scores[0] > 0:
        scores[np.searchsorted(candidates, final_embedding_indices[row])] += (
            FINAL_COSINE_COEFFICIENT * embedding_scores / embedding_scores[0]
        )
    return candidates, scores + final_geo_scores(row, candidates)


item_filter_matrix, query_filter_matrix, filter_token_counts = (
    build_filter_token_matrices(
        benchmark_items["item_infm_params_text"], final_unique_filters
    )
)
required_matches = np.ceil(
    FINAL_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
final_top50_indices = np.full(
    (len(final_query_ids), TOP_50), -1, dtype=np.int32
)

for row in np.flatnonzero(final_filter_codes < 0):
    candidates, scores = final_candidates_and_scores(row)
    final_top50_indices[row] = candidates[top_positions(scores, TOP_50)]

candidate_mark = np.zeros(len(benchmark_items), dtype=np.int32)
for position, filter_code in enumerate(range(len(final_unique_filters)), start=1):
    rows = np.flatnonzero(final_filter_codes == filter_code)
    for row in rows:
        candidates, _ = final_candidates_and_scores(row)
        candidate_mark[candidates] = filter_code + 1

    group_candidates = np.flatnonzero(candidate_mark == filter_code + 1)
    if required_matches[filter_code] and len(group_candidates):
        overlap = (
            query_filter_matrix[filter_code]
            @ item_filter_matrix[group_candidates].T
        ).tocsr()
        matching_items = group_candidates[
            overlap.indices[overlap.data >= required_matches[filter_code]]
        ]
    else:
        matching_items = np.empty(0, dtype=np.int32)

    for row in rows:
        candidates, scores = final_candidates_and_scores(row)
        filter_match = np.isin(candidates, matching_items, assume_unique=True)
        final_top50_indices[row] = candidates[top_positions(
            scores + FINAL_FILTER_BONUS * filter_match, TOP_50
        )]
    if position % 100 == 0 or position == len(final_unique_filters):
        print(f"Финальный benchmark: {position:,}/{len(final_unique_filters):,}")

FINAL_ANSWER_PATH = Path("answer_geo_e5_mixed.csv")
final_answer = pd.DataFrame(
    {
        "query_id": final_query_ids,
        "answer": [
            " ".join(final_item_ids[indices[indices >= 0]])
            for indices in final_top50_indices
        ],
    }
)
final_answer.to_csv(FINAL_ANSWER_PATH, index=False, encoding="utf-8")

answer_check = pd.read_csv(
    FINAL_ANSWER_PATH,
    dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
answer_items = answer_check["answer"].map(str.split)
assert answer_check.columns.tolist() == ["query_id", "answer"]
assert len(answer_check) == len(final_query_ids)
assert answer_check["query_id"].is_unique
assert set(answer_check["query_id"]) == set(final_query_ids)
assert answer_items.map(len).le(TOP_50).all()
assert answer_items.map(lambda values: len(values) == len(set(values))).all()
assert answer_items.map(
    lambda values: set(values).issubset(set(final_item_ids))
).all()

display(pd.DataFrame([{
    "file": str(FINAL_ANSWER_PATH),
    "rows": len(answer_check),
    "min_item_ids": answer_items.map(len).min(),
    "max_item_ids": answer_items.map(len).max(),
    "all_checks_passed": True,
}]))
print(f"Создан файл: {FINAL_ANSWER_PATH.resolve()}")

del candidate_mark, item_filter_matrix, query_filter_matrix
gc.collect()


Финальный benchmark: 100/127
Финальный benchmark: 127/127


,file,rows,min_item_ids,max_item_ids,all_checks_passed
0,answer_geo_e5_mixed.csv,2452,50,50,True


Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_geo_e5_mixed.csv


0

## 24. Уточнение географических коэффициентов

После benchmark-расчёта снова открываем validation-массивы и проверяем городской и региональный коэффициенты по отдельности.

In [64]:
# Восстанавливаем validation top-5000 после benchmark-расчёта.
# mmap открывает сохранённые массивы с диска без полной загрузки в RAM.

extended_top_indices, extended_top_scores = {}, {}
for name in SOURCE_NAMES:
    indices_path, scores_path, _ = extended_top_k_paths(name, EXTENDED_TOP_K)
    extended_top_indices[name] = np.load(indices_path, mmap_mode="r")
    extended_top_scores[name] = np.load(scores_path, mmap_mode="r")

embedding_top_indices = np.load(
    SAVE_DIR / f"{EMBEDDING_SOURCE_NAME}_top{EMBEDDING_TOP_K}_indices.npy",
    mmap_mode="r",
)
embedding_top_scores = np.load(
    SAVE_DIR / f"{EMBEDDING_SOURCE_NAME}_top{EMBEDDING_TOP_K}_scores.npy",
    mmap_mode="r",
)
print("Validation top-5000 восстановлены из сохранённых файлов.")

Validation top-5000 восстановлены из сохранённых файлов.


In [65]:
# Грубый подбор отдельных коэффициентов географии на уже готовом validation-пуле.
# Новые текстовые кандидаты, E5 и BM25 здесь не рассчитываются.
# filter_match фиксирован на лучшем текущем значении 0.40.

CITY_GEO_COEFFICIENTS = np.array(
    [0.00, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00],
    dtype=np.float32,
)
REGION_GEO_COEFFICIENTS = np.array(
    [0.00, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00],
    dtype=np.float32,
)
FIXED_FILTER_BONUS = 0.40
FIXED_COSINE_COEFFICIENT = 0.05

city_geo_recall_sum = np.zeros(len(CITY_GEO_COEFFICIENTS), dtype=np.float64)
region_geo_recall_sum = np.zeros(
    len(REGION_GEO_COEFFICIENTS), dtype=np.float64
)


def add_geo_grid_result(row, candidates, scores, filter_match):
    # scores уже содержат старый бонус same_location; заменяем его
    # тестируемым гео-скором, не меняя остальные части формулы.
    direct_score = location_bonus(
        candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    fixed_scores = (
        scores
        - direct_score
        + FIXED_FILTER_BONUS * filter_match
        + FIXED_COSINE_COEFFICIENT
        * cosine_values_in_embedding_pool(row, candidates)
    )

    if geo_valid_is_city[row]:
        coefficients = CITY_GEO_COEFFICIENTS
        geo_score = geo_scores_for_candidates(
            row, candidates, "static_distance"
        )
        recall_sum = city_geo_recall_sum
    else:
        coefficients = REGION_GEO_COEFFICIENTS
        geo_score = geo_scores_for_candidates(row, candidates, "learned_geo")
        recall_sum = region_geo_recall_sum

    score_grid = fixed_scores[None, :] + coefficients[:, None] * geo_score
    top50_positions = np.argpartition(score_grid, -TOP_50, axis=1)[:, -TOP_50:]
    for position, top50 in enumerate(top50_positions):
        recall_sum[position] += recall_for_item_indices(
            row, candidates[top50]
        )


# У запросов без фильтров filter_match всегда равен нулю.
for row in np.flatnonzero(parameter_filter_codes < 0):
    candidates, scores = candidates_with_embedding_pool(row)
    add_geo_grid_result(
        row,
        candidates,
        scores,
        np.zeros(len(candidates), dtype=np.float32),
    )

# Для запросов с фильтрами повторно используем ту же проверку 90 % токенов.
item_matrix, filter_matrix, filter_token_counts = build_filter_token_matrices(
    train_items["item_infm_params_text"],
    unique_parameter_filters,
)
required_matches = np.ceil(
    EMBEDDING_FORMULA_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)
candidate_mark = np.zeros(len(train_items), dtype=np.int32)

for position, filter_code in enumerate(range(len(unique_parameter_filters)), start=1):
    rows = np.flatnonzero(parameter_filter_codes == filter_code)
    for row in rows:
        candidates, _ = candidates_with_embedding_pool(row)
        candidate_mark[candidates] = filter_code + 1

    group_candidates = np.flatnonzero(candidate_mark == filter_code + 1)
    overlap = (
        filter_matrix[filter_code] @ item_matrix[group_candidates].T
    ).tocsr()
    matching_items = group_candidates[
        overlap.indices[overlap.data >= required_matches[filter_code]]
    ]

    for row in rows:
        candidates, scores = candidates_with_embedding_pool(row)
        filter_match = np.isin(
            candidates, matching_items, assume_unique=True
        ).astype(np.float32)
        add_geo_grid_result(row, candidates, scores, filter_match)

    if position % 100 == 0 or position == len(unique_parameter_filters):
        print(
            f"Гео-коэффициенты: {position:,}/"
            f"{len(unique_parameter_filters):,} фильтров"
        )

city_geo_grid = pd.DataFrame(
    {
        "city_geo_coefficient": CITY_GEO_COEFFICIENTS,
        "city_recall_at_50": city_geo_recall_sum / geo_city_query_count,
    }
)
region_geo_grid = pd.DataFrame(
    {
        "region_geo_coefficient": REGION_GEO_COEFFICIENTS,
        "region_recall_at_50": region_geo_recall_sum / geo_region_query_count,
    }
)
geo_coefficient_grid = (
    city_geo_grid.merge(region_geo_grid, how="cross")
    .assign(
        recall_at_50=lambda frame: (
            city_geo_recall_sum[
                np.searchsorted(
                    CITY_GEO_COEFFICIENTS,
                    frame["city_geo_coefficient"].to_numpy(),
                )
            ]
            + region_geo_recall_sum[
                np.searchsorted(
                    REGION_GEO_COEFFICIENTS,
                    frame["region_geo_coefficient"].to_numpy(),
                )
            ]
        ) / len(saved_query_ids)
    )
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)

display(geo_coefficient_grid.head(15))
display(city_geo_grid.sort_values("city_recall_at_50", ascending=False))
display(region_geo_grid.sort_values("region_recall_at_50", ascending=False))

del candidate_mark, item_matrix, filter_matrix
gc.collect()


Гео-коэффициенты: 100/1,196 фильтров
Гео-коэффициенты: 200/1,196 фильтров
Гео-коэффициенты: 300/1,196 фильтров
Гео-коэффициенты: 400/1,196 фильтров
Гео-коэффициенты: 500/1,196 фильтров
Гео-коэффициенты: 600/1,196 фильтров
Гео-коэффициенты: 700/1,196 фильтров
Гео-коэффициенты: 800/1,196 фильтров
Гео-коэффициенты: 900/1,196 фильтров
Гео-коэффициенты: 1,000/1,196 фильтров
Гео-коэффициенты: 1,100/1,196 фильтров
Гео-коэффициенты: 1,196/1,196 фильтров


,city_geo_coefficient,city_recall_at_50,region_geo_coefficient,region_recall_at_50,recall_at_50
0,2.0000,0.8642,0.7500,0.6812,0.8416
1,1.5000,0.8639,0.7500,0.6812,0.8414
2,1.2500,0.8625,0.7500,0.6812,0.8402
3,2.0000,0.8642,1.0000,0.6685,0.8400
4,1.5000,0.8639,1.0000,0.6685,0.8398
5,1.2500,0.8625,1.0000,0.6685,0.8386
6,2.0000,0.8642,1.2500,0.6495,0.8377
7,2.0000,0.8642,0.5000,0.6479,0.8375
8,1.5000,0.8639,1.2500,0.6495,0.8375
9,1.5000,0.8639,0.5000,0.6479,0.8373


,city_geo_coefficient,city_recall_at_50
7,2.0000,0.8642
6,1.5000,0.8639
5,1.2500,0.8625
4,1.0000,0.8589
3,0.7500,0.8277
2,0.5000,0.7314
1,0.2500,0.5317
0,0.0000,0.1989


,region_geo_coefficient,region_recall_at_50
3,0.7500,0.6812
4,1.0000,0.6685
5,1.2500,0.6495
2,0.5000,0.6479
6,1.5000,0.6342
7,2.0000,0.6104
1,0.2500,0.5439
0,0.0000,0.3104


1831

## 25. Локальные источники кандидатов

Измеряем прирост полноты от BM25 в географическом подкорпусе и среди объявлений с подходящими параметрами.

In [74]:
# Проверка двух локальных BM25-источников на validation.
# Универсальные функции отбора и локального поиска находятся выше,
# в ячейке retrieval-helpers.
#
# BM25 ранжирует максимум top-3000 один раз. Затем оцениваем его префиксы
# top-100, 500, 1000 и 3000 — без повторных поисковых расчётов.
# В общий пул эта ячейка ничего не добавляет.

from scipy.sparse import load_npz

LOCAL_TOP_K = 3_000
LOCAL_CANDIDATE_DEPTHS = np.array(
    [100, 500, 1_000, 3_000],
    dtype=np.int32,
)
LOCAL_TEXT_SOURCE = "bm25_title_description"
LOCAL_FILTER_THRESHOLD = 0.90
LOCAL_CITY_RADIUS_KM = 75.0

local_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_query"]
    .fillna("")
)
local_vectorizer = joblib.load(
    SAVE_DIR / f"{LOCAL_TEXT_SOURCE}_vectorizer.joblib"
)
local_item_matrix = load_npz(
    SAVE_DIR / f"{LOCAL_TEXT_SOURCE}_item_matrix.npz"
).tocsr()
local_query_matrix = local_vectorizer.transform(local_query_text).astype(
    np.float32
)
local_query_matrix.data.fill(1.0)

filter_item_matrix, filter_query_matrix, filter_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"],
        unique_parameter_filters,
    )
)
required_matches = np.ceil(
    LOCAL_FILTER_THRESHOLD * filter_token_counts
).astype(np.int32)


def geo_candidate_groups():
    """Город: радиус 75 км. Регион: города с train-связью с регионом."""
    for location_id, rows in query_row_groups(valid_query_locations):
        city_code = validation_geo_city_ids.get_indexer([location_id])[0]
        if city_code >= 0:
            item_indices = item_indices_within_radius(
                location_id,
                LOCAL_CITY_RADIUS_KM,
                item_locations,
                validation_geo_item_city_codes,
                validation_geo_city_ids,
                validation_geo_city_distances_km,
            )
        else:
            region_code = validation_geo_region_codes[rows[0]]
            item_indices = item_indices_from_region(
                region_code,
                validation_region_city_scores,
                validation_geo_item_city_codes,
            )
        if len(item_indices):
            yield rows, item_indices


def filter_candidate_groups():
    for filter_code, rows in query_row_groups(parameter_filter_codes):
        if filter_code >= 0:
            item_indices = item_indices_matching_filter(
                filter_code,
                filter_item_matrix,
                filter_query_matrix,
                required_matches,
            )
            if len(item_indices):
                yield rows, item_indices


relevant_items_by_row = [
    np.fromiter(relevant_indices_by_query[query_id], dtype=np.int32)
    for query_id in saved_query_ids
]
base_found_by_row = []
for row, relevant in enumerate(relevant_items_by_row):
    base_candidates, _ = candidates_with_embedding_pool(row)
    base_found_by_row.append(np.isin(relevant, base_candidates))
    if (row + 1) % 10_000 == 0 or row + 1 == len(saved_query_ids):
        print(f"Текущий пул: {row + 1:,}/{len(saved_query_ids):,}")
base_candidate_recall = np.mean([
    found.mean() for found in base_found_by_row
])


def evaluate_local_bm25_source(source_name, groups):
    """Измеряет прирост полноты для нескольких top-K одного источника."""
    source_recall_sum = np.zeros(
        len(LOCAL_CANDIDATE_DEPTHS), dtype=np.float64
    )
    added_recall_sum = np.zeros_like(source_recall_sum)
    new_relevant_items = np.zeros_like(source_recall_sum, dtype=np.int64)
    queries_with_new_items = np.zeros_like(
        source_recall_sum, dtype=np.int32
    )

    for group_number, (rows, item_indices) in enumerate(groups, start=1):
        for row, source_candidates, _ in iter_top_k_in_item_subset(
            local_query_matrix,
            local_item_matrix,
            rows,
            item_indices,
            top_k=LOCAL_TOP_K,
        ):
            for depth_position, top_k in enumerate(LOCAL_CANDIDATE_DEPTHS):
                source_found = np.isin(
                    relevant_items_by_row[row],
                    source_candidates[:top_k],
                )
                added = source_found & ~base_found_by_row[row]
                source_recall_sum[depth_position] += source_found.mean()
                added_recall_sum[depth_position] += added.mean()
                new_relevant_items[depth_position] += int(added.sum())
                queries_with_new_items[depth_position] += int(added.any())

        if group_number % 100 == 0:
            print(f"{source_name}: {group_number:,} подкорпусов")

    return pd.DataFrame(
        {
            "source": source_name,
            "top_k": LOCAL_CANDIDATE_DEPTHS,
            "current_pool_recall": base_candidate_recall,
            "source_candidate_recall": (
                source_recall_sum / len(saved_query_ids)
            ),
            "union_candidate_recall": (
                base_candidate_recall
                + added_recall_sum / len(saved_query_ids)
            ),
            "added_candidate_recall": (
                added_recall_sum / len(saved_query_ids)
            ),
            "new_relevant_items": new_relevant_items,
            "queries_with_new_relevant_items": queries_with_new_items,
        }
    )


geo_source_results = evaluate_local_bm25_source(
    "local_bm25_75km_or_region",
    geo_candidate_groups(),
)
filter_source_results = evaluate_local_bm25_source(
    "local_bm25_matching_filter",
    filter_candidate_groups(),
)
local_source_results = pd.concat(
    [geo_source_results, filter_source_results],
    ignore_index=True,
)
display(local_source_results)

# Ячейка только измеряет полноту источников; объединение кандидатов
# и подбор глубины происходят в следующих экспериментах.

del local_vectorizer, local_item_matrix, local_query_matrix
del filter_item_matrix, filter_query_matrix
gc.collect()


Текущий пул: 10,000/67,186
Текущий пул: 20,000/67,186
Текущий пул: 30,000/67,186
Текущий пул: 40,000/67,186
Текущий пул: 50,000/67,186
Текущий пул: 60,000/67,186
Текущий пул: 67,186/67,186
local_bm25_75km_or_region: 100 подкорпусов
local_bm25_75km_or_region: 200 подкорпусов
local_bm25_75km_or_region: 300 подкорпусов
local_bm25_75km_or_region: 400 подкорпусов
local_bm25_75km_or_region: 500 подкорпусов
local_bm25_75km_or_region: 600 подкорпусов
local_bm25_75km_or_region: 700 подкорпусов
local_bm25_75km_or_region: 800 подкорпусов
local_bm25_75km_or_region: 900 подкорпусов
local_bm25_75km_or_region: 1,000 подкорпусов
local_bm25_75km_or_region: 1,100 подкорпусов
local_bm25_75km_or_region: 1,200 подкорпусов
local_bm25_75km_or_region: 1,300 подкорпусов
local_bm25_75km_or_region: 1,400 подкорпусов
local_bm25_75km_or_region: 1,500 подкорпусов
local_bm25_75km_or_region: 1,600 подкорпусов
local_bm25_matching_filter: 100 подкорпусов
local_bm25_matching_filter: 200 подкорпусов
local_bm25_matching_f

,source,top_k,current_pool_recall,source_candidate_recall,union_candidate_recall,added_candidate_recall,new_relevant_items,queries_with_new_relevant_items
0,local_bm25_75km_or_region,100,0.9482,0.7318,0.9546,0.0064,647,595
1,local_bm25_75km_or_region,500,0.9482,0.8159,0.9611,0.0129,1327,1153
2,local_bm25_75km_or_region,1000,0.9482,0.8349,0.9633,0.0151,1563,1341
3,local_bm25_75km_or_region,3000,0.9482,0.8509,0.9658,0.0176,1869,1568
4,local_bm25_matching_filter,100,0.9482,0.1464,0.9482,0.0000,0,0
5,local_bm25_matching_filter,500,0.9482,0.3062,0.9483,0.0001,5,5
6,local_bm25_matching_filter,1000,0.9482,0.3815,0.9485,0.0003,18,18
7,local_bm25_matching_filter,3000,0.9482,0.4639,0.9497,0.0015,150,138


3717

### 25.1. Смешивание локального и глобального поиска

In [75]:
# Грубый подбор смешивания текущей формулы и локального BM25.
# Гео top-5000 добавляется к имеющемуся пулу, а не заменяет его.

from scipy.sparse import load_npz

LOCAL_MIX_TOP_K = 5_000
LOCAL_MIX_DEPTHS = np.array([3_000, 5_000], dtype=np.int32)
LOCAL_MIX_WEIGHTS = np.array(
    [0.00, 0.05, 0.10, 0.20, 0.30, 0.40, 0.60, 0.80, 0.90],
    dtype=np.float32,
)
LOCAL_MIX_CITY_GEO = 2.00
LOCAL_MIX_REGION_GEO = 0.75
LOCAL_MIX_FILTER_BONUS = 0.40
LOCAL_MIX_COSINE = 0.05

needed = (
    "validation_geo_city_ids",
    "validation_geo_item_city_codes",
    "validation_geo_city_distances_km",
    "validation_geo_region_codes",
    "validation_region_city_scores",
    "geo_candidate_groups",
)
missing = [name for name in needed if name not in globals()]
assert not missing, (
    "Запустите retrieval-helpers, гео-ячейку и "
    "local-pool-expansion-check. Нет: " + ", ".join(missing)
)

local_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_query"]
    .fillna("")
)
local_vectorizer = joblib.load(
    SAVE_DIR / "bm25_title_description_vectorizer.joblib"
)
local_item_matrix = load_npz(
    SAVE_DIR / "bm25_title_description_item_matrix.npz"
).tocsr()
local_query_matrix = local_vectorizer.transform(local_query_text).astype(
    np.float32
)
local_query_matrix.data.fill(1.0)

# Фильтр остаётся частью текущей формулы, но не отдельным источником.
filter_item_matrix, filter_query_matrix, filter_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"],
        unique_parameter_filters,
    )
)
required_matches = np.ceil(0.90 * filter_token_counts).astype(np.int32)
filter_items = {
    code: item_indices_matching_filter(
        code,
        filter_item_matrix,
        filter_query_matrix,
        required_matches,
    )
    for code in range(len(unique_parameter_filters))
}
del filter_item_matrix, filter_query_matrix


def local_geo_score(row, candidates):
    item_codes = validation_geo_item_city_codes[candidates]
    query_code = validation_geo_city_ids.get_indexer(
        [valid_query_locations[row]]
    )[0]
    score = np.zeros(len(candidates), dtype=np.float32)

    if query_code >= 0:
        known = item_codes >= 0
        bins = np.full(len(candidates), 3, dtype=np.int8)
        distances = validation_geo_city_distances_km[
            query_code, item_codes[known]
        ]
        bins[known] = np.select(
            [
                item_codes[known] == query_code,
                distances <= 25.0,
                distances <= 75.0,
            ],
            [0, 1, 2],
            default=3,
        ).astype(np.int8)
        return (
            LOCAL_MIX_CITY_GEO
            * np.array([1.00, 0.55, 0.25, 0.00])[bins]
        ).astype(np.float32)

    region_code = validation_geo_region_codes[row]
    known = item_codes >= 0
    if 0 <= region_code < len(validation_region_city_scores):
        score[known] = (
            LOCAL_MIX_REGION_GEO
            * validation_region_city_scores[region_code, item_codes[known]]
        )
    return score


def current_score(row, candidates, base_candidates, base_scores):
    """Текущая формула на расширенном пуле, нормированная по запросу."""
    score = np.zeros(len(candidates), dtype=np.float32)
    base_positions = np.searchsorted(candidates, base_candidates)
    direct_geo = location_bonus(
        base_candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    score[base_positions] = base_scores - direct_geo
    score += local_geo_score(row, candidates)
    score += LOCAL_MIX_COSINE * cosine_values_in_embedding_pool(
        row, candidates
    )

    filter_code = parameter_filter_codes[row]
    if filter_code >= 0:
        score += LOCAL_MIX_FILTER_BONUS * np.isin(
            candidates,
            filter_items[filter_code],
            assume_unique=True,
        )

    maximum = score.max()
    return score / maximum if maximum > 0 else score


# Базовый фактический Recall@50: понадобится для запросов без локального
# подкорпуса и как точка отсчёта в итоговой таблице.
baseline_recall = np.empty(len(saved_query_ids), dtype=np.float32)
for row in range(len(saved_query_ids)):
    base_candidates, base_scores = candidates_with_embedding_pool(row)
    score = current_score(
        row, base_candidates, base_candidates, base_scores
    )
    baseline_recall[row] = recall_for_item_indices(
        row, base_candidates[top_positions(score, TOP_50)]
    )
    if (row + 1) % 10_000 == 0 or row + 1 == len(saved_query_ids):
        print(f"Базовая формула: {row + 1:,}/{len(saved_query_ids):,}")

recall_sums = np.full(
    (len(LOCAL_MIX_DEPTHS), len(LOCAL_MIX_WEIGHTS)),
    baseline_recall.sum(),
    dtype=np.float64,
)
local_query_count = 0

# geo_candidate_groups уже реализует: до 75 км для города либо
# связанные с локацией города для запроса без координатного центра.
for group_number, (rows, item_indices) in enumerate(
    geo_candidate_groups(), start=1
):
    for row, local_candidates, local_scores in iter_top_k_in_item_subset(
        local_query_matrix,
        local_item_matrix,
        rows,
        item_indices,
        top_k=LOCAL_MIX_TOP_K,
    ):
        if not len(local_scores):
            continue

        local_query_count += 1
        base_candidates, base_scores = candidates_with_embedding_pool(row)

        for depth_position, depth in enumerate(LOCAL_MIX_DEPTHS):
            selected_candidates = local_candidates[:depth]
            selected_scores = local_scores[:depth]
            candidates = np.union1d(base_candidates, selected_candidates)
            score = current_score(
                row, candidates, base_candidates, base_scores
            )

            local_score = np.zeros(len(candidates), dtype=np.float32)
            local_positions = np.searchsorted(
                candidates, selected_candidates
            )
            local_score[local_positions] = (
                selected_scores / selected_scores[0]
            )

            # Порядок кандидатов без локального скора не меняется.
            # Поэтому достаточно их сильнейших 50; результат точный.
            has_local_score = local_score > 0
            nonlocal_positions = np.flatnonzero(~has_local_score)
            strongest_nonlocal = nonlocal_positions[
                top_positions(score[nonlocal_positions], TOP_50)
            ]
            keep = np.unique(np.concatenate([
                np.flatnonzero(has_local_score),
                strongest_nonlocal,
            ]))

            mixed_scores = (
                (1.0 - LOCAL_MIX_WEIGHTS[:, None])
                * score[keep][None, :]
                + LOCAL_MIX_WEIGHTS[:, None]
                * local_score[keep][None, :]
            )
            top50_grid = np.argpartition(
                mixed_scores, -TOP_50, axis=1
            )[:, -TOP_50:]

            recall_sums[depth_position] -= baseline_recall[row]
            for weight_position, positions in enumerate(top50_grid):
                recall_sums[depth_position, weight_position] += (
                    recall_for_item_indices(
                        row, candidates[keep[positions]]
                    )
                )

    if group_number % 100 == 0:
        print(f"Локальное смешивание: {group_number:,} подкорпусов")

local_geo_mixing_results = (
    pd.DataFrame(
        [
            {
                "local_top_k": int(depth),
                "local_weight": float(weight),
                "recall_at_50": (
                    recall_sums[depth_position, weight_position]
                    / len(saved_query_ids)
                ),
                "baseline_recall_at_50": float(baseline_recall.mean()),
            }
            for depth_position, depth in enumerate(LOCAL_MIX_DEPTHS)
            for weight_position, weight in enumerate(LOCAL_MIX_WEIGHTS)
        ]
    )
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(local_geo_mixing_results)
print(f"Запросов с ненулевым локальным BM25: {local_query_count:,}")

del local_vectorizer, local_item_matrix, local_query_matrix
del filter_items
gc.collect()



Базовая формула: 10,000/67,186
Базовая формула: 20,000/67,186
Базовая формула: 30,000/67,186
Базовая формула: 40,000/67,186
Базовая формула: 50,000/67,186
Базовая формула: 60,000/67,186
Базовая формула: 67,186/67,186
Локальное смешивание: 100 подкорпусов
Локальное смешивание: 200 подкорпусов
Локальное смешивание: 300 подкорпусов
Локальное смешивание: 400 подкорпусов
Локальное смешивание: 500 подкорпусов
Локальное смешивание: 600 подкорпусов
Локальное смешивание: 700 подкорпусов
Локальное смешивание: 800 подкорпусов
Локальное смешивание: 900 подкорпусов
Локальное смешивание: 1,000 подкорпусов
Локальное смешивание: 1,100 подкорпусов
Локальное смешивание: 1,200 подкорпусов
Локальное смешивание: 1,300 подкорпусов
Локальное смешивание: 1,400 подкорпусов
Локальное смешивание: 1,500 подкорпусов
Локальное смешивание: 1,600 подкорпусов


,local_top_k,local_weight,recall_at_50,baseline_recall_at_50
0,3000,0.0500,0.8463,0.8416
1,5000,0.0500,0.8461,0.8416
2,3000,0.0000,0.8455,0.8416
3,5000,0.0000,0.8453,0.8416
4,3000,0.1000,0.8451,0.8416
5,5000,0.1000,0.8449,0.8416
6,5000,0.2000,0.8402,0.8416
7,3000,0.2000,0.8402,0.8416
8,5000,0.3000,0.8335,0.8416
9,3000,0.3000,0.8333,0.8416


Запросов с ненулевым локальным BM25: 66,416


0

### 25.2. Подбор географии и фильтра после расширения пула

In [76]:
# Подбор городского, регионального и filter-коэффициентов
# на расширенном гео top-3000 при зафиксированном local_weight = 0.05.

from scipy.sparse import load_npz

TUNE_LOCAL_TOP_K = 3_000
TUNE_LOCAL_WEIGHT = 0.05
TUNE_CITY_COEFFICIENTS = np.array(
    [1.50, 2.00, 2.50, 3.00, 4.00], dtype=np.float32
)
TUNE_REGION_COEFFICIENTS = np.array(
    [0.50, 0.75, 1.00, 1.25], dtype=np.float32
)
TUNE_FILTER_BONUSES = np.array(
    [0.25, 0.35, 0.40, 0.45, 0.55], dtype=np.float32
)
TUNE_COSINE_COEFFICIENT = 0.05

needed = (
    "validation_geo_city_ids",
    "validation_geo_item_city_codes",
    "validation_geo_city_distances_km",
    "validation_geo_region_codes",
    "validation_region_city_scores",
    "geo_candidate_groups",
)
missing = [name for name in needed if name not in globals()]
assert not missing, (
    "Запустите retrieval-helpers, гео-ячейку и "
    "local-pool-expansion-check. Нет: " + ", ".join(missing)
)

tune_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_query"]
    .fillna("")
)
tune_vectorizer = joblib.load(
    SAVE_DIR / "bm25_title_description_vectorizer.joblib"
)
tune_item_matrix = load_npz(
    SAVE_DIR / "bm25_title_description_item_matrix.npz"
).tocsr()
tune_query_matrix = tune_vectorizer.transform(tune_query_text).astype(
    np.float32
)
tune_query_matrix.data.fill(1.0)

filter_item_matrix, filter_query_matrix, filter_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"],
        unique_parameter_filters,
    )
)
required_matches = np.ceil(0.90 * filter_token_counts).astype(np.int32)
tune_filter_items = {
    code: item_indices_matching_filter(
        code,
        filter_item_matrix,
        filter_query_matrix,
        required_matches,
    )
    for code in range(len(unique_parameter_filters))
}
del filter_item_matrix, filter_query_matrix


def tuning_geo_signal(row, candidates):
    """Ненормированный гео-сигнал и тип локации запроса."""
    item_codes = validation_geo_item_city_codes[candidates]
    query_code = validation_geo_city_ids.get_indexer(
        [valid_query_locations[row]]
    )[0]
    score = np.zeros(len(candidates), dtype=np.float32)

    if query_code >= 0:
        known = item_codes >= 0
        bins = np.full(len(candidates), 3, dtype=np.int8)
        distances = validation_geo_city_distances_km[
            query_code, item_codes[known]
        ]
        bins[known] = np.select(
            [
                item_codes[known] == query_code,
                distances <= 25.0,
                distances <= 75.0,
            ],
            [0, 1, 2],
            default=3,
        ).astype(np.int8)
        return "city", np.array([1.00, 0.55, 0.25, 0.00])[bins]

    region_code = validation_geo_region_codes[row]
    known = item_codes >= 0
    if 0 <= region_code < len(validation_region_city_scores):
        score[known] = validation_region_city_scores[
            region_code, item_codes[known]
        ]
    return "region", score


def tuning_recall_grid(
    row,
    candidates,
    base_candidates,
    base_scores,
    local_candidates=None,
    local_scores=None,
):
    """Recall@50 по сетке нужного сегмента: город или локация без центра."""
    base_signal = np.zeros(len(candidates), dtype=np.float32)
    base_positions = np.searchsorted(candidates, base_candidates)
    direct_geo = location_bonus(
        base_candidates,
        valid_query_locations[row],
        best_location_bonus,
        item_locations,
    )
    base_signal[base_positions] = base_scores - direct_geo
    base_signal += TUNE_COSINE_COEFFICIENT * cosine_values_in_embedding_pool(
        row, candidates
    )

    filter_signal = np.zeros(len(candidates), dtype=np.float32)
    filter_code = parameter_filter_codes[row]
    if filter_code >= 0:
        filter_signal = np.isin(
            candidates,
            tune_filter_items[filter_code],
            assume_unique=True,
        ).astype(np.float32)

    local_signal = np.zeros(len(candidates), dtype=np.float32)
    if local_candidates is not None and len(local_candidates):
        local_positions = np.searchsorted(candidates, local_candidates)
        local_signal[local_positions] = local_scores / local_scores[0]

    segment, geo_signal = tuning_geo_signal(row, candidates)
    geo_coefficients = (
        TUNE_CITY_COEFFICIENTS
        if segment == "city"
        else TUNE_REGION_COEFFICIENTS
    )

    # Кандидаты без local_score сохраняют порядок при фиксированной
    # паре geo/filter. Оставляем объединение их top-50 по всем парам.
    nonlocal_positions = np.flatnonzero(local_signal == 0)
    keep_nonlocal = []
    for geo_coefficient in geo_coefficients:
        for filter_bonus in TUNE_FILTER_BONUSES:
            raw_score = (
                base_signal[nonlocal_positions]
                + geo_coefficient * geo_signal[nonlocal_positions]
                + filter_bonus * filter_signal[nonlocal_positions]
            )
            keep_nonlocal.append(
                nonlocal_positions[top_positions(raw_score, TOP_50)]
            )

    keep = np.unique(np.concatenate([
        np.flatnonzero(local_signal > 0),
        *keep_nonlocal,
    ]))
    base_keep = base_signal[keep]
    geo_keep = geo_signal[keep]
    filter_keep = filter_signal[keep]
    local_keep = local_signal[keep]

    raw_grid = (
        base_keep[None, None, :]
        + geo_coefficients[:, None, None] * geo_keep[None, None, :]
        + TUNE_FILTER_BONUSES[None, :, None] * filter_keep[None, None, :]
    ).reshape(-1, len(keep))
    raw_maximum = raw_grid.max(axis=1, keepdims=True)
    normalized_current = np.divide(
        raw_grid,
        raw_maximum,
        out=np.zeros_like(raw_grid),
        where=raw_maximum > 0,
    )
    mixed_grid = (
        (1.0 - TUNE_LOCAL_WEIGHT) * normalized_current
        + TUNE_LOCAL_WEIGHT * local_keep[None, :]
    )
    top50_grid = np.argpartition(
        mixed_grid, -TOP_50, axis=1
    )[:, -TOP_50:]

    values = np.empty(len(top50_grid), dtype=np.float32)
    for position, top50_positions_ in enumerate(top50_grid):
        values[position] = recall_for_item_indices(
            row, candidates[keep[top50_positions_]]
        )
    return segment, values.reshape(
        len(geo_coefficients), len(TUNE_FILTER_BONUSES)
    )


n_queries = len(saved_query_ids)
city_recall_sums = np.zeros(
    (len(TUNE_CITY_COEFFICIENTS), len(TUNE_FILTER_BONUSES)),
    dtype=np.float64,
)
region_recall_sums = np.zeros(
    (len(TUNE_REGION_COEFFICIENTS), len(TUNE_FILTER_BONUSES)),
    dtype=np.float64,
)
city_baseline_by_query = np.zeros(
    (len(TUNE_CITY_COEFFICIENTS), len(TUNE_FILTER_BONUSES), n_queries),
    dtype=np.float32,
)
region_baseline_by_query = np.zeros(
    (len(TUNE_REGION_COEFFICIENTS), len(TUNE_FILTER_BONUSES), n_queries),
    dtype=np.float32,
)

# Считаем сетку на исходном пуле и сохраняем только значения Recall,
# чтобы при добавлении локальных кандидатов заменить вклад каждого запроса.
for row in range(n_queries):
    base_candidates, base_scores = candidates_with_embedding_pool(row)
    segment, values = tuning_recall_grid(
        row, base_candidates, base_candidates, base_scores
    )
    if segment == "city":
        city_recall_sums += values
        city_baseline_by_query[:, :, row] = values
    else:
        region_recall_sums += values
        region_baseline_by_query[:, :, row] = values

    if (row + 1) % 10_000 == 0 or row + 1 == n_queries:
        print(f"Базовая сетка: {row + 1:,}/{n_queries:,}")

local_query_count = 0
for group_number, (rows, item_indices) in enumerate(
    geo_candidate_groups(), start=1
):
    for row, local_candidates, local_scores in iter_top_k_in_item_subset(
        tune_query_matrix,
        tune_item_matrix,
        rows,
        item_indices,
        top_k=TUNE_LOCAL_TOP_K,
    ):
        if not len(local_scores):
            continue

        local_query_count += 1
        base_candidates, base_scores = candidates_with_embedding_pool(row)
        candidates = np.union1d(base_candidates, local_candidates)
        segment, values = tuning_recall_grid(
            row,
            candidates,
            base_candidates,
            base_scores,
            local_candidates,
            local_scores,
        )

        if segment == "city":
            city_recall_sums -= city_baseline_by_query[:, :, row]
            city_recall_sums += values
        else:
            region_recall_sums -= region_baseline_by_query[:, :, row]
            region_recall_sums += values

    if group_number % 100 == 0:
        print(f"Коэффициенты на локальном пуле: {group_number:,} подкорпусов")

coefficient_tuning_results = (
    pd.DataFrame(
        [
            {
                "city_geo_coefficient": float(city_coefficient),
                "region_geo_coefficient": float(region_coefficient),
                "filter_bonus": float(filter_bonus),
                "recall_at_50": (
                    city_recall_sums[city_position, filter_position]
                    + region_recall_sums[region_position, filter_position]
                ) / n_queries,
            }
            for city_position, city_coefficient in enumerate(
                TUNE_CITY_COEFFICIENTS
            )
            for region_position, region_coefficient in enumerate(
                TUNE_REGION_COEFFICIENTS
            )
            for filter_position, filter_bonus in enumerate(
                TUNE_FILTER_BONUSES
            )
        ]
    )
    .sort_values("recall_at_50", ascending=False)
    .reset_index(drop=True)
)
display(coefficient_tuning_results.head(20))
print(f"Запросов с локальным BM25: {local_query_count:,}")

del tune_vectorizer, tune_item_matrix, tune_query_matrix
del tune_filter_items, city_baseline_by_query, region_baseline_by_query
gc.collect()



Базовая сетка: 10,000/67,186
Базовая сетка: 20,000/67,186
Базовая сетка: 30,000/67,186
Базовая сетка: 40,000/67,186
Базовая сетка: 50,000/67,186
Базовая сетка: 60,000/67,186
Базовая сетка: 67,186/67,186
Коэффициенты на локальном пуле: 100 подкорпусов
Коэффициенты на локальном пуле: 200 подкорпусов
Коэффициенты на локальном пуле: 300 подкорпусов
Коэффициенты на локальном пуле: 400 подкорпусов
Коэффициенты на локальном пуле: 500 подкорпусов
Коэффициенты на локальном пуле: 600 подкорпусов
Коэффициенты на локальном пуле: 700 подкорпусов
Коэффициенты на локальном пуле: 800 подкорпусов
Коэффициенты на локальном пуле: 900 подкорпусов
Коэффициенты на локальном пуле: 1,000 подкорпусов
Коэффициенты на локальном пуле: 1,100 подкорпусов
Коэффициенты на локальном пуле: 1,200 подкорпусов
Коэффициенты на локальном пуле: 1,300 подкорпусов
Коэффициенты на локальном пуле: 1,400 подкорпусов
Коэффициенты на локальном пуле: 1,500 подкорпусов
Коэффициенты на локальном пуле: 1,600 подкорпусов


,city_geo_coefficient,region_geo_coefficient,filter_bonus,recall_at_50
0,1.5000,0.7500,0.5500,0.8480
1,1.5000,0.7500,0.4500,0.8474
2,2.0000,0.7500,0.5500,0.8473
3,1.5000,0.7500,0.4000,0.8470
4,2.0000,0.7500,0.4500,0.8469
5,1.5000,1.0000,0.5500,0.8468
6,1.5000,0.7500,0.3500,0.8466
7,2.0000,0.7500,0.4000,0.8464
8,2.5000,0.7500,0.5500,0.8462
9,2.0000,1.0000,0.5500,0.8461


Запросов с локальным BM25: 66,416


0

### 25.3. Benchmark-ответ с локальными кандидатами

In [77]:
# Финальный benchmark: расширенный гео-пул + подобранные коэффициенты.
# Предыдущие CSV не перезаписываются.

FINAL_LOCAL_TOP_K = 3_000
FINAL_LOCAL_WEIGHT = 0.05
FINAL_CITY_GEO_COEFFICIENT = 1.50
FINAL_REGION_GEO_COEFFICIENT = 0.75
FINAL_FILTER_BONUS = 0.55
FINAL_COSINE_COEFFICIENT = 0.05
FINAL_LOCAL_RADIUS_KM = 75.0
FINAL_LOCAL_ANSWER_PATH = Path("answer_geo_local_mixed.csv")
FINAL_EMBEDDING_DIR = Path("artifacts/retrieval/benchmark_mixed_geo_e5")

needed = (
    "final_source_indices",
    "final_source_scores",
    "final_source_names",
    "final_parameter_indices",
)
missing = [name for name in needed if name not in globals()]
assert not missing, (
    "Сначала запустите benchmark-final-sources-e5. Нет: "
    + ", ".join(missing)
)

final_query_ids = benchmark_queries["query_id"].astype(str).to_numpy()
final_item_ids = benchmark_items["item_id"].astype(str).to_numpy()
final_query_locations = benchmark_queries[
    "search_location_id"
].to_numpy()
final_item_locations = benchmark_items[
    "item_location_id"
].to_numpy()
final_query_text = benchmark_queries["search_query"].fillna("")

if (
    "final_embedding_indices" not in globals()
    or "final_embedding_scores" not in globals()
):
    final_embedding_indices = np.load(
        FINAL_EMBEDDING_DIR / "top5000_indices.npy",
        mmap_mode="r",
    )
    final_embedding_scores = np.load(
        FINAL_EMBEDDING_DIR / "top5000_scores.npy",
        mmap_mode="r",
    )

assert final_embedding_indices.shape == (
    len(final_query_ids), FINAL_TOP_K
)

# Геометрия benchmark строится заново по train + benchmark_items.
# Поэтому новые benchmark-локации с координатами тоже участвуют в радиусе.
final_local_coordinates = pd.concat(
    [
        train_items[
            ["item_location_id", "item_latitude", "item_longitude"]
        ],
        benchmark_items[
            ["item_location_id", "item_latitude", "item_longitude"]
        ],
    ],
    ignore_index=True,
).astype({"item_latitude": "float64", "item_longitude": "float64"})
final_local_centers = (
    final_local_coordinates.dropna(
        subset=["item_location_id", "item_latitude", "item_longitude"]
    )
    .groupby("item_location_id")[["item_latitude", "item_longitude"]]
    .median()
    .sort_index()
)
final_local_city_ids = pd.Index(final_local_centers.index)
final_local_latitude = np.deg2rad(
    final_local_centers["item_latitude"].to_numpy()
)
final_local_longitude = np.deg2rad(
    final_local_centers["item_longitude"].to_numpy()
)
final_local_vectors = np.column_stack(
    [
        np.cos(final_local_latitude) * np.cos(final_local_longitude),
        np.cos(final_local_latitude) * np.sin(final_local_longitude),
        np.sin(final_local_latitude),
    ]
).astype(np.float32)
final_local_distances = (
    np.arccos(
        np.clip(
            final_local_vectors @ final_local_vectors.T, -1.0, 1.0
        )
    )
    * 6_371.0088
).astype(np.float32)
np.fill_diagonal(final_local_distances, 0.0)
final_local_item_city_codes = final_local_city_ids.get_indexer(
    final_item_locations
)

# Для локаций без координатного центра используем train-связи с городами.
final_local_train_pairs = (
    relevance
    .merge(
        train_queries[["train_query_id", "search_location_id"]],
        on="train_query_id",
        validate="many_to_one",
    )
    .merge(
        train_items[["item_id", "item_location_id"]],
        on="item_id",
        validate="many_to_one",
    )
)
final_local_train_query_codes = final_local_city_ids.get_indexer(
    final_local_train_pairs["search_location_id"]
)
final_local_train_item_codes = final_local_city_ids.get_indexer(
    final_local_train_pairs["item_location_id"]
)
final_local_region_mask = (
    (final_local_train_query_codes < 0)
    & (final_local_train_item_codes >= 0)
)
final_local_region_ids = pd.Index(
    pd.unique(
        final_local_train_pairs.loc[
            final_local_region_mask, "search_location_id"
        ].dropna()
    )
)
final_local_region_codes = final_local_region_ids.get_indexer(
    final_local_train_pairs.loc[
        final_local_region_mask, "search_location_id"
    ]
)
final_local_region_counts = np.zeros(
    (len(final_local_region_ids), len(final_local_city_ids)),
    dtype=np.int32,
)
np.add.at(
    final_local_region_counts,
    (
        final_local_region_codes,
        final_local_train_item_codes[final_local_region_mask],
    ),
    1,
)
final_local_region_scores = np.zeros_like(
    final_local_region_counts, dtype=np.float32
)
for region_code, city_counts in enumerate(final_local_region_counts):
    maximum = city_counts.max()
    if maximum:
        final_local_region_scores[region_code] = (
            np.log1p(city_counts) / np.log1p(maximum)
        )
final_local_query_region_codes = final_local_region_ids.get_indexer(
    final_query_locations
)


def final_local_geo_score(row, candidates):
    item_codes = final_local_item_city_codes[candidates]
    query_code = final_local_city_ids.get_indexer(
        [final_query_locations[row]]
    )[0]
    score = np.zeros(len(candidates), dtype=np.float32)

    if query_code >= 0:
        known = item_codes >= 0
        bins = np.full(len(candidates), 3, dtype=np.int8)
        distances = final_local_distances[
            query_code, item_codes[known]
        ]
        bins[known] = np.select(
            [
                item_codes[known] == query_code,
                distances <= 25.0,
                distances <= FINAL_LOCAL_RADIUS_KM,
            ],
            [0, 1, 2],
            default=3,
        ).astype(np.int8)
        return (
            FINAL_CITY_GEO_COEFFICIENT
            * np.array([1.00, 0.55, 0.25, 0.00])[bins]
        ).astype(np.float32)

    region_code = final_local_query_region_codes[row]
    known = item_codes >= 0
    if 0 <= region_code < len(final_local_region_scores):
        score[known] = (
            FINAL_REGION_GEO_COEFFICIENT
            * final_local_region_scores[region_code, item_codes[known]]
        )
    return score


# Строгое 90%-совпадение фильтра — часть финального скора.
final_filter_text = benchmark_queries[
    "search_infm_params_text"
].fillna("")
final_unique_filters = pd.Index(
    final_filter_text.loc[final_filter_text.ne("")].unique()
)
final_filter_codes = final_unique_filters.get_indexer(final_filter_text)
filter_item_matrix, filter_query_matrix, filter_token_counts = (
    build_filter_token_matrices(
        benchmark_items["item_infm_params_text"],
        final_unique_filters,
    )
)
required_matches = np.ceil(0.90 * filter_token_counts).astype(np.int32)
final_filter_items = {
    code: item_indices_matching_filter(
        code,
        filter_item_matrix,
        filter_query_matrix,
        required_matches,
    )
    for code in range(len(final_unique_filters))
}
del filter_item_matrix, filter_query_matrix


def final_base_candidates_and_score(row):
    text_candidates, text_scores = source_candidates_and_scores(
        row,
        BASE_HYBRID_WEIGHTS,
        final_source_indices,
        final_source_scores,
        final_source_names,
    )
    parts = [text_candidates]
    embedding_indices = np.asarray(final_embedding_indices[row])
    valid_embedding = embedding_indices >= 0
    embedding_indices = embedding_indices[valid_embedding]
    parts.append(embedding_indices)

    filter_code = final_filter_codes[row]
    if filter_code >= 0:
        parameter_indices = final_parameter_indices[filter_code]
        parts.append(parameter_indices[parameter_indices >= 0])

    candidates = np.unique(np.concatenate(parts))
    score = np.zeros(len(candidates), dtype=np.float32)
    score[np.searchsorted(candidates, text_candidates)] = text_scores

    embedding_scores = np.asarray(
        final_embedding_scores[row], dtype=np.float32
    )[valid_embedding]
    if len(embedding_scores) and embedding_scores[0] > 0:
        score[np.searchsorted(candidates, embedding_indices)] += (
            FINAL_COSINE_COEFFICIENT
            * embedding_scores / embedding_scores[0]
        )
    return candidates, score


def final_current_score(row, candidates, base_candidates, base_score):
    score = np.zeros(len(candidates), dtype=np.float32)
    score[np.searchsorted(candidates, base_candidates)] = base_score
    score += final_local_geo_score(row, candidates)

    filter_code = final_filter_codes[row]
    if filter_code >= 0:
        score += FINAL_FILTER_BONUS * np.isin(
            candidates,
            final_filter_items[filter_code],
            assume_unique=True,
        )
    return score


# Локальный BM25 строится один раз; E5 и глобальные источники не пересчитываются.
final_local_item_text = combine_text(
    benchmark_items, ["item_title_raw", "item_description_raw"]
)
final_local_vectorizer, final_local_item_matrix = build_bm25_item_matrix(
    final_local_item_text
)
final_local_query_matrix = final_local_vectorizer.transform(
    final_query_text
).astype(np.float32)
final_local_query_matrix.data.fill(1.0)


def final_local_candidate_groups():
    for location_id, rows in query_row_groups(final_query_locations):
        city_code = final_local_city_ids.get_indexer([location_id])[0]
        if city_code >= 0:
            item_indices = item_indices_within_radius(
                location_id,
                FINAL_LOCAL_RADIUS_KM,
                final_item_locations,
                final_local_item_city_codes,
                final_local_city_ids,
                final_local_distances,
            )
        else:
            region_code = final_local_query_region_codes[rows[0]]
            item_indices = item_indices_from_region(
                region_code,
                final_local_region_scores,
                final_local_item_city_codes,
            )
        if len(item_indices):
            yield rows, item_indices


final_top50_indices = np.full(
    (len(final_query_ids), TOP_50), -1, dtype=np.int32
)

# Базовый результат: он сохраняется для запросов без локального подкорпуса.
for row in range(len(final_query_ids)):
    base_candidates, base_score = final_base_candidates_and_score(row)
    current_score = final_current_score(
        row, base_candidates, base_candidates, base_score
    )
    final_top50_indices[row] = base_candidates[
        top_positions(current_score, TOP_50)
    ]

for group_number, (rows, item_indices) in enumerate(
    final_local_candidate_groups(), start=1
):
    for row, local_candidates, local_scores in iter_top_k_in_item_subset(
        final_local_query_matrix,
        final_local_item_matrix,
        rows,
        item_indices,
        top_k=FINAL_LOCAL_TOP_K,
    ):
        if not len(local_scores):
            continue

        base_candidates, base_score = final_base_candidates_and_score(row)
        candidates = np.union1d(base_candidates, local_candidates)
        current_score = final_current_score(
            row, candidates, base_candidates, base_score
        )
        local_score = np.zeros(len(candidates), dtype=np.float32)
        local_score[np.searchsorted(candidates, local_candidates)] = (
            local_scores / local_scores[0]
        )

        # Среди кандидатов без local_score порядок не меняется.
        nonlocal_positions = np.flatnonzero(local_score == 0)
        keep_nonlocal = nonlocal_positions[
            top_positions(current_score[nonlocal_positions], TOP_50)
        ]
        keep = np.unique(np.concatenate([
            np.flatnonzero(local_score > 0),
            keep_nonlocal,
        ]))
        current_max = current_score[keep].max()
        current_normalized = (
            current_score[keep] / current_max
            if current_max > 0
            else current_score[keep]
        )
        mixed_score = (
            (1.0 - FINAL_LOCAL_WEIGHT) * current_normalized
            + FINAL_LOCAL_WEIGHT * local_score[keep]
        )
        final_top50_indices[row] = candidates[
            keep[top_positions(mixed_score, TOP_50)]
        ]

    if group_number % 100 == 0:
        print(f"Benchmark локальный поиск: {group_number:,} подкорпусов")

final_answer = pd.DataFrame(
    {
        "query_id": final_query_ids,
        "answer": [
            " ".join(final_item_ids[indices[indices >= 0]])
            for indices in final_top50_indices
        ],
    }
)
final_answer.to_csv(FINAL_LOCAL_ANSWER_PATH, index=False, encoding="utf-8")

answer_check = pd.read_csv(
    FINAL_LOCAL_ANSWER_PATH,
    dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
answer_items = answer_check["answer"].map(str.split)
assert answer_check.columns.tolist() == ["query_id", "answer"]
assert len(answer_check) == len(final_query_ids)
assert answer_check["query_id"].is_unique
assert set(answer_check["query_id"]) == set(final_query_ids)
assert answer_items.map(len).le(TOP_50).all()
assert answer_items.map(lambda values: len(values) == len(set(values))).all()
assert answer_items.map(
    lambda values: set(values).issubset(set(final_item_ids))
).all()

display(pd.DataFrame([{
    "file": str(FINAL_LOCAL_ANSWER_PATH),
    "rows": len(answer_check),
    "min_item_ids": answer_items.map(len).min(),
    "max_item_ids": answer_items.map(len).max(),
    "all_checks_passed": True,
}]))
print(f"Создан файл: {FINAL_LOCAL_ANSWER_PATH.resolve()}")

del final_local_item_matrix, final_local_query_matrix
del final_filter_items
gc.collect()



Benchmark локальный поиск: 100 подкорпусов
Benchmark локальный поиск: 200 подкорпусов


,file,rows,min_item_ids,max_item_ids,all_checks_passed
0,answer_geo_local_mixed.csv,2452,50,50,True


Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_geo_local_mixed.csv


0

## 26. История кликов как отдельный источник

Проверяем, какая часть benchmark-запросов и объявлений пересекается с train и может получить исторический сигнал.

In [78]:
# Диагностика точного переноса истории train → benchmark.
# Это не меняет текущий ответ и не считает эмбеддинги.

history_query_columns = [
    "train_query_id",
    "search_query",
    "search_category",
    "search_infm_params_text",
    "search_is_delivery_search",
    "search_location_id",
]
history_clicks = (
    relevance.merge(
        train_queries[history_query_columns],
        on="train_query_id",
        how="inner",
        validate="many_to_one",
    )
    .assign(item_id=lambda frame: frame["item_id"].astype(str))
    .drop_duplicates()
)
history_clicks["item_is_in_benchmark"] = history_clicks["item_id"].isin(
    benchmark_items["item_id"].astype(str)
)

history_rules = {
    "same_text": ["search_query"],
    "text_category_filters_delivery": [
        "search_query",
        "search_category",
        "search_infm_params_text",
        "search_is_delivery_search",
    ],
    "full_query_features": QUERY_FEATURE_COLUMNS,
}

history_stats_by_rule = {}
history_overlap_rows = []
for rule_name, rule_columns in history_rules.items():
    rule_stats = (
        history_clicks.groupby(rule_columns, dropna=False)
        .agg(
            train_clicks=("item_id", "size"),
            historical_items=("item_id", "nunique"),
            benchmark_items=("item_is_in_benchmark", "sum"),
        )
        .reset_index()
    )
    history_stats_by_rule[rule_name] = rule_stats

    matched_benchmark = benchmark_queries[rule_columns].merge(
        rule_stats, on=rule_columns, how="left", validate="many_to_one"
    )
    has_history = matched_benchmark["historical_items"].notna()
    has_usable_history = matched_benchmark["benchmark_items"].fillna(0).gt(0)
    history_overlap_rows.append(
        {
            "rule": rule_name,
            "benchmark_queries_with_history": int(has_history.sum()),
            "history_share_pct": round(has_history.mean() * 100, 2),
            "queries_with_historical_item_in_benchmark": int(
                has_usable_history.sum()
            ),
            "usable_history_share_pct": round(
                has_usable_history.mean() * 100, 2
            ),
            "median_historical_items_in_benchmark": (
                matched_benchmark.loc[has_usable_history, "benchmark_items"]
                .median()
            ),
        }
    )

history_overlap_report = pd.DataFrame(history_overlap_rows)
display(history_overlap_report)


,rule,benchmark_queries_with_history,history_share_pct,queries_with_historical_item_in_benchmark,usable_history_share_pct,median_historical_items_in_benchmark
0,same_text,919,37.4800,366,14.9300,1.0000
1,text_category_filters_delivery,682,27.8100,247,10.0700,1.0000
2,full_query_features,115,4.6900,64,2.6100,1.0000


### 26.1. Точный перенос выбранных объявлений

In [79]:
# Проверка переноса истории между локациями.
# Основная validation для этого не подходит: в ней нет тех же текстов, что в train.

history_key = history_rules["text_category_filters_delivery"]
history_group_size = train_queries.groupby(history_key, dropna=False)[
    "train_query_id"
].transform("size")

# В каждой повторяющейся группе прячем один запрос, а остальные оставляем историей.
history_valid_query_ids = (
    train_queries.loc[history_group_size.gt(1)]
    .groupby(history_key, dropna=False, group_keys=False)
    .sample(n=1, random_state=42)["train_query_id"]
)
history_valid_mask = relevance["train_query_id"].isin(history_valid_query_ids)

history_source = (
    relevance.loc[~history_valid_mask]
    .merge(
        train_queries[["train_query_id"] + history_key],
        on="train_query_id",
        validate="many_to_one",
    )
    .groupby(history_key + ["item_id"], dropna=False)
    .size()
    .rename("click_count")
    .reset_index()
    .sort_values(
        history_key + ["click_count", "item_id"],
        ascending=[True] * len(history_key) + [False, True],
    )
)
history_source["history_rank"] = (
    history_source.groupby(history_key, dropna=False).cumcount() + 1
)

history_targets = relevance.loc[history_valid_mask].merge(
    train_queries[["train_query_id"] + history_key],
    on="train_query_id",
    validate="many_to_one",
)
history_target_sizes = history_targets.groupby("train_query_id").size()
history_hits = history_targets.merge(
    history_source,
    on=history_key + ["item_id"],
    how="inner",
    validate="many_to_one",
)

history_validation_rows = []
for top_k in [1, 3, 10, 50]:
    hit_counts = (
        history_hits.loc[history_hits["history_rank"].le(top_k)]
        .groupby("train_query_id")["item_id"]
        .nunique()
    )
    recall_by_query = (
        hit_counts.reindex(history_valid_query_ids, fill_value=0)
        / history_target_sizes.reindex(history_valid_query_ids)
    )
    history_validation_rows.append(
        {
            "top_k_from_history": top_k,
            "validation_queries": len(history_valid_query_ids),
            "recall_at_k_from_history": recall_by_query.mean(),
            "queries_with_any_hit_pct": recall_by_query.gt(0).mean() * 100,
        }
    )

history_transfer_validation = pd.DataFrame(history_validation_rows)
display(history_transfer_validation)


,top_k_from_history,validation_queries,recall_at_k_from_history,queries_with_any_hit_pct
0,1,35350,0.0054,0.5941
1,3,35350,0.0076,0.8600
2,10,35350,0.0093,1.0863
3,50,35350,0.0102,1.2023


### 26.2. BM25 по историческим запросам объявления

In [81]:
# Новый источник: запрос ищется среди исторических запросов каждого объявления.
# В его документах используются только relevance_train, без validation-кликов.
HISTORY_BM25_TOP_K = 3_000
HISTORY_BM25_DEPTHS = np.array([50, 500, 1_000, 3_000], dtype=np.int32)

assert len(base_found_by_row) == len(saved_query_ids), (
    "Сначала выполните ячейку проверки локальных источников: "
    "она сохраняет текущий пул для сравнения."
)

history_item_documents = (
    relevance_train.merge(
        queries_train[["train_query_id", "search_query"]],
        on="train_query_id",
        validate="many_to_one",
    )[["item_id", "search_query"]]
    .drop_duplicates()
    .groupby("item_id", sort=False)["search_query"]
    .agg(" ".join)
)
history_item_positions = pd.Index(train_items["item_id"].astype(str)).get_indexer(
    history_item_documents.index.astype(str)
)
assert np.all(history_item_positions >= 0)

history_vectorizer, history_item_matrix = build_bm25_item_matrix(
    history_item_documents.to_numpy()
)
history_valid_query_text = (
    queries_valid.set_index("train_query_id")
    .reindex(saved_query_ids)["search_query"]
    .fillna("")
    .to_numpy()
)
history_unique_query_text, history_query_codes = np.unique(
    history_valid_query_text, return_inverse=True
)
history_query_matrix = history_vectorizer.transform(
    history_unique_query_text
).astype(np.float32)
history_query_matrix.data.fill(1.0)

# Один и тот же текст запроса ищется один раз, затем результат применяется
# ко всем его версиям с разными локациями и фильтрами.
history_order = np.argsort(history_query_codes, kind="stable")
history_split = np.flatnonzero(
    np.diff(history_query_codes[history_order])
) + 1
history_rows_by_query = np.split(history_order, history_split)

history_relevant_indices = (
    relevance_valid.merge(
        train_items.reset_index(names="item_index")[
            ["item_id", "item_index"]
        ],
        on="item_id",
        validate="many_to_one",
    )
    .groupby("train_query_id")["item_index"]
    .agg(list)
)
history_relevant_by_row = [
    np.asarray(history_relevant_indices.loc[query_id], dtype=np.int32)
    for query_id in saved_query_ids
]

history_source_recall = np.zeros(len(HISTORY_BM25_DEPTHS), dtype=np.float64)
history_added_recall = np.zeros_like(history_source_recall)
history_new_relevant = np.zeros(len(HISTORY_BM25_DEPTHS), dtype=np.int64)
history_queries_with_new = np.zeros(len(HISTORY_BM25_DEPTHS), dtype=np.int32)

for source_number, (unique_row, document_candidates, _) in enumerate(
    iter_top_k_in_item_subset(
        history_query_matrix,
        history_item_matrix,
        np.arange(len(history_unique_query_text)),
        np.arange(len(history_item_positions)),
        top_k=HISTORY_BM25_TOP_K,
    ),
    start=1,
):
    source_candidates = history_item_positions[document_candidates]
    for row in history_rows_by_query[unique_row]:
        relevant = history_relevant_by_row[row]
        base_found = np.asarray(base_found_by_row[row], dtype=bool)
        for depth_position, top_k in enumerate(HISTORY_BM25_DEPTHS):
            source_found = np.isin(relevant, source_candidates[:top_k])
            added = source_found & ~base_found
            history_source_recall[depth_position] += source_found.mean()
            history_added_recall[depth_position] += added.mean()
            history_new_relevant[depth_position] += int(added.sum())
            history_queries_with_new[depth_position] += int(added.any())
    if source_number % 2_000 == 0 or source_number == len(history_unique_query_text):
        print(
            f"Исторический BM25: {source_number:,}/"
            f"{len(history_unique_query_text):,} уникальных текстов"
        )

history_bm25_validation = pd.DataFrame(
    {
        "top_k": HISTORY_BM25_DEPTHS,
        "history_source_recall": (
            history_source_recall / len(saved_query_ids)
        ),
        "current_global_pool_recall": base_candidate_recall,
        "union_candidate_recall": (
            base_candidate_recall
            + history_added_recall / len(saved_query_ids)
        ),
        "added_candidate_recall": (
            history_added_recall / len(saved_query_ids)
        ),
        "new_relevant_items": history_new_relevant,
        "queries_with_new_relevant_items": history_queries_with_new,
    }
)
display(history_bm25_validation)

del history_vectorizer, history_item_matrix, history_query_matrix
del history_item_documents, history_relevant_indices
gc.collect()


Исторический BM25: 2,000/14,741 уникальных текстов
Исторический BM25: 4,000/14,741 уникальных текстов
Исторический BM25: 6,000/14,741 уникальных текстов
Исторический BM25: 8,000/14,741 уникальных текстов
Исторический BM25: 10,000/14,741 уникальных текстов
Исторический BM25: 12,000/14,741 уникальных текстов
Исторический BM25: 14,000/14,741 уникальных текстов
Исторический BM25: 14,741/14,741 уникальных текстов


,top_k,history_source_recall,current_global_pool_recall,union_candidate_recall,added_candidate_recall,new_relevant_items,queries_with_new_relevant_items
0,50,0.0309,0.9482,0.9484,0.0002,21,21
1,500,0.0930,0.9482,0.9491,0.0009,112,111
2,1000,0.1166,0.9482,0.9496,0.0014,172,171
3,3000,0.1523,0.9482,0.9503,0.0021,268,251


0

## 27. CatBoost на текущем пуле

Сравниваем повторное ранжирование первых 100 и 300 объявлений текущей формулы. Первые 3 000 запросов с разными текстами служат обучением модели, следующие 1 500 — её проверкой. Исходное место объявления, координаты и расстояние передаются как отдельные признаки.

In [82]:
# Новый CatBoost: проверяем, сколько релевантных объявлений доходит до top-100/300.
# Запросы модели имеют разные тексты; совпадающий текст не попадёт в обе части.
from scipy.sparse import load_npz

CB_TRAIN_QUERIES = 3_000
CB_VALID_QUERIES = 1_500
CB_MAX_K = 300
CB_CITY_WEIGHT = 1.50
CB_REGION_WEIGHT = 0.75
CB_FILTER_WEIGHT = 0.55
CB_LOCAL_WEIGHT = 0.05
CB_COSINE_WEIGHT = 0.05

# После финального benchmark исходные validation-массивы могли быть выгружены.
extended_top_indices = {
    name: np.load(SAVE_DIR / f"{name}_top5000_indices.npy", mmap_mode="r")
    for name in SOURCE_NAMES
}
extended_top_scores = {
    name: np.load(SAVE_DIR / f"{name}_top5000_scores.npy", mmap_mode="r")
    for name in SOURCE_NAMES
}
embedding_top_indices = np.load(
    SAVE_DIR / "multilingual_e5_title_description_top5000_indices.npy",
    mmap_mode="r",
)
embedding_top_scores = np.load(
    SAVE_DIR / "multilingual_e5_title_description_top5000_scores.npy",
    mmap_mode="r",
)

query_table = queries_valid.set_index("train_query_id").reindex(saved_query_ids)
unique_text_rows = query_table.reset_index(drop=True).drop_duplicates(
    "search_query"
).index.to_numpy(dtype=np.int32)
np.random.default_rng(42).shuffle(unique_text_rows)
model_rows = unique_text_rows[:CB_TRAIN_QUERIES + CB_VALID_QUERIES]
assert len(model_rows) == CB_TRAIN_QUERIES + CB_VALID_QUERIES
model_train_rows = model_rows[:CB_TRAIN_QUERIES]
model_valid_rows = model_rows[CB_TRAIN_QUERIES:]
model_row_position = np.full(len(saved_query_ids), -1, dtype=np.int32)
model_row_position[model_rows] = np.arange(len(model_rows))

# Фильтр нужен для точного воспроизведения текущего скора.
cb_item_tokens, cb_query_tokens, cb_token_counts = (
    build_filter_token_matrices(
        train_items["item_infm_params_text"], unique_parameter_filters
    )
)
cb_required_matches = np.ceil(0.90 * cb_token_counts).astype(np.int32)
cb_filter_items = {
    code: item_indices_matching_filter(
        code, cb_item_tokens, cb_query_tokens, cb_required_matches
    )
    for code in np.unique(parameter_filter_codes[model_rows]) if code >= 0
}
del cb_item_tokens, cb_query_tokens

cb_query_matrix_text = query_table["search_query"].fillna("")
cb_local_vectorizer = joblib.load(
    SAVE_DIR / "bm25_title_description_vectorizer.joblib"
)
cb_local_item_matrix = load_npz(
    SAVE_DIR / "bm25_title_description_item_matrix.npz"
).tocsr()
cb_local_query_matrix = cb_local_vectorizer.transform(
    cb_query_matrix_text
).astype(np.float32)
cb_local_query_matrix.data.fill(1.0)

cb_items = np.full((len(model_rows), CB_MAX_K), -1, dtype=np.int32)
cb_scores = np.zeros_like(cb_items, dtype=np.float32)
cb_local_scores = np.zeros_like(cb_scores)
cb_done = np.zeros(len(model_rows), dtype=bool)

def store_cb_candidates(row, local_candidates=None, local_scores=None):
    base_items, base_scores = candidates_with_embedding_pool(row)
    candidates = (
        np.union1d(base_items, local_candidates)
        if local_candidates is not None else base_items
    )
    raw = np.zeros(len(candidates), dtype=np.float32)
    raw[np.searchsorted(candidates, base_items)] = (
        base_scores - location_bonus(
            base_items, valid_query_locations[row],
            best_location_bonus, item_locations,
        )
    )
    segment, geo = tuning_geo_signal(row, candidates)
    raw += (CB_CITY_WEIGHT if segment == "city" else CB_REGION_WEIGHT) * geo
    raw += CB_COSINE_WEIGHT * cosine_values_in_embedding_pool(row, candidates)
    filter_code = parameter_filter_codes[row]
    if filter_code >= 0:
        raw += CB_FILTER_WEIGHT * np.isin(
            candidates, cb_filter_items[filter_code], assume_unique=True
        )
    if raw.max() > 0:
        raw /= raw.max()

    local = np.zeros(len(candidates), dtype=np.float32)
    if local_candidates is not None and len(local_candidates) and local_scores[0] > 0:
        local[np.searchsorted(candidates, local_candidates)] = (
            local_scores / local_scores[0]
        )
    mixed = (1 - CB_LOCAL_WEIGHT) * raw + CB_LOCAL_WEIGHT * local
    top = top_positions(mixed, CB_MAX_K)
    position = model_row_position[row]
    assert len(top) == CB_MAX_K
    cb_items[position] = candidates[top]
    cb_scores[position] = mixed[top]
    cb_local_scores[position] = local[top]
    cb_done[position] = True

for group_number, (rows, item_indices) in enumerate(
    geo_candidate_groups(), start=1
):
    selected_rows = rows[model_row_position[rows] >= 0]
    if not len(selected_rows):
        continue
    for row, local_candidates, local_scores in iter_top_k_in_item_subset(
        cb_local_query_matrix, cb_local_item_matrix, selected_rows,
        item_indices, top_k=3_000,
    ):
        store_cb_candidates(row, local_candidates, local_scores)
    if group_number % 100 == 0:
        print(f"CatBoost-кандидаты: {group_number:,} локаций")

for row in model_rows[~cb_done]:
    store_cb_candidates(row)
assert cb_done.all()

cb_ceiling = pd.DataFrame([
    {
        "top_k": top_k,
        "candidate_recall_train": np.mean([
            recall_for_item_indices(row, cb_items[pos, :top_k])
            for pos, row in enumerate(model_train_rows)
        ]),
        "candidate_recall_valid": np.mean([
            recall_for_item_indices(
                row, cb_items[CB_TRAIN_QUERIES + pos, :top_k]
            )
            for pos, row in enumerate(model_valid_rows)
        ]),
    }
    for top_k in [50, 100, 200, 300]
])
display(cb_ceiling)

del cb_local_vectorizer, cb_local_item_matrix, cb_local_query_matrix
gc.collect()


CatBoost-кандидаты: 200 локаций
CatBoost-кандидаты: 300 локаций
CatBoost-кандидаты: 400 локаций
CatBoost-кандидаты: 600 локаций
CatBoost-кандидаты: 900 локаций


,top_k,candidate_recall_train,candidate_recall_valid
0,50,0.8058,0.7984
1,100,0.8570,0.8597
2,200,0.8912,0.8921
3,300,0.9058,0.9109


55

In [87]:
# Признаки только для уже отобранных top-300: исходные индексы не пересчитываем.
CB_NUMERIC_FEATURES = [
    "mixed_score", "hybrid_rank",
    "bm25_title_description", "char_tfidf_title",
    "bm25_title", "word_tfidf_title",
    "e5_cosine", "local_bm25", "geo_score",
    "same_location", "query_latitude", "query_longitude",
    "item_latitude", "item_longitude", "distance_km",
    "filter_match", "parameter_bm25",
    "log_price", "price_missing", "rating",
    "rating_missing", "log_reviews", "reviews_missing",
    "phone_hidden", "message_forbidden", "delivery",
    "has_query_coordinates",
]
CB_CATEGORICAL_FEATURES = [
    "search_category", "item_category_id", "item_microcat_id"
]

def cb_lookup_scores(candidates, indices, scores):
    """Скор одного сохранённого источника для выбранных объявлений."""
    valid = indices >= 0
    ids = np.asarray(indices[valid], dtype=np.int32)
    values = np.asarray(scores[valid], dtype=np.float32)
    result = np.zeros(len(candidates), dtype=np.float32)
    if not len(ids) or values[0] <= 0:
        return result
    order = np.argsort(ids)
    sorted_ids = ids[order]
    positions = np.searchsorted(sorted_ids, candidates)
    in_range = positions < len(sorted_ids)
    found = np.zeros(len(candidates), dtype=bool)
    found[in_range] = sorted_ids[positions[in_range]] == candidates[in_range]
    result[found] = values[order[positions[found]]] / values[0]
    return result

cb_item_lat = train_items["item_latitude"].astype("float64").to_numpy(np.float32)
cb_item_lon = train_items["item_longitude"].astype("float64").to_numpy(np.float32)
cb_query_centres = geo_city_centers.reindex(
    valid_query_locations[model_rows]
).to_numpy(dtype=np.float32)

cb_price = pd.to_numeric(train_items["item_price"], errors="coerce").astype("float64")
cb_rating = pd.to_numeric(train_items["item_rating"], errors="coerce").astype("float64")
cb_reviews = pd.to_numeric(
    train_items["item_rating_reviews_count"], errors="coerce"
).astype("float64")
cb_log_price = np.log1p(cb_price.fillna(0).clip(lower=0)).to_numpy(np.float32)
cb_log_reviews = np.log1p(cb_reviews.fillna(0)).to_numpy(np.float32)
cb_rating_values = cb_rating.fillna(0).to_numpy(np.float32)
cb_price_missing = cb_price.isna().to_numpy(np.float32)
cb_rating_missing = cb_rating.isna().to_numpy(np.float32)
cb_reviews_missing = cb_reviews.isna().to_numpy(np.float32)
cb_phone_hidden = train_items["item_is_phone_hidden"].fillna(False).to_numpy(np.float32)
cb_message_forbidden = train_items["item_is_message_forbidden"].fillna(False).to_numpy(np.float32)
cb_item_category = pd.to_numeric(
    train_items["item_category_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_item_microcat = pd.to_numeric(
    train_items["item_microcat_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_query_category = pd.to_numeric(
    query_table["search_category"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_query_delivery = pd.to_numeric(
    query_table["search_is_delivery_search"], errors="coerce"
).fillna(0).to_numpy(np.float32)

cb_numeric = np.empty(
    (len(model_rows) * CB_MAX_K, len(CB_NUMERIC_FEATURES)), dtype=np.float32
)
cb_categorical = np.empty(
    (len(model_rows) * CB_MAX_K, len(CB_CATEGORICAL_FEATURES)),
    dtype=np.int32,
)
cb_labels = np.empty(len(model_rows) * CB_MAX_K, dtype=np.int8)

for position, row in enumerate(model_rows):
    items = cb_items[position]
    query_lat, query_lon = cb_query_centres[position]
    item_lat, item_lon = cb_item_lat[items], cb_item_lon[items]
    # У неизвестной локации расстояние остаётся NaN; CatBoost умеет его учитывать.
    lat_delta = np.deg2rad(item_lat - query_lat)
    lon_delta = np.deg2rad(item_lon - query_lon)
    distance_part = (
        np.sin(lat_delta / 2) ** 2
        + np.cos(np.deg2rad(query_lat))
        * np.cos(np.deg2rad(item_lat))
        * np.sin(lon_delta / 2) ** 2
    )
    distance_km = 2 * 6371.0088 * np.arcsin(
        np.sqrt(np.clip(distance_part, 0, 1))
    )
    segment, geo_signal = tuning_geo_signal(row, items)
    geo_score = (CB_CITY_WEIGHT if segment == "city" else CB_REGION_WEIGHT) * geo_signal
    filter_code = parameter_filter_codes[row]
    filter_match = (
        np.isin(items, cb_filter_items[filter_code]).astype(np.float32)
        if filter_code >= 0 else np.zeros(CB_MAX_K, dtype=np.float32)
    )
    parameter_score = (
        cb_lookup_scores(
            items, parameter_top_indices[filter_code],
            parameter_top_scores[filter_code],
        ) if filter_code >= 0 else np.zeros(CB_MAX_K, dtype=np.float32)
    )
    text_scores = [
        cb_lookup_scores(
            items, extended_top_indices[name][row],
            extended_top_scores[name][row],
        )
        for name in SOURCE_NAMES
    ]
    e5_score = cb_lookup_scores(
        items, embedding_top_indices[row], embedding_top_scores[row]
    )
    start = position * CB_MAX_K
    stop = start + CB_MAX_K
    cb_numeric[start:stop] = np.column_stack([
        cb_scores[position], np.arange(1, CB_MAX_K + 1),
        *text_scores, e5_score, cb_local_scores[position], geo_score,
        (item_locations[items] == valid_query_locations[row]).astype(np.float32),
        np.full(CB_MAX_K, query_lat), np.full(CB_MAX_K, query_lon),
        item_lat, item_lon, distance_km, filter_match, parameter_score,
        cb_log_price[items], cb_price_missing[items],
        cb_rating_values[items], cb_rating_missing[items],
        cb_log_reviews[items], cb_reviews_missing[items],
        cb_phone_hidden[items], cb_message_forbidden[items],
        np.full(CB_MAX_K, cb_query_delivery[row]),
        np.full(CB_MAX_K, np.isfinite(query_lat)),
    ])
    cb_categorical[start:stop] = np.column_stack([
        np.full(CB_MAX_K, cb_query_category[row]),
        cb_item_category[items], cb_item_microcat[items],
    ])
    cb_labels[start:stop] = np.isin(
        items, np.fromiter(
            relevant_indices_by_query[saved_query_ids[row]], dtype=np.int32
        )
    )
    if (position + 1) % 500 == 0 or position + 1 == len(model_rows):
        print(f"Признаки CatBoost: {position + 1:,}/{len(model_rows):,}")

cb_features = pd.DataFrame(cb_numeric, columns=CB_NUMERIC_FEATURES)
for column_number, name in enumerate(CB_CATEGORICAL_FEATURES):
    cb_features[name] = cb_categorical[:, column_number]
print(f"Признаков на кандидата: {cb_features.shape[1]}")
del cb_numeric, cb_categorical
gc.collect()


Признаки CatBoost: 500/4,500
Признаки CatBoost: 1,000/4,500
Признаки CatBoost: 1,500/4,500
Признаки CatBoost: 2,000/4,500
Признаки CatBoost: 2,500/4,500
Признаки CatBoost: 3,000/4,500
Признаки CatBoost: 3,500/4,500
Признаки CatBoost: 4,000/4,500
Признаки CatBoost: 4,500/4,500
Признаков на кандидата: 30


8469

In [88]:
# CatBoost получает уже отсортированные кандидаты и их прежний скор.
# Учим только на запросах, где среди этих кандидатов есть выбранное объявление.
from catboost import CatBoostRanker, Pool

def fit_current_catboost(top_k):
    labels_by_query = cb_labels.reshape(len(model_rows), CB_MAX_K)
    train_groups = np.flatnonzero(
        labels_by_query[:CB_TRAIN_QUERIES, :top_k].any(axis=1)
    )
    train_indices = (
        train_groups[:, None] * CB_MAX_K + np.arange(top_k)
    ).ravel()
    valid_indices = (
        np.arange(CB_TRAIN_QUERIES, len(model_rows))[:, None] * CB_MAX_K
        + np.arange(top_k)
    ).ravel()
    train_pool = Pool(
        cb_features.iloc[train_indices], label=cb_labels[train_indices],
        group_id=np.repeat(train_groups, top_k),
        cat_features=CB_CATEGORICAL_FEATURES,
    )
    valid_pool = Pool(
        cb_features.iloc[valid_indices], label=cb_labels[valid_indices],
        group_id=np.repeat(
            np.arange(CB_TRAIN_QUERIES, len(model_rows)), top_k
        ),
        cat_features=CB_CATEGORICAL_FEATURES,
    )
    print(f"CatBoost: top-{top_k}, обучающих запросов {len(train_groups):,}")
    model = CatBoostRanker(
        loss_function="YetiRank", eval_metric="NDCG:top=50",
        iterations=200, depth=6, learning_rate=0.08, l2_leaf_reg=5,
        random_seed=42, thread_count=4,
        allow_writing_files=False, verbose=50,
    )
    model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=25)
    model_scores = model.predict(valid_pool).reshape(
        CB_VALID_QUERIES, top_k
    ).astype(np.float32)
    model_min = model_scores.min(axis=1, keepdims=True)
    model_range = np.ptp(model_scores, axis=1, keepdims=True)
    model_scores = (model_scores - model_min) / np.maximum(model_range, 1e-8)
    original_scores = cb_scores[CB_TRAIN_QUERIES:, :top_k]
    validation_items = cb_items[CB_TRAIN_QUERIES:, :top_k]
    results = []
    for model_weight in [0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00]:
        mixed_scores = (
            (1 - model_weight) * original_scores
            + model_weight * model_scores
        )
        recall = np.mean([
            recall_for_item_indices(
                row, validation_items[position][
                    top_positions(mixed_scores[position], TOP_50)
                ],
            )
            for position, row in enumerate(model_valid_rows)
        ])
        results.append({
            "top_k": top_k, "model_weight": model_weight,
            "recall_at_50": recall,
        })
    model_path = Path("artifacts/models") / f"catboost_current_top{top_k}.cbm"
    model_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_path))
    print(f"Модель сохранена: {model_path}")
    del train_pool, valid_pool
    gc.collect()
    return pd.DataFrame(results), model

cb100_results, cb100_model = fit_current_catboost(100)
display(cb100_results.sort_values("recall_at_50", ascending=False))


CatBoost: top-100, обучающих запросов 2,593
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.4985720	best: 0.4985720 (0)	total: 4.88s	remaining: 16m 12s
50:	test: 0.6197253	best: 0.6197253 (50)	total: 1m 26s	remaining: 4m 12s
100:	test: 0.6251618	best: 0.6257487 (97)	total: 2m 9s	remaining: 2m 6s
150:	test: 0.6264828	best: 0.6272994 (134)	total: 2m 50s	remaining: 55.5s
Stopped by overfitting detector  (25 iterations wait)

bestTest = 0.6272993752
bestIteration = 134

Shrink model to first 135 iterations.
Модель сохранена: artifacts\models\catboost_current_top100.cbm


,top_k,model_weight,recall_at_50
4,100,0.3500,0.8180
5,100,0.5000,0.8175
6,100,0.7500,0.8138
2,100,0.1000,0.8133
7,100,1.0000,0.8123
3,100,0.2000,0.8123
1,100,0.0500,0.8048
0,100,0.0000,0.7988


In [89]:
# Тот же эксперимент, но CatBoost может переставить первые 300.
cb300_results, cb300_model = fit_current_catboost(300)
display(pd.concat([cb100_results, cb300_results]).sort_values(
    "recall_at_50", ascending=False
))
display(pd.DataFrame({
    "feature": cb300_model.feature_names_,
    "importance": cb300_model.get_feature_importance(
        type="PredictionValuesChange"
    ),
}).sort_values("importance", ascending=False).head(20))


CatBoost: top-300, обучающих запросов 2,735
Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.4652898	best: 0.4652898 (0)	total: 6.28s	remaining: 20m 49s
50:	test: 0.5675335	best: 0.5675335 (50)	total: 2m 21s	remaining: 6m 54s
100:	test: 0.5722737	best: 0.5727479 (91)	total: 4m	remaining: 3m 55s
150:	test: 0.5772046	best: 0.5777239 (139)	total: 5m 39s	remaining: 1m 50s
Stopped by overfitting detector  (25 iterations wait)

bestTest = 0.5777239476
bestIteration = 139

Shrink model to first 140 iterations.
Модель сохранена: artifacts\models\catboost_current_top300.cbm


,top_k,model_weight,recall_at_50
6,300,0.7500,0.8256
5,300,0.5000,0.8236
7,300,1.0000,0.8218
4,300,0.3500,0.8208
4,100,0.3500,0.8180
5,100,0.5000,0.8175
3,300,0.2000,0.8174
6,100,0.7500,0.8138
2,100,0.1000,0.8133
7,100,1.0000,0.8123


,feature,importance
6,e5_cosine,35.4370
0,mixed_score,28.2611
1,hybrid_rank,14.9904
8,geo_score,10.0690
5,word_tfidf_title,5.7560
7,local_bm25,1.9566
2,bm25_title_description,1.1517
14,distance_km,0.8243
3,char_tfidf_title,0.5347
4,bm25_title,0.4779


## 28. CatBoost на benchmark

Повторяем для benchmark тот же top-300 и те же признаки. Модель уже обучена; создаём отдельный CSV, не меняя предыдущий ответ.

In [90]:
# Текущая формула + локальный BM25; готовые E5 и глобальные top-5000 не пересчитываем.
needed = [
    "final_source_indices", "final_embedding_indices",
    "final_base_candidates_and_score", "final_current_score",
    "final_local_candidate_groups", "final_top50_indices",
]
missing = [name for name in needed if name not in globals()]
assert not missing, "Сначала запустите предыдущие benchmark-ячейки: " + ", ".join(missing)
assert tuple(SOURCE_NAMES) == final_source_names

cb_filter_item_matrix, cb_filter_query_matrix, cb_filter_counts = (
    build_filter_token_matrices(
        benchmark_items["item_infm_params_text"], final_unique_filters
    )
)
cb_filter_required = np.ceil(0.90 * cb_filter_counts).astype(np.int32)
final_filter_items = {
    code: item_indices_matching_filter(
        code, cb_filter_item_matrix, cb_filter_query_matrix, cb_filter_required
    )
    for code in range(len(final_unique_filters))
}
del cb_filter_item_matrix, cb_filter_query_matrix

cb_test_vectorizer, cb_test_item_matrix = build_bm25_item_matrix(
    final_local_item_text
)
cb_test_query_matrix = cb_test_vectorizer.transform(
    final_query_text
).astype(np.float32)
cb_test_query_matrix.data.fill(1.0)
cb_test_items = np.full((len(final_query_ids), 300), -1, dtype=np.int32)
cb_test_scores = np.zeros_like(cb_test_items, dtype=np.float32)
cb_test_local_scores = np.zeros_like(cb_test_scores)
cb_test_done = np.zeros(len(final_query_ids), dtype=bool)

def store_cb_test_candidates(row, local_items=None, local_scores=None):
    base_items, base_scores = final_base_candidates_and_score(row)
    candidates = (
        np.union1d(base_items, local_items)
        if local_items is not None else base_items
    )
    current = final_current_score(row, candidates, base_items, base_scores)
    if current.max() > 0:
        current /= current.max()
    local = np.zeros(len(candidates), dtype=np.float32)
    if local_items is not None and len(local_items) and local_scores[0] > 0:
        local[np.searchsorted(candidates, local_items)] = (
            local_scores / local_scores[0]
        )
    mixed = (1 - FINAL_LOCAL_WEIGHT) * current + FINAL_LOCAL_WEIGHT * local
    top = top_positions(mixed, 300)
    assert len(top) == 300
    cb_test_items[row] = candidates[top]
    cb_test_scores[row] = mixed[top]
    cb_test_local_scores[row] = local[top]
    cb_test_done[row] = True

for group_number, (rows, item_indices) in enumerate(
    final_local_candidate_groups(), start=1
):
    for row, local_items, local_scores in iter_top_k_in_item_subset(
        cb_test_query_matrix, cb_test_item_matrix, rows, item_indices,
        top_k=FINAL_LOCAL_TOP_K,
    ):
        store_cb_test_candidates(row, local_items, local_scores)
    if group_number % 100 == 0:
        print(f"CatBoost benchmark: {group_number:,} локаций")

for row in np.flatnonzero(~cb_test_done):
    store_cb_test_candidates(row)
assert cb_test_done.all()
baseline_overlap = np.mean([
    np.isin(cb_test_items[row, :50], final_top50_indices[row]).mean()
    for row in range(len(final_query_ids))
])
print(f"Совпадение с прежним top-50: {baseline_overlap:.2%}")
assert baseline_overlap > 0.95, "Новый пул не повторил прежний benchmark-скор"
del cb_test_vectorizer, cb_test_item_matrix, cb_test_query_matrix
gc.collect()


CatBoost benchmark: 100 локаций
CatBoost benchmark: 200 локаций
Совпадение с прежним top-50: 99.53%


0

In [98]:
# Те же 30 признаков и тот же порядок столбцов, что при обучении.
_, cb_parameter_matrix, cb_parameter_indices, cb_parameter_scores = retrieve_top_k(
    "bm25", benchmark_items["item_infm_params_text"].fillna(""),
    final_unique_filters, FINAL_TOP_K, batch_size=32,
    source_name="bm25_parameters",
)
del cb_parameter_matrix

cb_test_item_lat = benchmark_items["item_latitude"].astype("float64").to_numpy(np.float32)
cb_test_item_lon = benchmark_items["item_longitude"].astype("float64").to_numpy(np.float32)
cb_test_query_centres = final_local_centers.reindex(
    final_query_locations
).to_numpy(dtype=np.float32)
cb_test_price = pd.to_numeric(benchmark_items["item_price"], errors="coerce").astype("float64")
cb_test_rating = pd.to_numeric(benchmark_items["item_rating"], errors="coerce").astype("float64")
cb_test_reviews = pd.to_numeric(
    benchmark_items["item_rating_reviews_count"], errors="coerce"
).astype("float64")
cb_test_log_price = np.log1p(cb_test_price.fillna(0).clip(lower=0)).to_numpy(np.float32)
cb_test_log_reviews = np.log1p(cb_test_reviews.fillna(0)).to_numpy(np.float32)
cb_test_rating_values = cb_test_rating.fillna(0).to_numpy(np.float32)
cb_test_price_missing = cb_test_price.isna().to_numpy(np.float32)
cb_test_rating_missing = cb_test_rating.isna().to_numpy(np.float32)
cb_test_reviews_missing = cb_test_reviews.isna().to_numpy(np.float32)
cb_test_phone = benchmark_items["item_is_phone_hidden"].fillna(False).to_numpy(np.float32)
cb_test_message = benchmark_items["item_is_message_forbidden"].fillna(False).to_numpy(np.float32)
cb_test_category = pd.to_numeric(
    benchmark_items["item_category_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_test_microcat = pd.to_numeric(
    benchmark_items["item_microcat_id"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_test_query_category = pd.to_numeric(
    benchmark_queries["search_category"], errors="coerce"
).fillna(-1).to_numpy(np.int32)
cb_test_delivery = pd.to_numeric(
    benchmark_queries["search_is_delivery_search"], errors="coerce"
).fillna(0).to_numpy(np.float32)

cb_test_numeric = np.empty(
    (len(final_query_ids) * 300, len(CB_NUMERIC_FEATURES)), dtype=np.float32
)
cb_test_categorical = np.empty(
    (len(final_query_ids) * 300, len(CB_CATEGORICAL_FEATURES)), dtype=np.int32
)
for row in range(len(final_query_ids)):
    items = cb_test_items[row]
    query_lat, query_lon = cb_test_query_centres[row]
    item_lat, item_lon = cb_test_item_lat[items], cb_test_item_lon[items]
    lat_delta = np.deg2rad(item_lat - query_lat)
    lon_delta = np.deg2rad(item_lon - query_lon)
    distance_part = (
        np.sin(lat_delta / 2) ** 2
        + np.cos(np.deg2rad(query_lat)) * np.cos(np.deg2rad(item_lat))
        * np.sin(lon_delta / 2) ** 2
    )
    distance_km = 2 * 6371.0088 * np.arcsin(
        np.sqrt(np.clip(distance_part, 0, 1))
    )
    filter_code = final_filter_codes[row]
    filter_match = (
        np.isin(items, final_filter_items[filter_code]).astype(np.float32)
        if filter_code >= 0 else np.zeros(300, dtype=np.float32)
    )
    parameter_score = (
        cb_lookup_scores(
            items, cb_parameter_indices[filter_code],
            cb_parameter_scores[filter_code],
        ) if filter_code >= 0 else np.zeros(300, dtype=np.float32)
    )
    text_scores = [
        cb_lookup_scores(
            items, final_source_indices[name][row],
            final_source_scores[name][row],
        )
        for name in SOURCE_NAMES
    ]
    e5_score = cb_lookup_scores(
        items, final_embedding_indices[row], final_embedding_scores[row]
    )
    start, stop = row * 300, (row + 1) * 300
    cb_test_numeric[start:stop] = np.column_stack([
        cb_test_scores[row], np.arange(1, 301),
        *text_scores, e5_score, cb_test_local_scores[row],
        final_local_geo_score(row, items),
        (final_item_locations[items] == final_query_locations[row]).astype(np.float32),
        np.full(300, query_lat), np.full(300, query_lon),
        item_lat, item_lon, distance_km, filter_match, parameter_score,
        cb_test_log_price[items], cb_test_price_missing[items],
        cb_test_rating_values[items], cb_test_rating_missing[items],
        cb_test_log_reviews[items], cb_test_reviews_missing[items],
        cb_test_phone[items], cb_test_message[items],
        np.full(300, cb_test_delivery[row]),
        np.full(300, np.isfinite(query_lat)),
    ])
    cb_test_categorical[start:stop] = np.column_stack([
        np.full(300, cb_test_query_category[row]),
        cb_test_category[items], cb_test_microcat[items],
    ])
    if (row + 1) % 500 == 0 or row + 1 == len(final_query_ids):
        print(f"CatBoost-признаки benchmark: {row + 1:,}/{len(final_query_ids):,}")

cb_test_features = pd.DataFrame(cb_test_numeric, columns=CB_NUMERIC_FEATURES)
for number, name in enumerate(CB_CATEGORICAL_FEATURES):
    cb_test_features[name] = cb_test_categorical[:, number]
assert cb_test_features.columns.tolist() == CB_NUMERIC_FEATURES + CB_CATEGORICAL_FEATURES
print(f"Признаков на кандидата: {cb_test_features.shape[1]}")
del cb_test_numeric, cb_test_categorical, cb_parameter_indices, cb_parameter_scores
gc.collect()


bm25_parameters: 32/127 запросов
bm25_parameters: 127/127 запросов
CatBoost-признаки benchmark: 500/2,452
CatBoost-признаки benchmark: 1,000/2,452
CatBoost-признаки benchmark: 1,500/2,452
CatBoost-признаки benchmark: 2,000/2,452
CatBoost-признаки benchmark: 2,452/2,452
Признаков на кандидата: 30


2055

In [92]:
# Лучшая на validation смесь: 25% прежнего скора + 75% CatBoost.
from catboost import CatBoostRanker
cb_model_path = Path("artifacts/models/catboost_current_top300.cbm")
assert cb_model_path.exists(), "Сначала обучите модель top-300"
cb_test_model = CatBoostRanker()
cb_test_model.load_model(str(cb_model_path))
assert cb_test_features.columns.tolist() == cb_test_model.feature_names_
cb_model_scores = cb_test_model.predict(cb_test_features).reshape(
    len(final_query_ids), 300
).astype(np.float32)
cb_model_scores = (
    (cb_model_scores - cb_model_scores.min(axis=1, keepdims=True))
    / np.maximum(np.ptp(cb_model_scores, axis=1, keepdims=True), 1e-8)
)
cb_final_scores = 0.25 * cb_test_scores + 0.75 * cb_model_scores
cb_final_indices = np.stack([
    cb_test_items[row, top_positions(cb_final_scores[row], 50)]
    for row in range(len(final_query_ids))
])
CB_ANSWER_PATH = Path("answer_geo_local_catboost_top300.csv")
cb_answer = pd.DataFrame({
    "query_id": final_query_ids,
    "answer": [" ".join(final_item_ids[indices]) for indices in cb_final_indices],
})
cb_answer.to_csv(CB_ANSWER_PATH, index=False, encoding="utf-8")

# Проверяем формат CSV и принадлежность всех item_id корпусу.
cb_check = pd.read_csv(
    CB_ANSWER_PATH, dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
cb_lists = cb_check["answer"].str.split()
known_items = set(final_item_ids)
assert cb_check.columns.tolist() == ["query_id", "answer"]
assert len(cb_check) == len(final_query_ids)
assert cb_check["query_id"].is_unique
assert set(cb_check["query_id"]) == set(final_query_ids)
assert cb_check["query_id"].str.len().eq(16).all()
assert cb_lists.map(lambda ids: len(ids) == 50 and len(ids) == len(set(ids))).all()
assert cb_lists.map(lambda ids: set(ids).issubset(known_items)).all()
assert all(len(item_id) == 16 for ids in cb_lists for item_id in ids)
changed_queries = sum(
    set(cb_final_indices[row]) != set(final_top50_indices[row])
    for row in range(len(final_query_ids))
)
print(f"Создан файл: {CB_ANSWER_PATH.resolve()}")
print(f"Изменился top-50 у {changed_queries:,} из {len(final_query_ids):,} запросов")
# Оставляем cb_test_features для следующего эксперимента с новыми признаками.
del cb_model_scores, cb_final_scores
gc.collect()


Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_geo_local_catboost_top300.csv
Изменился top-50 у 2,409 из 2,452 запросов


0

## 29. Дополнительные признаки и финальное обучение CatBoost

Сначала проверяем новые признаки на прежнем разделении 3 000/1 500 запросов. Старую модель и её CSV не меняем. После сравнения обучаем новую модель на всех 4 500 запросах и отдельно готовим benchmark-ответ.

In [93]:
# Позиции в источниках, частичное совпадение фильтра, слова и относительная цена.
CB_EXTRA_FEATURES = [
    "bm25_td_rank", "char_title_rank", "bm25_title_rank",
    "word_title_rank", "e5_rank", "parameter_rank",
    "filter_coverage", "has_filter", "query_in_title",
    "query_title_overlap", "query_word_count",
    "title_word_count", "description_chars",
    "price_vs_query_median",
]

def cb_source_rank(candidates, source_indices):
    """Относительная позиция в источнике; 1.1 означает отсутствие."""
    source = np.asarray(source_indices)
    source = source[source >= 0]
    result = np.full(len(candidates), 1.1, dtype=np.float32)
    if not len(source):
        return result
    order = np.argsort(source)
    positions = np.searchsorted(source[order], candidates)
    in_range = positions < len(source)
    found = np.zeros(len(candidates), dtype=bool)
    found[in_range] = source[order[positions[in_range]]] == candidates[in_range]
    result[found] = (order[positions[found]] + 1) / len(source)
    return result

def add_cb_extra_features(
    frame, candidate_matrix, query_rows, query_texts, item_frame,
    source_indices, e5_indices, parameter_indices, filter_codes, filters,
):
    item_tokens, filter_tokens, filter_counts = build_filter_token_matrices(
        item_frame["item_infm_params_text"], filters
    )
    titles = item_frame["item_title_raw"].fillna("").astype(str).to_numpy()
    description_chars = item_frame["item_description_raw"].fillna("").str.len().to_numpy(np.float32)
    frame_prices = frame["log_price"].to_numpy()
    frame_price_missing = frame["price_missing"].to_numpy()
    extra = np.empty((candidate_matrix.size, len(CB_EXTRA_FEATURES)), dtype=np.float32)
    for position, row in enumerate(query_rows):
        items = candidate_matrix[position]
        start, stop = position * 300, (position + 1) * 300
        query = str(query_texts[row])
        query_words = set(query.split())
        item_titles = titles[items]
        title_words = [title.split() for title in item_titles]
        filter_code = filter_codes[row]
        coverage = np.zeros(300, dtype=np.float32)
        parameter_rank = np.full(300, 1.1, dtype=np.float32)
        if filter_code >= 0:
            coverage = (
                (filter_tokens[filter_code] @ item_tokens[items].T)
                .toarray().ravel() / max(1, filter_counts[filter_code])
            ).astype(np.float32)
            parameter_rank = cb_source_rank(
                items, parameter_indices[filter_code]
            )
        prices = frame_prices[start:stop]
        known_price = frame_price_missing[start:stop] == 0
        median_price = np.median(prices[known_price]) if known_price.any() else 0
        extra[start:stop] = np.column_stack([
            *[cb_source_rank(items, source_indices[name][row]) for name in SOURCE_NAMES],
            cb_source_rank(items, e5_indices[row]), parameter_rank,
            coverage, np.full(300, filter_code >= 0),
            np.fromiter((query in title for title in item_titles), np.float32, 300),
            np.fromiter(
                (len(query_words.intersection(words)) / max(1, len(query_words))
                 for words in title_words), np.float32, 300,
            ),
            np.full(300, len(query_words)),
            np.fromiter((len(words) for words in title_words), np.float32, 300),
            description_chars[items],
            np.where(known_price, prices - median_price, 0),
        ])
        if (position + 1) % 500 == 0 or position + 1 == len(query_rows):
            print(f"Дополнительные признаки: {position + 1:,}/{len(query_rows):,}")
    for number, name in enumerate(CB_EXTRA_FEATURES):
        frame[name] = extra[:, number]
    del item_tokens, filter_tokens, extra
    gc.collect()
    return frame

assert cb_features.shape[1] == 30
cb_features = add_cb_extra_features(
    cb_features, cb_items, model_rows, query_table["search_query"].to_numpy(),
    train_items, extended_top_indices, embedding_top_indices,
    parameter_top_indices, parameter_filter_codes, unique_parameter_filters,
)
print(f"Признаков: {cb_features.shape[1]}")


Дополнительные признаки: 500/4,500
Дополнительные признаки: 1,000/4,500
Дополнительные признаки: 1,500/4,500
Дополнительные признаки: 2,000/4,500
Дополнительные признаки: 2,500/4,500
Дополнительные признаки: 3,000/4,500
Дополнительные признаки: 3,500/4,500
Дополнительные признаки: 4,000/4,500
Дополнительные признаки: 4,500/4,500
Признаков: 44


In [94]:
# Сначала сравниваем новые признаки со старой моделью на тех же 1 500 запросах.
from catboost import CatBoostRanker, Pool
def cb_extra_pool(group_positions):
    indices = (
        group_positions[:, None] * CB_MAX_K + np.arange(CB_MAX_K)
    ).ravel()
    return Pool(
        cb_features.iloc[indices], label=cb_labels[indices],
        group_id=np.repeat(group_positions, CB_MAX_K),
        cat_features=CB_CATEGORICAL_FEATURES,
    )

cb_extra_positive = cb_labels.reshape(len(model_rows), CB_MAX_K).any(axis=1)
cb_extra_train_groups = np.flatnonzero(
    cb_extra_positive[:CB_TRAIN_QUERIES]
)
cb_extra_train_pool = cb_extra_pool(cb_extra_train_groups)
cb_extra_valid_pool = cb_extra_pool(
    np.arange(CB_TRAIN_QUERIES, len(model_rows))
)
cb_extra_model = CatBoostRanker(
    loss_function="YetiRank", eval_metric="NDCG:top=50",
    iterations=300, depth=6, learning_rate=0.08, l2_leaf_reg=5,
    random_seed=42, thread_count=4,
    allow_writing_files=False, verbose=50,
)
cb_extra_model.fit(
    cb_extra_train_pool, eval_set=cb_extra_valid_pool,
    early_stopping_rounds=35,
)
cb_extra_model_path = Path("artifacts/models/catboost_extra_top300_valid.cbm")
cb_extra_model.save_model(str(cb_extra_model_path))

cb_extra_scores = cb_extra_model.predict(cb_extra_valid_pool).reshape(
    CB_VALID_QUERIES, CB_MAX_K
).astype(np.float32)
cb_extra_scores = (
    (cb_extra_scores - cb_extra_scores.min(axis=1, keepdims=True))
    / np.maximum(np.ptp(cb_extra_scores, axis=1, keepdims=True), 1e-8)
)
cb_extra_results = []
for weight in [0, 0.10, 0.25, 0.40, 0.55, 0.70, 0.80, 0.90, 1.00]:
    mixed = (
        (1 - weight) * cb_scores[CB_TRAIN_QUERIES:]
        + weight * cb_extra_scores
    )
    recall = np.mean([
        recall_for_item_indices(
            row, cb_items[CB_TRAIN_QUERIES + position][
                top_positions(mixed[position], 50)
            ],
        )
        for position, row in enumerate(model_valid_rows)
    ])
    cb_extra_results.append({"model_weight": weight, "recall_at_50": recall})
cb_extra_results = pd.DataFrame(cb_extra_results).sort_values(
    "recall_at_50", ascending=False
).reset_index(drop=True)
display(cb_extra_results)
print("Старая модель + формула на тех же запросах: 0.8256")
display(pd.DataFrame({
    "feature": cb_extra_model.feature_names_,
    "importance": cb_extra_model.get_feature_importance(
        type="PredictionValuesChange"
    ),
}).sort_values("importance", ascending=False).head(20))
del cb_extra_train_pool, cb_extra_valid_pool, cb_extra_scores
gc.collect()


Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.4074498	best: 0.4074498 (0)	total: 5s	remaining: 24m 54s
50:	test: 0.5701033	best: 0.5701474 (49)	total: 2m 27s	remaining: 12m 1s
100:	test: 0.5729252	best: 0.5733740 (87)	total: 4m 5s	remaining: 8m 3s
150:	test: 0.5766155	best: 0.5771749 (145)	total: 5m 43s	remaining: 5m 39s
200:	test: 0.5782042	best: 0.5782683 (197)	total: 7m 21s	remaining: 3m 37s
250:	test: 0.5794217	best: 0.5800917 (249)	total: 8m 58s	remaining: 1m 45s
299:	test: 0.5816324	best: 0.5824287 (293)	total: 10m 34s	remaining: 0us

bestTest = 0.5824286979
bestIteration = 293

Shrink model to first 294 iterations.


,model_weight,recall_at_50
0,0.7000,0.8320
1,0.8000,0.8305
2,0.5500,0.8294
3,0.9000,0.8288
4,1.0000,0.8281
5,0.4000,0.8243
6,0.2500,0.8180
7,0.1000,0.8114
8,0.0000,0.7984


Старая модель + формула на тех же запросах: 0.8256


,feature,importance
0,mixed_score,27.5741
6,e5_cosine,21.5980
8,geo_score,10.9149
1,hybrid_rank,10.2316
34,e5_rank,9.4315
33,word_title_rank,4.4542
2,bm25_title_description,2.4879
5,word_tfidf_title,2.0348
7,local_bm25,1.8249
14,distance_km,1.5472


0

In [95]:
# Финальная модель: все 4 500 подготовленных запросов, без нового поиска и E5.
cb_all_pool = cb_extra_pool(np.arange(len(model_rows)))
cb_final_iterations = cb_extra_model.best_iteration_ + 1
cb_extra_final_model = CatBoostRanker(
    loss_function="YetiRank", iterations=cb_final_iterations,
    depth=6, learning_rate=0.08, l2_leaf_reg=5,
    random_seed=42, thread_count=4,
    allow_writing_files=False, verbose=50,
)
cb_extra_final_model.fit(cb_all_pool)
cb_extra_final_path = Path("artifacts/models/catboost_extra_all4500_top300.cbm")
cb_extra_final_model.save_model(str(cb_extra_final_path))
cb_extra_best_weight = float(cb_extra_results.iloc[0]["model_weight"])
print(
    f"Модель: {cb_extra_final_path}; групп: {len(model_rows):,}; "
    f"деревьев: {cb_final_iterations}; вес: {cb_extra_best_weight:.2f}"
)
del cb_all_pool
gc.collect()


Groupwise loss function. OneHotMaxSize set to 10
0:	total: 9.82s	remaining: 47m 58s
50:	total: 3m 18s	remaining: 15m 45s
100:	total: 6m 6s	remaining: 11m 40s
150:	total: 8m 44s	remaining: 8m 17s
200:	total: 11m 36s	remaining: 5m 22s
250:	total: 14m 28s	remaining: 2m 28s
293:	total: 16m 56s	remaining: 0us
Модель: artifacts\models\catboost_extra_all4500_top300.cbm; групп: 4,500; деревьев: 294; вес: 0.70


0

In [99]:
# Дополняем benchmark-таблицу из 30 признаков ещё 14 признаками.
# Текущие top-300 кандидатов при этом не пересчитываются.
assert "cb_test_features" in globals(), (
    "Сначала повторите ячейку catboost-benchmark-features (30 признаков)"
)
assert cb_test_features.shape[1] == 30
cb_test_features = add_cb_extra_features(
    cb_test_features, cb_test_items, np.arange(len(final_query_ids)),
    benchmark_queries["search_query"].to_numpy(), benchmark_items,
    final_source_indices, final_embedding_indices,
    final_parameter_indices, final_filter_codes, final_unique_filters,
)
assert cb_test_features.columns.tolist() == cb_extra_final_model.feature_names_
print(f"Признаков benchmark: {cb_test_features.shape[1]}")


Дополнительные признаки: 500/2,452
Дополнительные признаки: 1,000/2,452
Дополнительные признаки: 1,500/2,452
Дополнительные признаки: 2,000/2,452
Дополнительные признаки: 2,452/2,452
Признаков benchmark: 44


In [100]:
# Новый ответ не перезаписывает успешный файл с Recall@50 = 0.828573.
cb_extra_benchmark_scores = cb_extra_final_model.predict(
    cb_test_features
).reshape(len(final_query_ids), 300).astype(np.float32)
cb_extra_benchmark_scores = (
    cb_extra_benchmark_scores
    - cb_extra_benchmark_scores.min(axis=1, keepdims=True)
) / np.maximum(
    np.ptp(cb_extra_benchmark_scores, axis=1, keepdims=True), 1e-8
)
cb_extra_mixed = (
    (1 - cb_extra_best_weight) * cb_test_scores
    + cb_extra_best_weight * cb_extra_benchmark_scores
)
cb_extra_top50 = np.stack([
    cb_test_items[row, top_positions(cb_extra_mixed[row], 50)]
    for row in range(len(final_query_ids))
])
CB_EXTRA_ANSWER_PATH = Path("answer_geo_local_catboost_extra_all4500.csv")
cb_extra_answer = pd.DataFrame({
    "query_id": final_query_ids,
    "answer": [" ".join(final_item_ids[items]) for items in cb_extra_top50],
})
cb_extra_answer.to_csv(
    CB_EXTRA_ANSWER_PATH, index=False, encoding="utf-8"
)
cb_extra_check = pd.read_csv(
    CB_EXTRA_ANSWER_PATH, dtype={"query_id": "string", "answer": "string"},
    keep_default_na=False,
)
cb_extra_lists = cb_extra_check["answer"].str.split()
known_items = set(final_item_ids)
assert cb_extra_check.columns.tolist() == ["query_id", "answer"]
assert len(cb_extra_check) == len(final_query_ids)
assert cb_extra_check["query_id"].is_unique
assert set(cb_extra_check["query_id"]) == set(final_query_ids)
assert cb_extra_check["query_id"].str.len().eq(16).all()
assert cb_extra_lists.map(
    lambda ids: len(ids) == 50 and len(set(ids)) == 50
).all()
assert cb_extra_lists.map(lambda ids: set(ids).issubset(known_items)).all()
changed = sum(
    set(cb_extra_top50[row]) != set(cb_final_indices[row])
    for row in range(len(final_query_ids))
)
print(f"Создан файл: {CB_EXTRA_ANSWER_PATH.resolve()}")
print(f"По сравнению с прежним CatBoost изменилось {changed:,} из {len(final_query_ids):,} выдач")
del cb_extra_benchmark_scores, cb_extra_mixed
gc.collect()


Создан файл: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_geo_local_catboost_extra_all4500.csv
По сравнению с прежним CatBoost изменилось 2,364 из 2,452 выдач


0

## 30. LightGBM на top-300

Восстанавливаем одинаковые признаки и разбиение для сравнения со сохранённой CatBoost-моделью.

In [1]:
# Восстанавливаем 4 500 запросов, top-300 кандидатов, признаки и метки.
# Глобальный текстовый поиск и E5 читаются из сохранённых массивов;
# локальный поиск при отсутствии его результатов вычисляется и сохраняется.
from importlib import reload
import avito_candidates.recovery as recovery
from avito_candidates.retrieval import top_positions
import gc
import numpy as np
import pandas as pd
from catboost import CatBoostRanker

# Распаковываем восстановленные данные для сравнения ранжировщиков.
restored = reload(recovery).restore_ranking_validation()
CB_MAX_K = restored['CB_MAX_K']
CB_TRAIN_QUERIES = restored['CB_TRAIN_QUERIES']
CB_VALID_QUERIES = restored['CB_VALID_QUERIES']
cb_features = restored['cb_features']
cb_items = restored['cb_items']
current_scores = restored['current_scores']
cb_labels = restored['cb_labels']
cb_extra_train_groups = restored['cb_extra_train_groups']
model_valid_rows = restored['model_valid_rows']
recall_for_item_indices = restored['recall_for_item_indices']

bm25_parameters: рассчитано и сохранено
Локальный BM25: 100 локаций
Локальный BM25: 200 локаций
Локальный BM25: 300 локаций
Локальный BM25: 400 локаций
Локальный BM25: 500 локаций
Признаки CatBoost: 500/4,500
Признаки CatBoost: 1,000/4,500
Признаки CatBoost: 1,500/4,500
Признаки CatBoost: 2,000/4,500
Признаки CatBoost: 2,500/4,500
Признаки CatBoost: 3,000/4,500
Признаки CatBoost: 3,500/4,500
Признаки CatBoost: 4,000/4,500
Признаки CatBoost: 4,500/4,500
Восстановлено 4,500 запросов; Recall@50 = 0.8327 (прежний ≈ 0.8320)


### 30.1. Обучение на 3 000 и проверка на 1 500 запросах

In [3]:
# Эксперимент: LightGBM на тех же top-300 и 44 признаках, что у CatBoost.
# Используем восстановленные признаки и разделение 3 000/1 500 запросов.
# Группы объявлений одного запроса не перемешиваем между собой.
import lightgbm as lgb

train_rows = (
    cb_extra_train_groups[:, None] * CB_MAX_K + np.arange(CB_MAX_K)
).ravel()
valid_start = CB_TRAIN_QUERIES * CB_MAX_K
X_train = cb_features.iloc[train_rows]
X_valid = cb_features.iloc[valid_start:]
y_train = cb_labels[train_rows]
y_valid = cb_labels[valid_start:]

lgb_model = lgb.LGBMRanker(
    objective="lambdarank", n_estimators=500, learning_rate=0.05,
    num_leaves=31, min_child_samples=100, reg_lambda=5.0,
    n_jobs=4, random_state=42, verbosity=-1,
)
lgb_model.fit(
    X_train, y_train,
    group=np.full(len(cb_extra_train_groups), CB_MAX_K),
    eval_set=[(X_valid, y_valid)],
    eval_group=[np.full(CB_VALID_QUERIES, CB_MAX_K)],
    eval_at=(50,),
    categorical_feature=["search_category", "item_category_id", "item_microcat_id"],
    callbacks=[lgb.early_stopping(40), lgb.log_evaluation(50)],
)
lgb_scores = lgb_model.predict(X_valid).reshape(CB_VALID_QUERIES, CB_MAX_K).astype(np.float32)
lgb_scores = (lgb_scores - lgb_scores.min(axis=1, keepdims=True)) / np.maximum(
    np.ptp(lgb_scores, axis=1, keepdims=True), 1e-8
)
del X_train, X_valid
gc.collect()

# Вес 0 — проверенный CatBoost-гибрид, вес 1 — только LightGBM.
lgb_results = []
for weight in [0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00]:
    blended = (1 - weight) * current_scores + weight * lgb_scores
    recall = np.array([
        recall_for_item_indices(
            row, cb_items[CB_TRAIN_QUERIES + pos][top_positions(blended[pos], 50)]
        )
        for pos, row in enumerate(model_valid_rows)
    ])
    lgb_results.append({
        "lightgbm_weight": weight, "recall_at_50": recall.mean(),
        "first_half": recall[::2].mean(), "second_half": recall[1::2].mean(),
    })

print(f"Лучшая итерация LightGBM: {lgb_model.best_iteration_}")
display(pd.DataFrame(lgb_results).sort_values("recall_at_50", ascending=False))

Training until validation scores don't improve for 40 rounds
[50]	valid_0's ndcg@50: 0.568315
[100]	valid_0's ndcg@50: 0.577386
Early stopping, best iteration is:
[100]	valid_0's ndcg@50: 0.577386
Лучшая итерация LightGBM: 100


,lightgbm_weight,recall_at_50,first_half,second_half
4,0.30,0.838474,0.832222,0.844726
3,0.20,0.838141,0.832889,0.843393
5,0.50,0.835474,0.832222,0.838726
2,0.10,0.834696,0.832889,0.836504
1,0.05,0.834030,0.831556,0.836504
0,0.00,0.832696,0.830222,0.835170
6,0.75,0.832385,0.830889,0.833881
7,1.00,0.831385,0.830889,0.831881


### 30.2. Число деревьев и вес в смеси

In [4]:
# Проверяем сохранённые итерации LightGBM в смеси с текущим CatBoost-гибридом.
# Переобучения нет: num_iteration берёт первые N деревьев готовой модели.
max_iteration = lgb_model.booster_.current_iteration()
iterations = sorted({
    step for step in [50, 75, 100, 125, lgb_model.best_iteration_, max_iteration]
    if 0 < step <= max_iteration
})
weights = [0.20, 0.30, 0.40]
validation_features = cb_features.iloc[CB_TRAIN_QUERIES * CB_MAX_K:]
iteration_results = []

for iteration in iterations:
    scores = lgb_model.predict(
        validation_features, num_iteration=iteration
    ).reshape(CB_VALID_QUERIES, CB_MAX_K).astype(np.float32)
    scores = (scores - scores.min(axis=1, keepdims=True)) / np.maximum(
        np.ptp(scores, axis=1, keepdims=True), 1e-8
    )
    for weight in weights:
        mixed = (1 - weight) * current_scores + weight * scores
        recall = np.array([
            recall_for_item_indices(
                row, cb_items[CB_TRAIN_QUERIES + pos][top_positions(mixed[pos], 50)]
            )
            for pos, row in enumerate(model_valid_rows)
        ])
        iteration_results.append({
            "iteration": iteration, "lightgbm_weight": weight,
            "recall_at_50": recall.mean(),
            "first_half": recall[::2].mean(),
            "second_half": recall[1::2].mean(),
        })
    print(f"Проверено деревьев: {iteration}/{max_iteration}")

print(f"Без LightGBM: Recall@50 = {lgb_results[0]['recall_at_50']:.6f}")
display(pd.DataFrame(iteration_results).sort_values("recall_at_50", ascending=False))

Проверено деревьев: 50/100
Проверено деревьев: 75/100
Проверено деревьев: 100/100
Без LightGBM: Recall@50 = 0.832696


,iteration,lightgbm_weight,recall_at_50,first_half,second_half
7,100,0.3,0.838474,0.832222,0.844726
6,100,0.2,0.838141,0.832889,0.843393
8,100,0.4,0.837941,0.832222,0.843659
0,50,0.2,0.837474,0.832889,0.842059
1,50,0.3,0.836807,0.832889,0.840726
4,75,0.3,0.836474,0.832222,0.840726
5,75,0.4,0.835807,0.830889,0.840726
2,50,0.4,0.835141,0.831556,0.838726
3,75,0.2,0.834807,0.830222,0.839393


### 30.3. Проверка дополнительных итераций

In [5]:
# Проверяем, улучшится ли Recall@50 после 100 деревьев LightGBM.
# Та же обучающая/валидационная выборка; ранней остановки по NDCG нет.
# Готовая модель сохраняется, чтобы после перезапуска не обучать её снова.
from sklearn.base import clone
from avito_candidates.config import ARTIFACTS

model_path = ARTIFACTS / "retrieval" / "logreg_validation" / "lightgbm_300_valid.txt"
if model_path.exists():
    long_booster = lgb.Booster(model_file=str(model_path))
    print("LightGBM: сохранённая модель загружена")
else:
    X_train = cb_features.iloc[train_rows]
    X_valid = cb_features.iloc[valid_start:]
    long_model = clone(lgb_model).set_params(n_estimators=300)
    long_model.fit(
        X_train, y_train,
        group=np.full(len(cb_extra_train_groups), CB_MAX_K),
        eval_set=[(X_valid, y_valid)],
        eval_group=[np.full(CB_VALID_QUERIES, CB_MAX_K)],
        eval_at=(50,),
        categorical_feature=["search_category", "item_category_id", "item_microcat_id"],
        callbacks=[lgb.log_evaluation(50)],
    )
    long_booster = long_model.booster_
    long_booster.save_model(str(model_path))
    del X_train, X_valid
    gc.collect()

validation_features = cb_features.iloc[valid_start:]
results_300 = []
for iteration in [100, 125, 150, 175, 200, 250, 300]:
    scores = long_booster.predict(
        validation_features, num_iteration=iteration
    ).reshape(CB_VALID_QUERIES, CB_MAX_K).astype(np.float32)
    scores = (scores - scores.min(axis=1, keepdims=True)) / np.maximum(
        np.ptp(scores, axis=1, keepdims=True), 1e-8
    )
    for weight in [0.20, 0.30, 0.40]:
        mixed = (1 - weight) * current_scores + weight * scores
        recall = np.array([
            recall_for_item_indices(
                row, cb_items[CB_TRAIN_QUERIES + pos][top_positions(mixed[pos], 50)]
            )
            for pos, row in enumerate(model_valid_rows)
        ])
        results_300.append({
            "iteration": iteration, "lightgbm_weight": weight,
            "recall_at_50": recall.mean(),
            "first_half": recall[::2].mean(),
            "second_half": recall[1::2].mean(),
        })
    print(f"Проверено деревьев: {iteration}/300")

print(f"Без LightGBM: Recall@50 = {lgb_results[0]['recall_at_50']:.6f}")
display(pd.DataFrame(results_300).sort_values("recall_at_50", ascending=False))

[50]	valid_0's ndcg@50: 0.568315
[100]	valid_0's ndcg@50: 0.577386
[150]	valid_0's ndcg@50: 0.571787
[200]	valid_0's ndcg@50: 0.569719
[250]	valid_0's ndcg@50: 0.566461
[300]	valid_0's ndcg@50: 0.565234
Проверено деревьев: 100/300
Проверено деревьев: 125/300
Проверено деревьев: 150/300
Проверено деревьев: 175/300
Проверено деревьев: 200/300
Проверено деревьев: 250/300
Проверено деревьев: 300/300
Без LightGBM: Recall@50 = 0.832696


,iteration,lightgbm_weight,recall_at_50,first_half,second_half
14,200,0.4,0.840274,0.834889,0.845659
8,150,0.4,0.839941,0.834222,0.845659
9,175,0.2,0.839474,0.835556,0.843393
12,200,0.2,0.839274,0.833556,0.844993
11,175,0.4,0.838941,0.834889,0.842993
3,125,0.2,0.838807,0.834222,0.843393
6,150,0.2,0.838807,0.834222,0.843393
13,200,0.3,0.838607,0.830889,0.846326
10,175,0.3,0.838607,0.832222,0.844993
1,100,0.3,0.838474,0.832222,0.844726


### 30.4. Top-500 со старыми моделями

In [6]:
# Проверка top-500 без переобучения CatBoost и LightGBM.
# Используем ту же отложенную часть (1 500 запросов), что и в предыдущей таблице.
from importlib import reload
import avito_candidates.features as feature_module
import avito_candidates.pipeline as pipeline_module
import avito_candidates.recovery as recovery
from avito_candidates.retrieval import top_positions

reload(feature_module)
reload(pipeline_module)
test500 = reload(recovery).restore_ranking_validation(top_k=500, validation_only=True)
booster = long_booster  # Модель из предыдущей ячейки; не загружаем её второй раз.
assert test500['cb_features'].columns.tolist() == booster.feature_name()
assert np.array_equal(test500['model_valid_rows'], model_valid_rows)

# Модель обучалась на top-300; здесь только применяем её к ещё 200 кандидатам.
lgb500 = booster.predict(test500['cb_features'], num_iteration=200).reshape(-1, 500).astype(np.float32)
lgb500 = (lgb500 - lgb500.min(axis=1, keepdims=True)) / np.maximum(
    np.ptp(lgb500, axis=1, keepdims=True), 1e-8
)
mixed500 = 0.6 * test500['current_scores'] + 0.4 * lgb500

def depth_recall(candidate_items, scores, query_rows, recall_fn):
    pool, top50 = [], []
    for pos, row in enumerate(query_rows):
        pool.append(recall_fn(row, candidate_items[pos]))
        top50.append(recall_fn(row, candidate_items[pos, top_positions(scores[pos], 50)]))
    top50 = np.asarray(top50)
    return np.mean(pool), top50.mean(), top50[::2].mean(), top50[1::2].mean()

baseline300 = next(result for result in results_300
                   if result['iteration'] == 200 and result['lightgbm_weight'] == 0.4)
old_items = cb_items[CB_TRAIN_QUERIES:]
pool300 = np.mean([recall_for_item_indices(row, old_items[pos])
                   for pos, row in enumerate(model_valid_rows)])
pool500, recall500, half1, half2 = depth_recall(
    test500['cb_items'], mixed500, test500['model_valid_rows'],
    test500['recall_for_item_indices']
)
display(pd.DataFrame([
    {'top_k': 300, 'pool_recall': pool300, 'recall_at_50': baseline300['recall_at_50'],
     'first_half': baseline300['first_half'], 'second_half': baseline300['second_half']},
    {'top_k': 500, 'pool_recall': pool500, 'recall_at_50': recall500,
     'first_half': half1, 'second_half': half2},
]))
print(f'Изменение Recall@50: {recall500 - baseline300["recall_at_50"]:+.6f}')

bm25_parameters: рассчитано и сохранено
Локальный BM25: 100 локаций
Локальный BM25: 200 локаций
Локальный BM25: 300 локаций
Признаки CatBoost: 500/1,500
Признаки CatBoost: 1,000/1,500
Признаки CatBoost: 1,500/1,500
top-500: 1,500 запросов; Recall@50 = 0.8314


,top_k,pool_recall,recall_at_50,first_half,second_half
0,300,0.911537,0.840274,0.834889,0.845659
1,500,0.925648,0.836941,0.830889,0.842993


Изменение Recall@50: -0.003333


### 30.5. Top-500 с переобученным LightGBM

In [7]:
# Top-500: обучаем LightGBM на 3 000 запросов, проверяем на прежних 1 500.
# Готовые E5 и текстовые результаты читаются из файлов; признаки validation уже сохранены.
import gc
import lightgbm as lgb
from importlib import reload
from avito_candidates.config import ARTIFACTS
import avito_candidates.recovery as recovery
from avito_candidates.retrieval import top_positions

model500_path = ARTIFACTS / 'retrieval' / 'top500_train' / 'lightgbm_250_valid.txt'
if model500_path.exists():
    booster500 = lgb.Booster(model_file=str(model500_path))
    print('LightGBM top-500: сохранённая модель загружена')
else:
    train500 = reload(recovery).restore_ranking_validation(top_k=500, train_only=True)
    groups500 = train500['cb_extra_train_groups']
    train_indices500 = (groups500[:, None] * 500 + np.arange(500)).ravel()
    X_train500 = train500['cb_features'].iloc[train_indices500]
    y_train500 = train500['cb_labels'][train_indices500]
    del train500
    gc.collect()
    model500 = lgb.LGBMRanker(
        objective='lambdarank', n_estimators=250, learning_rate=0.05,
        num_leaves=31, min_child_samples=100, reg_lambda=5.0,
        n_jobs=4, random_state=42, verbosity=-1,
    )
    model500.fit(
        X_train500, y_train500, group=np.full(len(groups500), 500),
        eval_set=[(test500['cb_features'], test500['cb_labels'])],
        eval_group=[np.full(1500, 500)], eval_at=(50,),
        categorical_feature=['search_category', 'item_category_id', 'item_microcat_id'],
        callbacks=[lgb.log_evaluation(50)],
    )
    booster500 = model500.booster_
    booster500.save_model(str(model500_path))
    del X_train500, y_train500
    gc.collect()

# NDCG печатается при обучении, но выбираем по целевой метрике Recall@50.
results_top500_lgb = []
for trees in [100, 150, 200, 250]:
    scores = booster500.predict(test500['cb_features'], num_iteration=trees).reshape(1500, 500).astype(np.float32)
    scores = (scores - scores.min(axis=1, keepdims=True)) / np.maximum(np.ptp(scores, axis=1, keepdims=True), 1e-8)
    for weight in [0.2, 0.4, 0.6, 0.8, 1.0]:
        mixed = (1 - weight) * test500['current_scores'] + weight * scores
        recall = np.array([
            test500['recall_for_item_indices'](
                row, test500['cb_items'][pos, top_positions(mixed[pos], 50)]
            ) for pos, row in enumerate(test500['model_valid_rows'])
        ])
        results_top500_lgb.append({
            'trees': trees, 'lightgbm_weight': weight, 'recall_at_50': recall.mean(),
            'first_half': recall[::2].mean(), 'second_half': recall[1::2].mean(),
        })
    print(f'Проверено деревьев: {trees}/250')
results_top500_lgb = pd.DataFrame(results_top500_lgb).sort_values('recall_at_50', ascending=False)
print('Лучший top-300 без переобучения: 0.840274')
display(results_top500_lgb.head(10))

bm25_parameters: рассчитано и сохранено
Локальный BM25: 100 локаций
Локальный BM25: 200 локаций
Локальный BM25: 300 локаций
Локальный BM25: 400 локаций
Признаки CatBoost: 500/3,000
Признаки CatBoost: 1,000/3,000
Признаки CatBoost: 1,500/3,000
Признаки CatBoost: 2,000/3,000
Признаки CatBoost: 2,500/3,000
Признаки CatBoost: 3,000/3,000
top-500: 3,000 обучающих запросов; группы с релевантными: 2,770
[50]	valid_0's ndcg@50: 0.55412
[100]	valid_0's ndcg@50: 0.560456
[150]	valid_0's ndcg@50: 0.556909
[200]	valid_0's ndcg@50: 0.556334
[250]	valid_0's ndcg@50: 0.555233
Проверено деревьев: 100/250
Проверено деревьев: 150/250
Проверено деревьев: 200/250
Проверено деревьев: 250/250
Лучший top-300 без переобучения: 0.840274


,trees,lightgbm_weight,recall_at_50,first_half,second_half
0,100,0.2,0.835141,0.830889,0.839393
11,200,0.4,0.834941,0.830889,0.838993
16,250,0.4,0.834941,0.830889,0.838993
12,200,0.6,0.834274,0.828222,0.840326
17,250,0.6,0.833941,0.828222,0.839659
5,150,0.2,0.833941,0.829556,0.838326
15,250,0.2,0.833807,0.829556,0.838059
7,150,0.6,0.833607,0.830889,0.836326
10,200,0.2,0.833274,0.828222,0.838326
13,200,0.8,0.833052,0.830889,0.835215


### 30.6. Обучение финального LightGBM на всех 4 500 запросах

In [8]:
# Финальный LightGBM: все 4 500 запросов из эксперимента top-300.
# Число деревьев (200) и вес в смеси (0.4) зафиксированы по отдельной валидации.
import gc
import lightgbm as lgb
from avito_candidates.config import ARTIFACTS

full_lgb_path = ARTIFACTS / 'retrieval' / 'logreg_validation' / 'lightgbm_all4500_top300.txt'
if full_lgb_path.exists():
    full_lgb_booster = lgb.Booster(model_file=str(full_lgb_path))
    print('LightGBM на 4 500 запросах: сохранённая модель загружена')
else:
    assert CB_MAX_K == 300 and len(cb_features) == 4500 * CB_MAX_K
    assert cb_features.columns.tolist() == long_booster.feature_name()
    full_groups = np.flatnonzero(cb_labels.reshape(4500, CB_MAX_K).any(axis=1))
    full_indices = (full_groups[:, None] * CB_MAX_K + np.arange(CB_MAX_K)).ravel()
    X_full = cb_features.iloc[full_indices]
    y_full = cb_labels[full_indices]
    full_lgb_model = lgb.LGBMRanker(
        objective='lambdarank', n_estimators=200, learning_rate=0.05,
        num_leaves=31, min_child_samples=100, reg_lambda=5.0,
        n_jobs=4, random_state=42, verbosity=-1,
    )
    full_lgb_model.fit(
        X_full, y_full, group=np.full(len(full_groups), CB_MAX_K),
        categorical_feature=['search_category', 'item_category_id', 'item_microcat_id'],
        callbacks=[lgb.log_evaluation(50)],
    )
    full_lgb_booster = full_lgb_model.booster_
    full_lgb_path.parent.mkdir(parents=True, exist_ok=True)
    full_lgb_booster.save_model(str(full_lgb_path))
    print(f'Обучено на {len(full_groups):,} группах с релевантными из 4 500; модель: {full_lgb_path}')
    del X_full, y_full, full_lgb_model
    gc.collect()
assert full_lgb_booster.current_iteration() == 200

Обучено на 4,109 группах с релевантными из 4 500; модель: D:\downloads_programs\Git_hub_progr\Avito_bootcamp\artifacts\retrieval\logreg_validation\lightgbm_all4500_top300.txt


## 31. Финальный benchmark-ответ

Проверяем совпадение прежнего CatBoost-ответа, добавляем LightGBM к top-300 и сохраняем CSV для отправки.

In [9]:
# Benchmark: те же top-300 и финальный CatBoost, затем + 0.4 × LightGBM.
# Поисковые выдачи и готовые модели читаются из сохранённых артефактов.
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRanker
from avito_candidates.config import ARTIFACTS, ROOT, CATBOOST_MODEL, CATBOOST_TOP_K, CATBOOST_WEIGHT
from avito_candidates.data import load_data
from avito_candidates.features import build_features
from avito_candidates.pipeline import candidate_top300, validate_answer
from avito_candidates.retrieval import lexical_sources, parameter_source, e5_source, top_positions
from avito_candidates.signals import Filters, Geography

bundle = load_data()
sources, source_scores = lexical_sources(bundle.queries, bundle.items)
filters = Filters.from_data(bundle.queries, bundle.items)
parameters = parameter_source(bundle.items, filters.texts)
e5 = e5_source(bundle.queries, bundle.items)
geo = Geography.from_data(bundle)
candidates, base_scores, local_scores = candidate_top300(
    bundle, geo, filters, sources, source_scores, e5, parameters
)
features = build_features(
    bundle, geo, filters, candidates, base_scores, local_scores,
    sources, source_scores, *e5, *parameters
)

catboost = CatBoostRanker()
catboost.load_model(str(CATBOOST_MODEL))
lightgbm = lgb.Booster(model_file=str(
    ARTIFACTS / 'retrieval' / 'logreg_validation' / 'lightgbm_all4500_top300.txt'
))
assert features.columns.tolist() == catboost.feature_names_ == lightgbm.feature_name()

def normalize_by_query(values):
    values = values.reshape(len(bundle.queries), CATBOOST_TOP_K).astype(np.float32)
    return (values - values.min(axis=1, keepdims=True)) / np.maximum(
        np.ptp(values, axis=1, keepdims=True), 1e-8
    )

catboost_scores = normalize_by_query(catboost.predict(features))
old_scores = (1 - CATBOOST_WEIGHT) * base_scores + CATBOOST_WEIGHT * catboost_scores
item_ids = bundle.items['item_id'].astype(str).to_numpy()
query_ids = bundle.queries['query_id'].astype(str).to_numpy()

def answer_from_scores(scores):
    return pd.DataFrame({
        'query_id': query_ids,
        'answer': [' '.join(item_ids[candidates[row, top_positions(scores[row], 50)]])
                   for row in range(len(query_ids))],
    })

# Проверяем, что модульный пайплайн точно воспроизводит прежний отправленный CSV.
reference = pd.read_csv(ROOT / 'answer_geo_local_catboost_extra_all4500.csv', dtype=str)
assert answer_from_scores(old_scores).equals(reference), 'Базовая выдача изменилась — не отправлять новый файл'

lightgbm_scores = normalize_by_query(lightgbm.predict(features, num_iteration=200))
answer = answer_from_scores(0.6 * old_scores + 0.4 * lightgbm_scores)
validate_answer(answer, bundle.queries, bundle.items)
output_path = ROOT / 'answer_catboost_lightgbm_top300.csv'
answer.to_csv(output_path, index=False, encoding='utf-8')
changed = sum(set(new.split()) != set(old.split())
              for new, old in zip(answer['answer'], reference['answer']))
print(f'Создан {output_path}; изменено выдач: {changed:,} из {len(answer):,}')

bm25_title_description: загружено
char_tfidf_title: загружено
bm25_title: загружено
word_tfidf_title: загружено
bm25_parameters: загружено
E5 top-5000: загружено
CatBoost top-300: загружено
Признаки CatBoost: 500/2,452
Признаки CatBoost: 1,000/2,452
Признаки CatBoost: 1,500/2,452
Признаки CatBoost: 2,000/2,452
Признаки CatBoost: 2,452/2,452
Создан D:\downloads_programs\Git_hub_progr\Avito_bootcamp\answer_catboost_lightgbm_top300.csv; изменено выдач: 2,406 из 2,452
